<a href="https://colab.research.google.com/github/FengruiJing/Multi-network-spatial-error-workflow/blob/main/Multi_network_spatial_error_framework_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#U.S. data and codes
shp_path = "US_HIV_Merged_total.shp"

Added 2018 STDS
excel_path = "2018Rates1.xlsx"

Download link:
https://drive.google.com/file/d/1-1HNcQqbDhUkA-33tdOnYuDYUJIYCkLY/view?usp=sharing

https://docs.google.com/spreadsheets/d/1cpXbaAEpu7A3Xy0_pjD0FsaJYjxGUizG/edit?usp=sharing&ouid=118218235341025234377&rtpof=true&sd=true

Twitter-based physical connectivity data are available from https://github.com/GIBDUSC/Place-Connectivity-Index.

Facebook-based social connectivity data are available from https://dataforgood.facebook.com/dfg/tools/social-connectedness-index.



#Link2018STDsData

In [ ]:
"""
Merge 2018 STI rate variables from an Excel file into one or more county-level
geospatial files (e.g., shapefiles), drop rows with missing required fields,
and save cleaned outputs.

Notes
-----
- FIPS should be treated as a 5-digit string. This script normalizes FIPS to
  digits-only and zero-pads to length 5 (e.g., "01001").
- If you output ESRI Shapefile (*.shp), be aware of field name length limits
  (often 10 characters). Consider GeoPackage (*.gpkg) for safer column names.
"""

from __future__ import annotations

import argparse
import re
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
import geopandas as gpd


# Columns expected in the shapefiles (kept if present)
BASE_COLS: List[str] = [
    "GonorrheaR", "ChlamydiaR", "PercenHIVp",
    "Population", "UrbanRural", "Female", "Old", "Black", "Noinsuranc",
    "Poverty", "crime16to1", "Dissimilar",
]

# Columns expected in the Excel file
NEW_RATE_COLS: List[str] = ["RateChlamydia2018", "RateGonorrhea2018", "RateHIV2018"]

# Rows are dropped if any of these fields are missing after merge
REQUIRED_COLS: List[str] = ["GonorrheaR", "ChlamydiaR", "PercenHIVp"] + NEW_RATE_COLS


def normalize_fips(x) -> Optional[str]:
    """
    Normalize a FIPS code to a 5-digit string:
    - Keep digits only
    - Zero-pad to length 5
    Returns None if input is missing or cannot be parsed.
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    digits = re.sub(r"\D+", "", s)
    if digits == "":
        return None
    # Some sources may provide longer codes; keep the last 5 digits if needed.
    if len(digits) > 5:
        digits = digits[-5:]
    return digits.zfill(5)


def load_rates(excel_path: str) -> pd.DataFrame:
    """Load and standardize the Excel rates table."""
    df = pd.read_excel(excel_path)

    if "FIPS" not in df.columns:
        raise KeyError("Excel file must contain a 'FIPS' column.")

    # Normalize FIPS and keep only required columns
    df["FIPS"] = df["FIPS"].apply(normalize_fips)
    keep_cols = ["FIPS"] + NEW_RATE_COLS

    missing_cols = [c for c in keep_cols if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Excel file is missing required columns: {missing_cols}")

    df = df[keep_cols].copy()
    df = df.dropna(subset=["FIPS"])
    df = df.drop_duplicates(subset=["FIPS"])

    return df


def merge_and_clean(
    shp_in: str,
    shp_out: str,
    rates_df: pd.DataFrame,
    verbose: bool = True,
) -> gpd.GeoDataFrame:
    """
    Read a geospatial file, merge 2018 rate columns by FIPS, drop rows with
    missing REQUIRED_COLS, keep only selected columns, and write output.
    """
    if verbose:
        print("\n========================================")
        print(f"Processing: {shp_in}")
        print("========================================")

    gdf = gpd.read_file(shp_in)

    if "FIPS" not in gdf.columns:
        raise KeyError(f"Input file '{shp_in}' must contain a 'FIPS' column.")

    gdf = gdf.copy()
    gdf["FIPS"] = gdf["FIPS"].apply(normalize_fips)
    gdf = gdf.dropna(subset=["FIPS"])
    gdf = gdf.drop_duplicates(subset=["FIPS"])

    # Left join: keep all geometries from the input file
    merged = gdf.merge(rates_df, on="FIPS", how="left")
    merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=gdf.crs)

    # Ensure REQUIRED_COLS exist after merge
    for col in REQUIRED_COLS:
        if col not in merged.columns:
            raise KeyError(f"Required column '{col}' not found after merge for '{shp_in}'.")

    if verbose:
        miss_counts = merged[REQUIRED_COLS].isna().sum()
        miss_pct = (merged[REQUIRED_COLS].isna().mean() * 100).round(2)

        print("[After merge] Missing counts (REQUIRED_COLS):")
        print(miss_counts.to_string())
        print("\n[After merge] Missing percentages (REQUIRED_COLS):")
        print(miss_pct.astype(str).add("%").to_string())

    cleaned = merged.dropna(subset=REQUIRED_COLS)

    if verbose:
        print(f"\nRows before dropna: {len(merged)}, after dropna: {len(cleaned)}")

    # Keep only selected columns that exist
    keep_cols = ["FIPS"] + BASE_COLS + NEW_RATE_COLS + ["geometry"]
    keep_cols = [c for c in keep_cols if c in cleaned.columns]
    final = cleaned[keep_cols].copy()

    if verbose:
        print("\n[Final columns]")
        print(list(final.columns))

    # Write output (driver inferred from extension)
    final.to_file(shp_out)

    if verbose:
        print(f"\nSaved: {shp_out}")

    return final


def main():
    parser = argparse.ArgumentParser(
        description="Merge 2018 rates from Excel into geospatial files by FIPS and export cleaned outputs."
    )
    parser.add_argument("--excel", default="2018Rates1.xlsx", help="Path to the Excel file containing 2018 rates.")
    parser.add_argument(
        "--pairs",
        nargs="+",
        default=[
            "US_HIV_Merged_total.shp:US_HIV_Merged_total_final.shp",
            "DeepSouth_HIV_Merged.shp:DeepSouth_HIV_Merged_final.shp",
            "CBSA_HIV_Merged.shp:CBSA_HIV_Merged_final.shp",
        ],
        help="Input:output pairs, e.g., in1.shp:out1.shp in2.shp:out2.shp",
    )
    parser.add_argument("--quiet", action="store_true", help="Reduce console output.")
    args = parser.parse_args()

    rates = load_rates(args.excel)

    verbose = not args.quiet
    for pair in args.pairs:
        if ":" not in pair:
            raise ValueError(f"Invalid pair '{pair}'. Use 'input:output'.")
        shp_in, shp_out = pair.split(":", 1)
        merge_and_clean(shp_in=shp_in, shp_out=shp_out, rates_df=rates, verbose=verbose)

    if verbose:
        print("\nDone. All files processed and saved.")


if __name__ == "__main__":
    main()

#ModelRunning

In [ ]:
!pip install pysal

In [ ]:
!pip install esda

In [ ]:
#mount google drive
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import os
os.chdir("/content/gdrive/My Drive/Colab Notebooks")

##Disease types

For different diseases, simply modify the dependent variable assignment in the first two codes for the main workflow.

If you are modeling HIV prevalence, use:

STD_Y_COL = 'PercenHIVp'

If you are modeling Gonorrhea rates, use:

STD_Y_COL = 'GonorrheaR'

If you are modeling Chlamydia rates, use:

STD_Y_COL = 'ChlamydiaR'


##Main workflow

In [ ]:
# ============================================
# 0. Imports & Global config
# ============================================

import os
import math
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from joblib import Parallel, delayed

from libpysal.weights import Queen, W
from libpysal import weights
from esda.moran import Moran
from spreg.diagnostics import log_likelihood, aic

from numpy.linalg import inv, slogdet
from scipy.optimize import minimize

from esda.moran import Moran_Local  # optional (not required)
from shapely.geometry import Point

warnings.filterwarnings("ignore")

# ------------------------------
# Global random seed
# ------------------------------
RANDOM_SEED = 7
rng_global = np.random.default_rng(RANDOM_SEED)

# Parallel / production mode
N_JOBS = -1        # -1: all cores; set to 1 if unstable
RUN_PROD = False   # If True, you may increase iterations / B_NULL / B_PERT, etc.

# ------------- Outcome config: switch STD outcome in one line ----------------
# Options: 'PercenHIVp', 'GonorrheaR', 'ChlamydiaR'
STD_Y_COL = 'PercenHIVp'   # <--- change this line only

# ------------------------------
# Output folder
# ------------------------------
OUTDIR = "./US_MAIN_ANALYSIS_OUTPUT"
os.makedirs(OUTDIR, exist_ok=True)

# Mapping to 2018 field names for temporal robustness
TEMPORAL_2018_COL = {
    'PercenHIVp': 'RateHIV2018',
    'GonorrheaR': 'RateGonorrhea2018',
    'ChlamydiaR': 'RateChlamydia2018',
}

# ============================================
# 1. Read US shapefile & basic preprocessing
# ============================================

# 1.1 Read US shapefile (field structure consistent with DeepSouth)
shp_path = "US_HIV_Merged_total_final.shp"

gdf = gpd.read_file(shp_path)
if 'FIPS' not in gdf.columns:
    raise ValueError("FIPS column not found in shapefile.")

# Strip leading zeros from FIPS and drop duplicates
gdf['FIPS'] = gdf['FIPS'].astype(str).str.zfill(5)
gdf = gdf.drop_duplicates(subset=['FIPS']).copy()

# Fix potentially truncated 2018 field names
# (In some systems, shapefile field names can be truncated to 10 chars)
# We handle known mappings here if needed:
rename_map_2018 = {
    'RateHIV201': 'RateHIV2018',
    'RateGonor': 'RateGonorrhea2018',
    'RateChlam': 'RateChlamydia2018',
}
for k, v in rename_map_2018.items():
    if k in gdf.columns and v not in gdf.columns:
        gdf = gdf.rename(columns={k: v})

# Covariates
COVARS = [
    'Poverty',
    'Black',
    'Hispanic',
    'Age1524',
    'Urban',
    'Uninsured',
]

# Drop missing values for the current 2019 outcome + covariates
need_cols = [STD_Y_COL] + COVARS
gdf = gdf.dropna(subset=need_cols).copy()

# State code
if 'STATEFP' in gdf.columns:
    gdf['STATE'] = gdf['STATEFP'].astype(str).str.zfill(2)
elif 'STATE' not in gdf.columns:
    raise ValueError("STATE code not found (expected STATEFP or STATE).")

# 1.2 Simple interannual correlation: 2018 vs 2019 (US)
def _print_temporal_corr(col_2019, col_2018, label):
    if {col_2019, col_2018}.issubset(gdf.columns):
        sub = gdf[[col_2019, col_2018]].dropna()
        if len(sub) > 10:
            corr = sub[col_2019].corr(sub[col_2018])
            print(f"[Temporal Corr] {label}: corr({col_2019},{col_2018}) = {corr:.4f}")
        else:
            print(f"[Temporal Corr] {label}: insufficient overlap after dropna.")
    else:
        print(f"[Temporal Corr] {label}: missing columns.")

y2018_col = TEMPORAL_2018_COL.get(STD_Y_COL, None)
if y2018_col is not None:
    _print_temporal_corr(STD_Y_COL, y2018_col, "US full sample")

# ============================================
# 2. Build W matrices: WP / WS / WQ
# ============================================

# Load precomputed WP / WS from the shapefile (assumed in columns)
# Example: WP and WS are stored as adjacency matrices in external files or as attributes.
# Here we assume WP and WS are read from external CSV/NPY for US main analysis.
# Replace paths if needed:

WP_PATH = "WP_matrix.npy"
WS_PATH = "WS_matrix.npy"

if not os.path.exists(WP_PATH):
    raise ValueError(f"WP matrix not found: {WP_PATH}")
if not os.path.exists(WS_PATH):
    raise ValueError(f"WS matrix not found: {WS_PATH}")

WP_full = np.load(WP_PATH)  # assumed already aligned to gdf ordering OR we will align by IDs
WS_full = np.load(WS_PATH)

# 2.1 Build IDs and alignment index
ids = list(gdf['FIPS'].values)
N = len(ids)

# Sanity check for matrix shapes
if WP_full.shape[0] != WP_full.shape[1] or WS_full.shape[0] != WS_full.shape[1]:
    raise ValueError("WP/WS matrices must be square.")
if WP_full.shape[0] != N or WS_full.shape[0] != N:
    raise ValueError("WP/WS matrices shape does not match gdf sample size. Please align first.")

# 2.2 WP: symmetrize + row-standardize
A = WP_full.astype(float)
A = A + A.T - np.diag(np.diag(A))        # Symmetrize
row_sum = A.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WP = A / row_sum                         # Row-standardize

# 2.3 WQ: queen contiguity (US)
# Ensure geometry is valid
gdf = gdf[gdf.geometry.notnull()].copy()
gdf = gdf.set_index('FIPS')

wQ = Queen.from_dataframe(gdf, ids=list(gdf.index))
# Row-standardize libpysal W object
wQ.transform = 'R'

# 2.4 Align common IDs & order (WP / WS / WQ / df_geo consistent)
# If gdf has dropped some geometries, we need to align matrices again:
ids_geo = list(gdf.index)
idx_map = {f: i for i, f in enumerate(ids)}
keep_idx = [idx_map[f] for f in ids_geo if f in idx_map]
WP = WP[np.ix_(keep_idx, keep_idx)]
WS = WS_full.astype(float)[np.ix_(keep_idx, keep_idx)]
ids = ids_geo
N = len(ids)

# Re-extract matrix form (aligned)
# Convert wQ to full matrix aligned with ids
WQ = wQ.full()[0]

# Aligned coordinates (EPSG:3857)
gdf = gdf.to_crs(epsg=3857)
coords = np.vstack([gdf.geometry.centroid.x.values, gdf.geometry.centroid.y.values]).T

# 2.5 Build outcome & covariates (current STD outcome, 2019)
y = gdf[STD_Y_COL].values.astype(float).reshape(-1, 1)
X_raw = gdf[COVARS].values.astype(float)
X = np.hstack([np.ones((N, 1)), X_raw])   # Add intercept
K = X.shape[1]

# ============================================
# 3. SEM (multi-W) fitting: Stage-1 + Stage-2 LOSO + gate
# ============================================

# 3.1 Core helper functions --------------------------------------------------------------

def ols_fit(y, X):
    """
    Standard OLS fit for baseline:
    - returns beta, residuals, logL, AICc
    """
    N = y.shape[0]
    K = X.shape[1]
    XtX = X.T @ X
    XtX_inv = inv(XtX)
    beta = XtX_inv @ (X.T @ y)
    resid = y - X @ beta
    s2 = float((resid.T @ resid) / N)
    logL = -0.5 * N * (np.log(2 * np.pi * s2) + 1.0)
    AICc = -2 * logL + 2 * K + (2 * K * (K + 1)) / max(N - K - 1, 1)
    return beta, resid, float(logL), float(AICc)

def build_A_matrix(W_list, lambdas, mode='add'):
    """
    Build SEM A matrix = I - Σ λ_k W_k (add) or product (prod)
    """
    N = W_list[0].shape[0]
    I = np.eye(N)
    if mode == 'add':
        S = np.zeros((N, N), dtype=float)
        for Wk, lk in zip(W_list, lambdas):
            S += lk * Wk
        A = I - S
    elif mode == 'prod':
        A = I.copy()
        for Wk, lk in zip(W_list, lambdas):
            A = A @ (I - lk * Wk)
    else:
        raise ValueError("mode must be 'add' or 'prod'.")
    return A

def sem_negloglik(params, y, X, W_list, mode='add'):
    """
    SEM negative log-likelihood for optimization.
    """
    N = y.shape[0]
    K = X.shape[1]
    beta = params[:K].reshape(-1, 1)
    lambdas = params[K:-1]
    sigma2 = float(np.exp(params[-1]))  # log sigma2

    A = build_A_matrix(W_list, lambdas, mode=mode)
    e = y - X @ beta
    Ae = A @ e

    sgn, logdetA = slogdet(A)
    if sgn <= 0:
        return 1e12

    ll = logdetA - 0.5 * N * np.log(2 * np.pi * sigma2) - 0.5 * float((Ae.T @ Ae) / sigma2)
    return -ll

def sem_fit_full(y, X, W_list, mode='add', maxiter=5000):
    """
    Full SEM fit (multi-weight matrices) for Stage-1 + LOSO.
    """
    N = y.shape[0]
    K = X.shape[1]
    beta_ols, resid_ols, logL_ols, _ = ols_fit(y, X)

    # Start: OLS beta + λ=0
    init_lambdas = np.zeros(len(W_list), dtype=float)
    init_logsig2 = np.log(float((resid_ols.T @ resid_ols) / N))
    init = np.concatenate([beta_ols.flatten(), init_lambdas, [init_logsig2]])

    bounds = [(None, None)] * K + [(-0.99, 0.99)] * len(W_list) + [(None, None)]

    opt = minimize(
        sem_negloglik,
        init,
        args=(y, X, W_list, mode),
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': maxiter, 'ftol': 1e-10},
    )
    if not opt.success:
        # fallback: still return best found
        pass

    params = opt.x
    beta_hat = params[:K].reshape(-1, 1)
    lambdas_hat = params[K:-1]
    sigma2_hat = float(np.exp(params[-1]))

    # log-likelihood
    logL = -sem_negloglik(params, y, X, W_list, mode=mode)

    # AICc
    p = K + len(W_list) + 1
    AICc = -2 * logL + 2 * p + (2 * p * (p + 1)) / max(N - p - 1, 1)

    # Approximate standard errors from Hessian (optional)
    se = None
    try:
        if opt.hess_inv is not None:
            Hinv = opt.hess_inv.todense() if hasattr(opt.hess_inv, "todense") else opt.hess_inv
            se = np.sqrt(np.diag(Hinv))
    except Exception:
        se = None

    return {
        'beta': beta_hat,
        'lambdas': lambdas_hat,
        'sigma2': sigma2_hat,
        'logL': float(logL),
        'AICc': float(AICc),
        'se': se,
        'opt_success': opt.success,
        'opt_message': opt.message if hasattr(opt, 'message') else "",
    }

def moran_I(resid, W_full):
    """
    Given residuals and full W (numpy array), compute Moran's I.
    """
    z = resid.flatten()
    z = z - z.mean()
    denom = np.sum(z ** 2)
    if denom <= 0:
        return np.nan
    W = W_full
    Wsum = W.sum()
    if Wsum <= 0:
        return np.nan
    num = z @ (W @ z)
    I = (len(z) / Wsum) * (num / denom)
    return float(I)

def row_standardize(W):
    """
    Row-standardize the weight matrix.
    """
    row_sum = W.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    return W / row_sum

def repair_connectivity_and_rowstd(W_sub, coords_sub, k_nn=1):
    """
    If the within-state WQ submatrix has isolates, add one nearest-neighbor edge to ensure connectivity, then standardize.
    """
    W = W_sub.copy()
    row_sum = W.sum(axis=1)
    iso = np.where(row_sum == 0)[0]
    if len(iso) == 0:
        return row_standardize(W)
    for i in iso:
        d = np.sqrt(np.sum((coords_sub - coords_sub[i]) ** 2, axis=1))
        d[i] = np.inf
        j = int(np.argmin(d))
        W[i, j] = 1.0
        W[j, i] = 1.0
    return row_standardize(W)

def submatrix(M, idx):
    """
    Take submatrix M[idx, idx]
    """
    return M[np.ix_(idx, idx)]

# 3.2 Candidate weight combos: WP / WS / WQ only + WQ-related pairs only -----------------------

def candidate_models():
    """
    Single-channel: WP, WS, WQ
    Pairs: keep only WQ+WP and WQ+WS (exclude WP+WS)
    """
    models = []
    # Single-channel
    models.append(("SEM_add_1_WP", [WP]))
    models.append(("SEM_add_1_WS", [row_standardize(WS)]))
    models.append(("SEM_add_1_WQ", [row_standardize(WQ)]))
    # Two-channel: allow only WQ+WP / WQ+WS
    models.append(("SEM_add_2_WP_WQ", [WP, row_standardize(WQ)]))
    models.append(("SEM_add_2_WS_WQ", [row_standardize(WS), row_standardize(WQ)]))
    return models

# 3.3 Stage-1: US 2019 in-sample ranking --------------------------------------

beta_ols, resid_ols, logL_ols, AICc_ols = ols_fit(y, X)
mi_ols = moran_I(resid_ols, row_standardize(WQ))

stage1_rows = []
models = candidate_models()

# OLS baseline (also compute Moran's I for readability)
stage1_rows.append({
    'model': 'OLS',
    'logL': logL_ols,
    'AICc': AICc_ols,
    'absMI': abs(mi_ols),
    'MI': mi_ols,
    'lambdas': None,
})

# All single/pair SEMs
MAXITER_STAGE1 = 10000 if RUN_PROD else 6000
for name, W_list in models:
    fit = sem_fit_full(y, X, W_list, mode='add', maxiter=MAXITER_STAGE1)
    resid_sem = y - X @ fit['beta']
    A = build_A_matrix(W_list, fit['lambdas'], mode='add')
    u = A @ resid_sem
    mi = moran_I(u, row_standardize(WQ))
    stage1_rows.append({
        'model': name,
        'logL': fit['logL'],
        'AICc': fit['AICc'],
        'absMI': abs(mi),
        'MI': mi,
        'lambdas': fit['lambdas'].copy(),
    })

df_stage1 = pd.DataFrame(stage1_rows).sort_values("AICc").reset_index(drop=True)
df_stage1.to_csv(os.path.join(OUTDIR, f"stage1_insample_rank_{STD_Y_COL}.csv"), index=False)

# 3.4 Stage-2: LOSO (by STATE, for shortlist models)----------------------------

TOP_K_SINGLE = 2
TOP_K_PAIR = 5  # not many pairs; 5 is fine

# shortlist: best single + top-K overall (excluding OLS)
df_nonols = df_stage1[df_stage1['model'] != 'OLS'].copy()
shortlist_models = [df_nonols.iloc[0]['model']]
# add best single among (WP, WS, WQ)
single_names = {"SEM_add_1_WP", "SEM_add_1_WS", "SEM_add_1_WQ"}
df_single = df_nonols[df_nonols['model'].isin(list(single_names))].sort_values("AICc")
if len(df_single) > 0:
    shortlist_models.append(df_single.iloc[0]['model'])

# add top-K overall (excluding duplicates)
for m in df_nonols.head(TOP_K_PAIR + TOP_K_SINGLE)['model'].tolist():
    if m not in shortlist_models:
        shortlist_models.append(m)

shortlist_models = list(dict.fromkeys(shortlist_models))

model_to_W = {name: W_list for name, W_list in models}

def loso_one_fold(state_code, y, X, WQ_full, coords_full, ids_full):
    """
    One fold (a given STATE) leave-one-state-out:
    - fit on the training set;
    - compute logL, AICc, MoranI on the holdout state.
    """
    hold_mask = (gdf.loc[ids_full, 'STATE'].values == state_code)
    if hold_mask.sum() < 10:
        return None  # Skip very small states

    idx_hold = np.where(hold_mask)[0]
    idx_train = np.where(~hold_mask)[0]

    y_train, X_train = y[idx_train], X[idx_train]
    y_hold, X_hold = y[idx_hold], X[idx_hold]

    WQ_hold = submatrix(WQ_full, idx_hold)
    coords_hold = coords_full[idx_hold]
    WQ_hold = repair_connectivity_and_rowstd(WQ_hold, coords_hold, k_nn=1)

    out = {'STATE': state_code}

    # OLS baseline (record logL only; for sanity check)
    beta_ols_tr, resid_ols_tr, logL_ols_tr, AICc_ols_tr = ols_fit(y_train, X_train)
    out['OLS_logL_tr'] = logL_ols_tr

    for m in shortlist_models:
        W_list_full = model_to_W[m]
        W_list_tr = [submatrix(Wk, idx_train) for Wk in W_list_full]
        W_list_tr = [row_standardize(Wk) for Wk in W_list_tr]

        # Train
        fit = sem_fit_full(y_train, X_train, W_list_tr, mode='add',
                           maxiter=8000 if RUN_PROD else 4000)

        # holdout logL
        beta_hat = fit['beta']
        lambdas = fit['lambdas']

        # for holdout residual Moran, use WQ connectivity repaired within holdout
        e_hold = y_hold - X_hold @ beta_hat

        # Need A matrix for holdout with same W structure
        W_list_hold = [submatrix(Wk, idx_hold) for Wk in W_list_full]
        W_list_hold = [row_standardize(Wk) for Wk in W_list_hold]

        A_hold = build_A_matrix(W_list_hold, lambdas, mode='add')
        u_hold = A_hold @ e_hold

        mi_hold = moran_I(u_hold, WQ_hold)

        # approximate holdout log-likelihood under fitted params (pseudo)
        # Use residuals transformed by A_hold and sigma2 from fit
        sigma2 = fit['sigma2']
        sgn, logdetA = slogdet(A_hold)
        if sgn <= 0:
            logL_hold = np.nan
        else:
            logL_hold = logdetA - 0.5 * len(idx_hold) * np.log(2 * np.pi * sigma2) - 0.5 * float((u_hold.T @ u_hold) / sigma2)

        p = X.shape[1] + len(W_list_full) + 1
        AICc_hold = -2 * logL_hold + 2 * p + (2 * p * (p + 1)) / max(len(idx_hold) - p - 1, 1)

        out[f'{m}_logL'] = float(logL_hold)
        out[f'{m}_AICc'] = float(AICc_hold)
        out[f'{m}_absMI'] = abs(float(mi_hold)) if mi_hold == mi_hold else np.nan

    return out

states = sorted(gdf['STATE'].unique().tolist())

# Prepare full objects aligned to ids list
gdf_aligned = gdf.loc[ids].copy()

# Build full arrays used in LOSO
WQ_full = row_standardize(WQ)
coords_full = coords

loso_results = Parallel(n_jobs=N_JOBS)(
    delayed(loso_one_fold)(st, y, X, WQ_full, coords_full, ids) for st in states
)
loso_results = [r for r in loso_results if r is not None]
df_loso = pd.DataFrame(loso_results)
df_loso.to_csv(os.path.join(OUTDIR, f"stage2_loso_{STD_Y_COL}.csv"), index=False)

# 3.5 Summary + Pareto + Forward gate (incl. super_lenient profile)-----------------------

def loso_summary(df_loso, model_names):
    rows = []
    for m in model_names:
        AICc_vals = df_loso[f"{m}_AICc"].values
        MI_vals = df_loso[f"{m}_absMI"].values
        mean_AICc = np.nanmean(AICc_vals)
        mean_absMI = np.nanmean(MI_vals)
        rows.append({'model': m, 'mean_AICc': mean_AICc, 'mean_absMI': mean_absMI})
    return pd.DataFrame(rows).sort_values(["mean_AICc", "mean_absMI"]).reset_index(drop=True)

df_loso_sum = loso_summary(df_loso, shortlist_models)
df_loso_sum.to_csv(os.path.join(OUTDIR, f"loso_summary_{STD_Y_COL}.csv"), index=False)

# Bi-objective: smaller AICc is better; smaller absMI is better
def pareto_front(df, col1='mean_AICc', col2='mean_absMI'):
    df = df.copy()
    df['pareto'] = True
    vals = df[[col1, col2]].values
    for i in range(len(df)):
        for j in range(len(df)):
            if i == j:
                continue
            if (vals[j, 0] <= vals[i, 0]) and (vals[j, 1] <= vals[i, 1]) and ((vals[j, 0] < vals[i, 0]) or (vals[j, 1] < vals[i, 1])):
                df.loc[df.index[i], 'pareto'] = False
                break
    return df

df_pareto = pareto_front(df_loso_sum)
df_pareto.to_csv(os.path.join(OUTDIR, f"loso_pareto_{STD_Y_COL}.csv"), index=False)

# ---- Gate config: strict / lenient / super_lenient ----
GATE_PROFILE        = 'super_lenient'   # ★ US main analysis: recommended super_lenient
TIE_DELTA_REL       = 0.01             # allow 'tie-preference' when relative AICc diff ≤1%

if GATE_PROFILE == 'strict':
    # Emphasize strict CV advantage
    GATE_DAICC_REL_MAX       = -0.01   # require pair improves by at least 1%
    GATE_MI_REL_IMPROVE_MIN  = 0.00    # no required improvement; just not worse
    GATE_WINRATE_MIN         = 0.70    # AICc better in at least 70% of states
elif GATE_PROFILE == 'lenient':
    # Balance statistics and interpretability
    GATE_DAICC_REL_MAX       = 0.00    # pair mean AICc not worse than single
    GATE_MI_REL_IMPROVE_MIN  = 0.05    # at least 5% reduction in |MI|
    GATE_WINRATE_MIN         = 0.50    # better in at least half the states
else:
    # ★ Structure-leaning: prefer pair unless clearly worse
    GATE_DAICC_REL_MAX       = 0.02    # mean AICc can be at most 2% worse
    GATE_MI_REL_IMPROVE_MIN  = -0.10   # allow |MI| up to 10% worse
    GATE_WINRATE_MIN         = 0.10    # only need 10% of states better

def forward_gate(df_loso, best_single, candidate_pair):
    A_single = df_loso[f"{best_single}_AICc"].values
    A_pair   = df_loso[f"{candidate_pair}_AICc"].values
    MI_single = df_loso[f"{best_single}_absMI"].values
    MI_pair   = df_loso[f"{candidate_pair}_absMI"].values

    # ΔAICc (absolute & relative)
    mean_single = np.nanmean(A_single)
    mean_pair   = np.nanmean(A_pair)
    dAICc = mean_pair - mean_single
    dAICc_rel = dAICc / max(abs(mean_single), 1e-9)

    # Relative Moran I improvement (>0 means smaller |MI|; <0 means worse)
    mean_MI_single = np.nanmean(MI_single)
    mean_MI_pair   = np.nanmean(MI_pair)
    mi_rel_improve = (mean_MI_single - mean_MI_pair) / max(mean_MI_single, 1e-9)

    # State-level win_rate: share of states where pair AICc is smaller
    win_rate = np.nanmean((A_pair < A_single).astype(float))

    # Gate decision
    ok = (dAICc_rel <= GATE_DAICC_REL_MAX) and (mi_rel_improve >= GATE_MI_REL_IMPROVE_MIN) and (win_rate >= GATE_WINRATE_MIN)

    # If gate fails but ΔAICc_rel is very small (near tie), still may prefer pair
    tie_pref = (dAICc_rel <= TIE_DELTA_REL)

    return {
        'mean_single': mean_single,
        'mean_pair': mean_pair,
        'dAICc': dAICc,
        'dAICc_rel': dAICc_rel,
        'mean_MI_single': mean_MI_single,
        'mean_MI_pair': mean_MI_pair,
        'mi_rel_improve': mi_rel_improve,
        'win_rate': win_rate,
        'gate_ok': bool(ok),
        'tie_pref': bool(tie_pref),
    }

# Identify best single (by LOSO mean_AICc among singles)
df_single_loso = df_loso_sum[df_loso_sum['model'].isin(list(single_names))].copy()
best_single = df_single_loso.sort_values('mean_AICc').iloc[0]['model']

# Candidate pairs among shortlist
pair_names = [m for m in shortlist_models if m in ("SEM_add_2_WP_WQ", "SEM_add_2_WS_WQ")]
gate_records = []
for p in pair_names:
    rec = forward_gate(df_loso, best_single, p)
    rec['best_single'] = best_single
    rec['pair'] = p
    gate_records.append(rec)
df_gate = pd.DataFrame(gate_records)
df_gate.to_csv(os.path.join(OUTDIR, f"gate_check_{STD_Y_COL}.csv"), index=False)

# Final model selection
final_model_main = best_single
if len(df_gate) > 0:
    # prefer pair that passes gate; if multiple, choose the one with smaller mean_pair
    df_ok = df_gate[df_gate['gate_ok'] == True].copy()
    if len(df_ok) > 0:
        best_pair = df_ok.sort_values('mean_pair').iloc[0]['pair']
        final_model_main = best_pair
    else:
        # tie preference
        df_tie = df_gate[df_gate['tie_pref'] == True].copy()
        if len(df_tie) > 0:
            best_pair = df_tie.sort_values('mean_pair').iloc[0]['pair']
            final_model_main = best_pair

print("[Final] best_single =", best_single, "final_model_main =", final_model_main)

# 3.6 Channel importance (LOSO wAICc weights)----------------------------------

def wAICc_weights(AICc_array):
    """
    Compute wAICc weights from AICc.
    """
    a = np.array(AICc_array, dtype=float)
    amin = np.nanmin(a)
    delta = a - amin
    w = np.exp(-0.5 * delta)
    w = w / np.nansum(w)
    return w

def channel_importance(df_loso, model_names):
    """
    Based on wAICc weights across all candidate SEM models in LOSO,
    summarize the mean importance of each channel.
    """
    # Compute mean AICc per model across states
    mean_AICc = np.array([np.nanmean(df_loso[f"{m}_AICc"].values) for m in model_names])
    w = wAICc_weights(mean_AICc)
    imp = {'WP': 0.0, 'WS': 0.0, 'WQ': 0.0}
    for mi, m in enumerate(model_names):
        if "WP" in m:
            imp['WP'] += w[mi]
        if "WS" in m:
            imp['WS'] += w[mi]
        if "WQ" in m:
            imp['WQ'] += w[mi]
    return imp

imp = channel_importance(df_loso, shortlist_models)
pd.Series(imp).to_csv(os.path.join(OUTDIR, f"channel_importance_{STD_Y_COL}.csv"))

# 3.7 Save main-analysis results --------------------------------------------------------

# Hero model: follow the same logic as the DeepSouth version
hero_model = final_model_main

# ============================================
# 4. Spillover classification (hero vs WQ-only baseline)
# ============================================

#    —— automatically handles hero = single-channel / two-channel
# 4.1 Parse W_list from model name (auto-detect WP / WS / WQ)
def parse_W_list(model_name):
    """
    For example:
      'SEM_add_1_WQ' -> ['WQ']
      'SEM_add_2_WP_WQ' -> ['WP', 'WQ']
    """
    parts = model_name.split("_")
    # find the channels after the count (e.g. ..._1_WQ or ..._2_WP_WQ)
    chs = []
    for p in parts:
        if p in ("WP", "WS", "WQ"):
            chs.append(p)
    return chs

hero_channels = parse_W_list(hero_model)
# hero_model has already been selected by the gate above

# 4.2 Fit hero model & WQ-only model (full data)
# baseline: WQ-only; if no WQ (rare), fall back to OLS linear prediction
def get_W_list_from_channels(channels):
    W_list = []
    for ch in channels:
        if ch == 'WP':
            W_list.append(WP)
        elif ch == 'WS':
            W_list.append(row_standardize(WS))
        elif ch == 'WQ':
            W_list.append(row_standardize(WQ))
    return W_list

W_list_hero = get_W_list_from_channels(hero_channels)

fit_hero = sem_fit_full(y, X, W_list_hero, mode='add', maxiter=12000 if RUN_PROD else 7000)

# baseline WQ-only
fit_wq = None
if 'WQ' in hero_channels or True:
    W_list_wq = [row_standardize(WQ)]
    fit_wq = sem_fit_full(y, X, W_list_wq, mode='add', maxiter=12000 if RUN_PROD else 7000)

def sem_fitted_values(y, X, fit, W_list):
    """
    Compute fitted values for SEM:
    """
    beta_hat = fit['beta']
    return (X @ beta_hat)

# hero fitted values (single or two-channel handled uniformly)
yhat_hero = sem_fitted_values(y, X, fit_hero, W_list_hero)

# baseline fitted values: prefer WQ-only SEM, otherwise OLS linear prediction
if fit_wq is not None:
    yhat_wq = sem_fitted_values(y, X, fit_wq, [row_standardize(WQ)])
else:
    beta_ols_all, _, _, _ = ols_fit(y, X)
    yhat_wq = X @ beta_ols_all

# 4.3 Compute rank shift (hero vs WQ baseline)
rank_hero = pd.Series(yhat_hero.flatten(), index=ids).rank(ascending=False, method='average')
rank_wq   = pd.Series(yhat_wq.flatten(),   index=ids).rank(ascending=False, method='average')
delta_rank = rank_hero - rank_wq

# 4.4 Build spill_type: Q = WQ, B = 'bridging channels' in hero excluding WQ

# Q strength
Q_strength = pd.Series(row_standardize(WQ).sum(axis=1), index=ids)

# bridging channels = hero_channels excluding WQ (may be [WP], [WS], [WP,WS], or empty)
bridging_channels = [ch for ch in hero_channels if ch != 'WQ']

if len(bridging_channels) == 0:
    B_strength = pd.Series(np.zeros(N), index=ids)
else:
    Bmats = []
    for ch in bridging_channels:
        if ch == 'WP':
            Bmats.append(WP)
        elif ch == 'WS':
            Bmats.append(row_standardize(WS))
    # Sum all bridging channels into one 'composite B'
    B_sum = np.zeros((N, N), dtype=float)
    for Bm in Bmats:
        B_sum += Bm
    B_strength = pd.Series(B_sum.sum(axis=1), index=ids)

# Split High / Low by median
Q_med = Q_strength.median()
B_med = B_strength.median()

is_highQ = (Q_strength >= Q_med)
is_highB = (B_strength >= B_med)

# Generalized 'bridging': LowQ & HighB (regardless of whether B is WP, WS, or both)
spill_type = pd.Series(index=ids, dtype=object)
spill_type[(is_highQ) & (is_highB)] = "HighQ_HighB"
spill_type[(is_highQ) & (~is_highB)] = "HighQ_LowB"
spill_type[(~is_highQ) & (is_highB)] = "LowQ_HighB"
spill_type[(~is_highQ) & (~is_highB)] = "LowQ_LowB"

# Record which bridging channels were used
bridging_label = "+".join(bridging_channels) if len(bridging_channels) > 0 else "OnlyQ"

# hero has only WQ (e.g., SEM_add_1_WQ)
if len(bridging_channels) == 0:
    spill_type[:] = "OnlyQ"

# 4.5 Write shapefile + CSV (US prefix + outcome_tag)

out_gdf = gdf_aligned.copy()
out_gdf['delta_rank'] = delta_rank.loc[out_gdf.index].values
out_gdf['spill_type'] = spill_type.loc[out_gdf.index].values
out_gdf['is_bridging'] = ((~is_highQ) & (is_highB)).loc[out_gdf.index].values
out_gdf['bridging_ch'] = bridging_label

out_shp = os.path.join(OUTDIR, f"US_spillover_{STD_Y_COL}.shp")
out_csv = os.path.join(OUTDIR, f"US_spillover_{STD_Y_COL}.csv")
out_gdf.to_file(out_shp)
out_gdf.drop(columns='geometry').to_csv(out_csv, index=True)

# ============================================
# 5. Rewiring null test (degree-preserving row-wise rewiring)
# ============================================

B_NULL = 200 if RUN_PROD else 40
MAXITER_NULL = 4000 if RUN_PROD else 2500

# 5.1 Choose the 'target channel' to rewire:
#     Priority: bridging channels in hero excluding WQ; if none, can only rewire WQ/single-channel
if len(bridging_channels) > 0:
    target_ch = bridging_channels[0]  # pick first
else:
    target_ch = "WQ"

# Original hero model AICc and |Moran I|
hero_AICc = fit_hero['AICc']
hero_resid = y - X @ fit_hero['beta']
A_hero = build_A_matrix(W_list_hero, fit_hero['lambdas'], mode='add')
u_hero = A_hero @ hero_resid
hero_MI = moran_I(u_hero, row_standardize(WQ))

# 5.2 Fast SEM fit (for nulls)
def sem_fit_fast(y, X, W_list, maxiter=2500):
    return sem_fit_full(y, X, W_list, mode='add', maxiter=maxiter)

# 5.3 Rewiring: shuffle nonzero weights within each row (degree-preserving)
def rowwise_rewire(W):
    """
    Degree-preserving rewiring for a given row-standardized matrix:
      - within each row, permute weights among nonzero positions only
      - keep island rows (all zeros) unchanged
    """
    W2 = W.copy()
    for i in range(W2.shape[0]):
        nz = np.where(W2[i, :] > 0)[0]
        if len(nz) <= 1:
            continue
        vals = W2[i, nz].copy()
        rng_global.shuffle(vals)
        W2[i, nz] = vals
    return W2

# 5.4 Generate rewired nulls and fit the 'same-structure hero'
def hero_fit_with_rewired(WP_in, WS_in, WQ_in, hero_channels, maxiter=3000):
    W_list = []
    for ch in hero_channels:
        if ch == "WP":
            W_list.append(WP_in)
        elif ch == "WS":
            W_list.append(WS_in)
        elif ch == "WQ":
            W_list.append(WQ_in)
    fit = sem_fit_fast(y, X, W_list, maxiter=maxiter)
    resid = y - X @ fit['beta']
    A = build_A_matrix(W_list, fit['lambdas'], mode='add')
    u = A @ resid
    mi = moran_I(u, row_standardize(WQ_in))
    return fit['AICc'], abs(mi)

null_AICc = []
null_absMI = []

for b in range(B_NULL):
    WP_b = WP.copy()
    WS_b = row_standardize(WS).copy()
    WQ_b = row_standardize(WQ).copy()

    if target_ch == "WP":
        WP_b = rowwise_rewire(WP_b)
    elif target_ch == "WS":
        WS_b = rowwise_rewire(WS_b)
    else:
        WQ_b = rowwise_rewire(WQ_b)

    # Row-wise rewiring on target_ch; keep other channels unchanged
    # Use the channel combination of the 'same hero_model' name (but one channel has been rewired)
    AICc_b, absMI_b = hero_fit_with_rewired(WP_b, WS_b, WQ_b, hero_channels, maxiter=MAXITER_NULL)
    null_AICc.append(AICc_b)
    null_absMI.append(absMI_b)

df_null = pd.DataFrame({'AICc_null': null_AICc, 'absMI_null': null_absMI})
df_null.to_csv(os.path.join(OUTDIR, f"rewiring_null_{STD_Y_COL}.csv"), index=False)

# ============================================
# 6. Perturbation tests (WP/WS only; keep WQ fixed)
# ============================================

#    —— same logic as DeepSouth: perturb WP/WS only; keep Q fixed
B_PERT = 200 if RUN_PROD else 40

# 6.1 Pre-construct Moran object for WQ (no perturbation)
WQ_base = row_standardize(WQ)

# 6.2 Fast SEM (lower optimization precision is fine)
MAXITER_PERT = 4500 if RUN_PROD else 2500

# 6.3 FAST Stage-1: only check WQ-only / WP+WQ / WS+WQ
fast_candidates = []
fast_candidates.append(("SEM_add_1_WQ", ["WQ"]))
if "SEM_add_2_WP_WQ" in model_to_W:
    # Add WP+WQ / WS+WQ if present
    fast_candidates.append(("SEM_add_2_WP_WQ", ["WP", "WQ"]))
if "SEM_add_2_WS_WQ" in model_to_W:
    fast_candidates.append(("SEM_add_2_WS_WQ", ["WS", "WQ"]))

def perturb_matrix(W, drop_prob=0.02, mult_noise=0.05):
    Wp = W.copy()
    # Randomly drop edges (symmetric)
    mask = rng_global.random(Wp.shape) < drop_prob
    Wp[mask] = 0.0
    # Multiplicative noise on edge weights
    noise = 1.0 + mult_noise * rng_global.normal(size=Wp.shape)
    Wp = Wp * noise
    Wp = (Wp + Wp.T) / 2.0
    Wp[Wp < 0] = 0.0
    Wp = row_standardize(Wp)  # Row-standardize
    return Wp

# 6.5 Main loop
pert_records = []
for b in range(B_PERT):
    WP_b = perturb_matrix(WP, drop_prob=0.02, mult_noise=0.05)
    WS_b = perturb_matrix(row_standardize(WS), drop_prob=0.02, mult_noise=0.05)
    WQ_b = WQ_base

    for name, chs in fast_candidates:
        W_list = []
        for ch in chs:
            if ch == "WP":
                W_list.append(WP_b)
            elif ch == "WS":
                W_list.append(WS_b)
            else:
                W_list.append(WQ_b)

        fit = sem_fit_fast(y, X, W_list, maxiter=MAXITER_PERT)
        resid = y - X @ fit['beta']
        A = build_A_matrix(W_list, fit['lambdas'], mode='add')
        u = A @ resid
        mi = moran_I(u, WQ_b)
        pert_records.append({
            'b': b,
            'model': name,
            'AICc': fit['AICc'],
            'absMI': abs(mi),
        })

df_pert = pd.DataFrame(pert_records)
df_pert.to_csv(os.path.join(OUTDIR, f"perturb_tests_{STD_Y_COL}.csv"), index=False)

# Model frequency
# Channel inclusion frequency
# (These can be computed downstream from outputs)

# ============================================
# 7. Temporal robustness: apply 2019 hero structure to 2018 outcomes
# ============================================

#    —— hero structure is automatically reused, adapted to current outcome
# 7.1 Parse hero channel structure (reuse the structure of 2019 final_model_main)
hero_chs = parse_W_list(final_model_main)

# 7.2 Intersection sample: complete 2018 & 2019 outcomes + covariates
y2018_col = TEMPORAL_2018_COL.get(STD_Y_COL, None)
if y2018_col is None or y2018_col not in gdf_aligned.columns:
    print("[Temporal] 2018 column not available; skip temporal robustness.")
else:
    gdf_tmp = gdf_aligned.copy()
    need_cols_tmp = [STD_Y_COL, y2018_col] + COVARS
    gdf_tmp = gdf_tmp.dropna(subset=need_cols_tmp).copy()
    ids_tmp = list(gdf_tmp.index)
    idx_map2 = {f: i for i, f in enumerate(ids)}
    keep_idx2 = [idx_map2[f] for f in ids_tmp]

    y2019_t = gdf_tmp[STD_Y_COL].values.astype(float).reshape(-1, 1)
    y2018_t = gdf_tmp[y2018_col].values.astype(float).reshape(-1, 1)
    X_t = np.hstack([np.ones((len(ids_tmp), 1)), gdf_tmp[COVARS].values.astype(float)])

    # W_list for hero on the intersection sample
    def slice_W_for_ids(W, keep_idx):
        return W[np.ix_(keep_idx, keep_idx)]

    WP_t = slice_W_for_ids(WP, keep_idx2)
    WS_t = slice_W_for_ids(row_standardize(WS), keep_idx2)
    WQ_t = slice_W_for_ids(row_standardize(WQ), keep_idx2)

    W_list_t = []
    for ch in hero_chs:
        if ch == "WP":
            W_list_t.append(row_standardize(WP_t))
        elif ch == "WS":
            W_list_t.append(row_standardize(WS_t))
        else:
            W_list_t.append(row_standardize(WQ_t))

    # Fit for 2019 and 2018 with same hero structure but re-estimate parameters
    fit_2019 = sem_fit_full(y2019_t, X_t, W_list_t, mode='add', maxiter=12000 if RUN_PROD else 7000)
    fit_2018 = sem_fit_full(y2018_t, X_t, W_list_t, mode='add', maxiter=12000 if RUN_PROD else 7000)

    # 3) Hero SEM (reuse 2019 channel structure)
    temporal_out = pd.DataFrame({
        'year': [2019, 2018],
        'logL': [fit_2019['logL'], fit_2018['logL']],
        'AICc': [fit_2019['AICc'], fit_2018['AICc']],
        'lambdas': [",".join([f"{x:.6f}" for x in fit_2019['lambdas']]),
                    ",".join([f"{x:.6f}" for x in fit_2018['lambdas']])],
    })

    # 7.4 Interannual comparison of hero λ (also exports hero λ)
    temporal_out.to_csv(os.path.join(OUTDIR, f"temporal_hero_{STD_Y_COL}.csv"), index=False)

    # 7.5 Save results (filename includes outcome_tag)
    print("[Temporal] saved temporal_hero CSV.")

# ============================================
# 8. Mapping figures: basemap + spillover types + delta_rank
# ============================================

#    —— Unified basemap: boundaries + missing-data background + standard legend
plt.rcParams['figure.facecolor'] = "white"  # Set white background

# 8.1 Basemap: US counties + missing flag (based on current y_col)
def build_us_basemap(shp_path, y_col):
    """
    Read US county shapefile and build a unified basemap:
    - clean FIPS
    - drop missing geometry
    - has_data: whether the current outcome is missing
    - project to an appropriate UTM CRS
    """
    g0 = gpd.read_file(shp_path)
    if 'FIPS' not in g0.columns:
        raise ValueError("FIPS column not found.")
    g0['FIPS'] = g0['FIPS'].astype(str).str.zfill(5)
    g0 = g0.dropna(subset=['geometry']).copy()
    if y_col in g0.columns:
        g0['has_data'] = ~g0[y_col].isna()
    else:
        g0['has_data'] = True

    # Choose UTM zone based on overall centroid longitude (assume original CRS is lon/lat)
    g0_ll = g0.to_crs(epsg=4326)
    lon = g0_ll.geometry.centroid.x.mean()
    utm_zone = int((lon + 180) / 6) + 1
    utm_crs = f"EPSG:326{utm_zone:02d}"   # Northern hemisphere UTM
    g0 = g0.to_crs(utm_crs)
    return g0

# Use shp_path (read above) and current y_col
g_base = build_us_basemap(shp_path, STD_Y_COL)

# Spillover shapefile for the model sample
g_spill = gpd.read_file(out_shp).to_crs(g_base.crs)

# Counties actually used by the model (to distinguish background: has outcome but excluded)
model_ids = set(g_spill['FIPS'].astype(str).str.zfill(5).tolist())
g_base['in_model'] = g_base['FIPS'].astype(str).str.zfill(5).apply(lambda x: x in model_ids)

# Background colors
missing_color   = "#4f4f4f"  # dark gray: current outcome missing
nonmodel_color  = "#d9d9d9"  # light gray: has outcome but not in SEM model

def draw_basemap(ax, g_base):
    """
    Draw the unified basemap in the given CRS:
    - thick black boundary
    - missing counties (has_data=False) in dark gray
    - counties with outcome but not in model sample in light gray
    """
    # Outer boundary
    g_base.boundary.plot(ax=ax, linewidth=0.2, edgecolor="black")

    # Counties missing the outcome
    g_base.loc[~g_base['has_data']].plot(ax=ax, color=missing_color, linewidth=0)

    # Counties with outcome but not in model sample
    g_base.loc[(g_base['has_data']) & (~g_base['in_model'])].plot(ax=ax, color=nonmodel_color, linewidth=0)

# 8.2 spillover type column + colors (dynamically supports OnlyQ)
if 'spill_type' not in g_spill.columns:
    raise ValueError("Cannot find spill_type column; please check shp field name.")

# Backward-compatible naming (...P → ...B)
g_spill['spill_type'] = g_spill['spill_type'].replace({
    "HighQ_HighP": "HighQ_HighB",
    "HighQ_LowP": "HighQ_LowB",
    "LowQ_HighP": "LowQ_HighB",
    "LowQ_LowP": "LowQ_LowB",
})

if (g_spill['spill_type'] == "OnlyQ").any():
    # Case 1: hero has only WQ → spill_type = 'OnlyQ'
    color_map = {"OnlyQ": "#3182bd"}  # blue
else:
    # Case 2: four Q×B combinations
    color_map = {
        "HighQ_HighB": "#d73027",  # deep red
        "HighQ_LowB":  "#fc8d59",  # light red
        "LowQ_HighB":  "#4575b4",  # blue
        "LowQ_LowB":   "#fee090",  # light yellow; distinct from light-gray background
    }

# bridging channel label (for title)
title_bridge = bridging_label

# 8.3 Figure A: four spillover types + missing/background legend
figA, axA = plt.subplots(1, 1, figsize=(12, 7))

# Draw the unified basemap first
draw_basemap(axA, g_base)

# Then overlay spillover categories
for k, c in color_map.items():
    g_spill.loc[g_spill['spill_type'] == k].plot(ax=axA, color=c, linewidth=0)

axA.set_axis_off()
axA.set_title(f"Spillover types ({STD_Y_COL}) | Hero={hero_model} | Bridging={title_bridge}")

# Legend: spillover types + missing/not-modeled
import matplotlib.patches as mpatches
handles = []
for k, c in color_map.items():
    handles.append(mpatches.Patch(color=c, label=k))
handles.append(mpatches.Patch(color=missing_color, label="Missing outcome"))
handles.append(mpatches.Patch(color=nonmodel_color, label="Has outcome, not in model"))
axA.legend(handles=handles, loc="lower left", frameon=True, fontsize=9)

figA.tight_layout()
figA.savefig(os.path.join(OUTDIR, f"FigA_spillover_{STD_Y_COL}.png"), dpi=200)

# Background explanation
# 8.4 Prepare data for rank plots: is_bridging / delta_rank

# bridging flag column (compatible with is_bridgin)
if 'is_bridging' not in g_spill.columns and 'is_bridgin' in g_spill.columns:
    g_spill['is_bridging'] = g_spill['is_bridgin']

if 'is_bridging' not in g_spill.columns:
    raise ValueError("Cannot find is_bridging / is_bridgin field; please check shp.")

# delta_rank column
if 'delta_rank' not in g_spill.columns:
    raise ValueError("Cannot find delta_rank field; please check shp.")

# 8.5 Figure B1: Δrank for all bridging counties (symmetric color scale) + background
figB1, axB1 = plt.subplots(1, 1, figsize=(12, 7))

# Basemap
draw_basemap(axB1, g_base)

# All bridging counties
g_bridge = g_spill[g_spill['is_bridging'] == True].copy()
if len(g_bridge) == 0:
    print("Warning: no counties with is_bridge == True; please check the field.")
else:
    vmax = np.nanmax(np.abs(g_bridge['delta_rank'].values))
    g_bridge.plot(ax=axB1,
                  column='delta_rank',
                  cmap='coolwarm',
                  linewidth=0,
                  legend=True,
                  vmin=-vmax, vmax=vmax,   # ★ symmetric color scale
                  missing_kwds={"color": missing_color})
axB1.set_axis_off()
axB1.set_title(f"Δrank (Hero - WQ) for bridging counties | {STD_Y_COL}")

# Small legend: label background only (avoid mixing with the colorbar)
handles_bg = [
    mpatches.Patch(color=missing_color, label="Missing outcome"),
    mpatches.Patch(color=nonmodel_color, label="Has outcome, not in model"),
]
axB1.legend(handles=handles_bg, loc="lower left", frameon=True, fontsize=9)

figB1.tight_layout()
figB1.savefig(os.path.join(OUTDIR, f"FigB1_delta_rank_bridging_{STD_Y_COL}.png"), dpi=200)

# 8.6 Figure B2: 'typical bridging counties' with Δrank ≥ THRESH + background
THRESH = 500
g_top_bridge = g_bridge[g_bridge['delta_rank'] >= THRESH].copy()
print(f"Number of bridging counties with Δrank ≥ {THRESH} (US):", len(g_top_bridge))

figB2, axB2 = plt.subplots(1, 1, figsize=(12, 7))

# Basemap
draw_basemap(axB2, g_base)

# All bridging counties in light blue
g_bridge.plot(ax=axB2, color="#9ecae1", linewidth=0)

# High-Δrank focus counties in a red gradient
if len(g_top_bridge) > 0:
    g_top_bridge.plot(ax=axB2, color="#de2d26", linewidth=0)

axB2.set_axis_off()
axB2.set_title(f"Typical bridging counties (Δrank ≥ {THRESH}) | {STD_Y_COL}")

# Background legend
handles_bg2 = [
    mpatches.Patch(color=missing_color, label="Missing outcome"),
    mpatches.Patch(color=nonmodel_color, label="Has outcome, not in model"),
    mpatches.Patch(color="#9ecae1", label="Bridging counties"),
    mpatches.Patch(color="#de2d26", label=f"Δrank ≥ {THRESH}"),
]
axB2.legend(handles=handles_bg2, loc="lower left", frameon=True, fontsize=9)

figB2.tight_layout()
figB2.savefig(os.path.join(OUTDIR, f"FigB2_top_bridging_{STD_Y_COL}.png"), dpi=200)

print("[Done] US main analysis pipeline complete.")

###Add or prod

In [ ]:
# ============================================
# 0. Imports & Global configs
# ============================================
import numpy as np
import pandas as pd
import geopandas as gpd

from numpy.linalg import slogdet
from scipy.optimize import minimize

from esda.moran import Moran
from libpysal.weights import util, W, Queen
from joblib import Parallel, delayed

np.seterr(all='ignore')

RANDOM_SEED = 7
rng_global = np.random.default_rng(RANDOM_SEED)

# Parallel configuration
N_JOBS = -1   # -1 uses all cores; set to 1 if unstable

# ------------- Outcome config: switch STD outcome with one line ----------------
# Options: 'PercenHIVp', 'GonorrheaR', 'ChlamydiaR'
y_col = 'PercenHIVp'

OUTCOME_TAG = {
    'PercenHIVp': 'HIV',
    'GonorrheaR': 'Gonorrhea',
    'ChlamydiaR': 'Chlamydia'
}
outcome_tag = OUTCOME_TAG.get(y_col, y_col)

# Temporal robustness mapping (we only need outcome_tag here; no need to actually run 2018)
TEMPORAL_2018_COL = {
    'PercenHIVp': 'RateHIV2018',
    'GonorrheaR': 'RateGonorrhea2018',
    'ChlamydiaR': 'RateChlamydia2018'
}
outcome_2018_col = TEMPORAL_2018_COL.get(y_col, None)

# ============================================
# 1. Read US shapefile & basic preprocessing
# ============================================

shp_path = "US_HIV_Merged_total_final.shp"
gdf = gpd.read_file(shp_path)

if 'FIPS' not in gdf.columns:
    raise KeyError("Shapefile needs a 'FIPS' field.")

# Strip leading zeros from FIPS and drop duplicates
gdf['FIPS'] = gdf['FIPS'].astype(str).str.lstrip('0')
gdf = gdf.drop_duplicates(subset=['FIPS'])

# Fix potentially truncated 2018 field names
rename_map = {
    'RateChlamy': 'RateChlamydia2018',
    'RateGonorr': 'RateGonorrhea2018',
    'RateHIV201': 'RateHIV2018'
}
for short, long_name in rename_map.items():
    if (short in gdf.columns) and (long_name not in gdf.columns):
        gdf = gdf.rename(columns={short: long_name})

# Covariates
X_cols = [
    'Population','UrbanRural','Female','Old','Black',
    'Noinsuranc','Poverty','crime16to1','Dissimilar'
]

cols_keep = [
    'FIPS',
    'GonorrheaR','ChlamydiaR','PercenHIVp',
    *X_cols,
    'RateChlamydia2018','RateGonorrhea2018','RateHIV2018',
    'geometry'
]

df_geo = gdf[cols_keep].copy()

# Drop missing values for the current 2019 outcome + covariates (coarse filter first)
df_geo = df_geo.dropna(subset=[y_col] + X_cols)

# State code
def fips2state(x):
    return str(x).zfill(5)[:2]

df_geo['STATE'] = df_geo['FIPS'].apply(fips2state)
df_geo = gpd.GeoDataFrame(df_geo, geometry='geometry', crs=gdf.crs)

# ============================================
# 2. Build spatial weights & design matrix (US)
# ============================================

# 2.1 WP：Twitter PCI mobility
pci_path = "US_County_PCI_2019.csv"
pci = pd.read_csv(pci_path)

pci['place_i'] = pci['place_i'].astype(str).str.lstrip('0')
pci['place_j'] = pci['place_j'].astype(str).str.lstrip('0')

pci_f = pci[
    pci['place_i'].isin(df_geo['FIPS']) &
    pci['place_j'].isin(df_geo['FIPS'])
].copy()

A_pci = pci_f.pivot_table(
    index='place_i', columns='place_j',
    values='pci', fill_value=0
)
np.fill_diagonal(A_pci.values, 0.0)
A = A_pci.values.astype(float)
A = A + A.T - np.diag(np.diag(A))  # Symmetrize
row_sum = A.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WP = A / row_sum
wP = util.full2W(WP, ids=list(A_pci.index))

# 2.2 WS：Facebook SCI social ties
sci_path = "processed_sci_summary.csv"
sci = pd.read_csv(sci_path)

sci['user_loc'] = sci['user_loc'].astype(str).str.lstrip('0')
sci['tfr_loc']  = sci['tfr_loc'].astype(str).str.lstrip('0')

sci_f = sci[
    sci['user_loc'].isin(df_geo['FIPS']) &
    sci['tfr_loc'].isin(df_geo['FIPS'])
].copy()

A_sci = sci_f.pivot_table(
    index='user_loc', columns='tfr_loc',
    values='tscaled_sci', fill_value=0
)
np.fill_diagonal(A_sci.values, 0.0)
B = A_sci.values.astype(float)
B = B + B.T - np.diag(np.diag(B))
row_sum = B.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WS = B / row_sum
wS = util.full2W(WS, ids=list(A_sci.index))

# 2.3 WQ: Queen contiguity (US-wide)
g3857 = df_geo.to_crs(epsg=3857)
wQ = Queen.from_dataframe(g3857, ids=list(df_geo['FIPS']))
wQ.transform = 'R'
WQ, _ = wQ.full()

# 2.4 Align common ID set & order (WP / WS / WQ / df_geo unified)

common = (
    set(df_geo['FIPS']) &
    set(wP.id_order) &
    set(wS.id_order) &
    set(wQ.id_order)
)

def subset_W(w: W, keep_ids):
    keep_ids = list(dict.fromkeys([str(x) for x in keep_ids]))
    keep = set(keep_ids)
    new_neighbors, new_weights = {}, {}
    for i in w.id_order:
        if i not in keep:
            continue
        nbs = w.neighbors.get(i, [])
        wts = w.weights.get(i, [])
        kn, kw = [], []
        for nb, wt in zip(nbs, wts):
            if nb in keep:
                kn.append(nb); kw.append(wt)
        new_neighbors[i] = kn
        new_weights[i] = kw
    return W(new_neighbors, new_weights, ids=keep_ids)

df_geo = df_geo[df_geo['FIPS'].isin(common)].copy()
wP = subset_W(wP, common)
wS = subset_W(wS, common)
wQ = subset_W(wQ, common)

df_geo = df_geo.set_index('FIPS').loc[wQ.id_order].reset_index()
IDs   = list(wQ.id_order)
STATE = df_geo['STATE'].values

# Convert to matrix form again (after alignment)
WP, _ = wP.full()
WS, _ = wS.full()
WQ, _ = wQ.full()

# Aligned coordinates (EPSG:3857)
XY_all = np.vstack([
    g3857.set_index('FIPS').loc[IDs].geometry.centroid.x.values,
    g3857.set_index('FIPS').loc[IDs].geometry.centroid.y.values
]).T

# 2.5 Build outcome & covariates (current STD outcome, 2019)
df = df_geo[['FIPS', y_col] + X_cols + ['STATE','geometry']].dropna().copy()

y = df[y_col].values
X_raw = df[X_cols].values
N = X_raw.shape[0]
X = np.hstack([np.ones((N, 1)), X_raw])   # Add intercept
K = X.shape[1]

# ============================================
# 3. SEM core helpers + Stage-1 & Stage-2 + Gate
# ============================================

# 3.1 Core helper functions --------------------------------------------------------------

def fit_ols(y, X):
    """
    Standard OLS fit for the baseline:
    - returns beta, residuals, logL, AICc
    """
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X.dot(beta)
    sse = float(e.T.dot(e))
    N_, K_ = X.shape
    df_ = N_ - K_
    sigma2 = sse / df_
    logL = -(N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    AIC  = -2*logL + 2*K_
    AICc = (AIC + (2*K_*(K_+1)) / (N_-K_-1)
            if (N_-K_-1) > 0 else np.nan)
    return {
        'model': 'OLS',
        'beta': beta,
        'logL': logL,
        'AICc': AICc,
        'resid': e
    }


def build_A_and_logdet(lam, W_list, combo='add'):
    """
    Build the SEM A matrix = I - Σ λ_k W_k (add) or a matrix product chain (prod)
    """
    N_ = W_list[0].shape[0]
    I = np.eye(N_)

    if combo == 'add':
        A = I.copy()
        for i, Wk in enumerate(W_list):
            A -= lam[i] * Wk
        sign, logdetA = slogdet(A)
        if sign <= 0:
            return None, None
        return A, logdetA

    elif combo == 'prod':
        A = I.copy()
        logdet_sum = 0.0
        for i, Wk in enumerate(W_list):
            Mi = I - lam[i] * Wk
            sign_i, logdet_i = slogdet(Mi)
            if sign_i <= 0:
                return None, None
            logdet_sum += logdet_i
            A = Mi @ A
        return A, logdet_sum

    else:
        raise ValueError("combo must be 'add' or 'prod'")


def neglog_sem(params, y, X, W_list, combo='add'):
    """
    SEM negative log-likelihood (for optimization).
    params = [λ_1,...,λ_m, β_0,...,β_{K-1}]
    """
    N_, K_ = X.shape
    m = len(W_list)
    lam = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(lam, W_list, combo=combo)
    if A is None:
        return 1e12

    e = y - X.dot(beta)
    v = A.dot(e)
    sse = float(v.T.dot(v))

    df_ = N_ - (K_ + m)
    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_
    if sigma2 <= 0:
        return 1e12

    logL = logdetA - (N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    return -logL


def fit_sem_multi(y, X, W_list, method='L-BFGS-B', combo='add'):
    """
    Full SEM fit (multi-weight-matrix), used for Stage-1 / LOSO / Add vs Prod。
    """
    N_, K_ = X.shape
    m = len(W_list)

    # Initial values: OLS beta + lambda=0
    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
    x0 = np.r_[np.zeros(m), beta0]
    bounds = [(-0.99, 0.99)]*m + [(-np.inf, np.inf)]*K_

    res = minimize(
        neglog_sem, x0,
        args=(y, X, W_list, combo),
        method=method,
        bounds=bounds,
        options={'maxiter': 200, 'ftol': 1e-4, 'gtol': 1e-4}
    )

    params = res.x
    lam = params[:m]
    beta = params[m:]
    logL = -res.fun
    p = m + K_
    AIC  = -2*logL + 2*p
    AICc = AIC + (2*p*(p+1))/(N_-p-1) if (N_-p-1) > 0 else np.nan

    A, _ = build_A_and_logdet(lam, W_list, combo=combo)
    v = A.dot(y - X.dot(beta))

    # Hessian-based approximate standard errors (optional)
    if hasattr(res, 'hess_inv'):
        try:
            Hin = res.hess_inv.todense()
        except Exception:
            Hin = np.asarray(res.hess_inv)
        se = np.sqrt(np.diag(Hin)) if Hin.shape == (p, p) else np.full(p, np.nan)
    else:
        se = np.full(p, np.nan)

    return {
        'model': f'SEM_{combo}_{len(W_list)}W',
        'lam': lam,
        'beta': beta,
        'logL': logL,
        'AICc': AICc,
        'resid': v,
        'combo': combo,
        'success': res.success,
        'se': se
    }


def moran_I_val_from_array(resid, W_array, ids):
    """
    Given residuals and full W (numpy array), compute Moran's I.
    """
    w_obj = util.full2W(W_array, ids=ids)
    mi = Moran(resid, w_obj, permutations=0)
    return mi.I, mi.p_norm


def row_standardize(A):
    """
    Row-standardize the weight matrix.
    """
    R = A.astype(float)
    rs = R.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1.0
    return R / rs


def ensure_connected_within_state(W_sub, XY_sub):
    """
    If a within-state WQ submatrix has isolates, connect each isolate once to its nearest neighbor, then re-standardize.
    """
    R = W_sub.copy()
    deg = R.sum(axis=1)

    if np.any(deg == 0):
        D = np.linalg.norm(
            XY_sub[:, None, :] - XY_sub[None, :, :], axis=2
        )
        np.fill_diagonal(D, np.inf)
        lonely = np.where(deg == 0)[0]
        for i in lonely:
            j = int(D[i].argmin())
            if np.isfinite(D[i, j]):
                R[i, j] = 1.0
                R[j, i] = 1.0
        R = row_standardize(R)

    return R


def submatrix(M, idx):
    """
    Take a submatrix M[idx, idx]
    """
    return M[np.ix_(idx, idx)]

# 3.2 Candidate weight combos: WP / WS / WQ only + WQ-anchored pairs only -----------------------

W_dict = {'WP': WP, 'WS': WS, 'WQ': WQ}

def make_tasks_add_single_pair(W_dict_local):
    """
    Single-channel: WP, WS, WQ
    Pairs: keep only WQ+WP and WQ+WS (exclude WP+WS)
    """
    keys = list(W_dict_local.keys())
    tasks = []
    # Single-channel
    for k in keys:
        tasks.append((f'SEM_add_1_{k}', [W_dict_local[k]], 'add'))

    # Two-channel: allow only WQ+WP / WQ+WS
    if ('WP' in keys) and ('WQ' in keys):
        tasks.append(('SEM_add_2_WP_WQ',
                      [W_dict_local['WP'], W_dict_local['WQ']],
                      'add'))
    if ('WS' in keys) and ('WQ' in keys):
        tasks.append(('SEM_add_2_WS_WQ',
                      [W_dict_local['WS'], W_dict_local['WQ']],
                      'add'))
    return tasks

TASKS_MAIN_ADD = make_tasks_add_single_pair(W_dict)

# 3.3 Stage-1: US 2019 in-sample ranking --------------------------------------

base_rows = []

# OLS baseline (for readability; also compute Moran's I)
ols = fit_ols(y, X)
I0, p0 = moran_I_val_from_array(ols['resid'], WQ, ids=IDs)
base_rows.append(['OLS', '-', 'OLS',
                  ols['logL'], ols['AICc'], I0, p0])

# All single-/two-channel SEMs
for name, Wset, combo in TASKS_MAIN_ADD:
    res = fit_sem_multi(y, X, Wset, combo=combo)
    I, pI = moran_I_val_from_array(res['resid'], WQ, ids=IDs)
    wmix = ';'.join(name.split('_')[3:])
    base_rows.append([name, wmix, combo,
                      res['logL'], res['AICc'], I, pI])

base_df = pd.DataFrame(
    base_rows,
    columns=['Model', 'W_mix', 'Combo', 'logL', 'AICc', 'MoranI', 'p_norm']
)
base_df = base_df.sort_values(['Combo', 'AICc'])

print(f"\n=== Stage-1: In-sample ranking for {outcome_tag} 2019 (US) ===")
print(base_df.head(10))

# 3.4 Stage-2: LOSO (by STATE, on shortlist models)----------------------------

stage1_sem = base_df[(base_df['Model'] != 'OLS') &
                     (base_df['Combo'] == 'add')].copy()

single_pool = stage1_sem[stage1_sem['Model'].str.match(r'^SEM_add_1_')]
best_single_name = single_pool.nsmallest(1, 'AICc')['Model'].iloc[0]

TOP_K_PAIR = 5  # Not many pairs here; 5 is sufficient
pair_pool = stage1_sem[stage1_sem['Model'].str.match(r'^SEM_add_2_')]
pair_shortlist = pair_pool.nsmallest(TOP_K_PAIR, 'AICc')['Model'].tolist()

stage2_models = [best_single_name] + pair_shortlist

print(f"\n[Stage-1] Shortlist for LOSO (best single + TOP_{TOP_K_PAIR} pairs):")
print(stage2_models)


def _loso_state_worker(st, IDs_all, df_all, y_all, X_all, tasks,
                       WQ_array, XY_all_, K_):
    """
    One fold (one STATE) leave-one-state-out:
    - fit on the training set;
    - evaluate logL, AICc, MoranI on the holdout state.
    """
    hold_ids = df_all.loc[df_all['STATE'] == st, 'FIPS'].tolist()
    hold_set = set(hold_ids)

    hold_idx = np.array(
        [i for i, idd in enumerate(IDs_all) if idd in hold_set],
        dtype=int
    )
    train_idx = np.array(
        [i for i in range(len(IDs_all)) if i not in hold_set],
        dtype=int
    )

    out_rows = []

    # Skip tiny states
    if len(hold_idx) < 5 or len(train_idx) < (K_ + 1 + 2):
        return out_rows

    # OLS baseline (record logL only)
    ols_tr = fit_ols(y_all[train_idx], X_all[train_idx, :])
    out_rows.append([st, 'OLS', '-', 'OLS',
                     ols_tr['logL'], np.nan, np.nan, np.nan])

    for name, Wset, combo in tasks:
        # Training
        y_tr = y_all[train_idx]
        X_tr = X_all[train_idx, :]
        W_tr = [submatrix(Wk, train_idx) for Wk in Wset]
        res_tr = fit_sem_multi(y_tr, X_tr, W_tr, combo=combo)

        # holdout
        y_te = y_all[hold_idx]
        X_te = X_all[hold_idx, :]
        W_te = [submatrix(Wk, hold_idx) for Wk in Wset]
        N_te, K_te = X_te.shape
        m = len(W_te)

        A_te, logdetA_te = build_A_and_logdet(res_tr['lam'], W_te, combo=combo)
        if A_te is None:
            out_rows.append([
                st, name, ';'.join(name.split('_')[3:]), combo,
                -np.inf, np.inf, np.nan, np.nan
            ])
            continue

        e_te = y_te - X_te.dot(res_tr['beta'])
        v_te = A_te.dot(e_te)
        sse = float(v_te.T.dot(v_te))

        df_ = N_te - (K_te + m)
        sigma2 = sse/df_ if df_ > 0 else sse / max(N_te, 1)
        logL = (logdetA_te
                - (N_te/2)*np.log(2*np.pi*sigma2)
                - sse/(2*sigma2))
        AIC  = -2*logL + 2*(K_te + m)
        AICc = (AIC + (2*(K_te+m)*((K_te+m)+1)) /
                (N_te-(K_te+m)-1) if (N_te-(K_te+m)-1) > 0 else np.nan)

        # MoranI: fix connectivity on the holdout state's WQ before evaluation
        WQ_te = submatrix(WQ_array, hold_idx)
        XY_te = XY_all_[hold_idx, :]
        WQ_eval = ensure_connected_within_state(WQ_te, XY_te)
        mi = Moran(
            v_te,
            util.full2W(WQ_eval, ids=[IDs_all[i] for i in hold_idx]),
            permutations=0
        )

        out_rows.append([
            st, name, ';'.join(name.split('_')[3:]), combo,
            logL, AICc, mi.I, mi.p_norm
        ])

    return out_rows


states = sorted(df['STATE'].unique())
tasks_for_loso = [t for t in TASKS_MAIN_ADD if t[0] in stage2_models]

res_lists = Parallel(
    n_jobs=N_JOBS, backend='loky', verbose=0
)(
    delayed(_loso_state_worker)(
        st, IDs, df, y, X, tasks_for_loso, WQ, XY_all, K
    )
    for st in states
)

cv_loso = pd.DataFrame(
    [row for rows in res_lists for row in rows],
    columns=['STATE','Model','W_mix','Combo',
             'pred_logL','pred_AICc','pred_MoranI','pred_p']
)

print(f"\n=== Stage-2: LOSO (shortlist, US, outcome={outcome_tag}) — preview ===")
print(cv_loso.head(10))

# 3.5 Forward gate：strict / lenient / super_lenient -------------------------

def summarize_models(cv_df, prefix='SEM_add_'):
    sub = cv_df[cv_df['Model'].str.startswith(prefix)].copy()
    g = (sub.groupby('Model', as_index=False)
           .agg(pred_AICc_mean=('pred_AICc','mean'),
                pred_AICc_sd  =('pred_AICc','std'),
                absMI_mean    =('pred_MoranI',
                                 lambda s: s.abs().mean()),
                absMI_sd      =('pred_MoranI',
                                 lambda s: s.abs().std())))
    return g.dropna(subset=['pred_AICc_mean','absMI_mean'])


sum_tbl = summarize_models(cv_loso, prefix='SEM_add_')
sum_tbl['dAICc_vs_best'] = (
    sum_tbl['pred_AICc_mean'] - sum_tbl['pred_AICc_mean'].min()
)

best_by_AICc_row  = sum_tbl.loc[sum_tbl['pred_AICc_mean'].idxmin()]
best_by_AICc_name = best_by_AICc_row['Model']

# Gate profiles
GATE_PROFILE        = 'super_lenient'   # ★ Recommended: super_lenient
PREFER_PAIR_IF_TIED = True
TIE_DELTA_REL       = 0.01             # Treat relative AICc difference ≤1% as a tie

if GATE_PROFILE == 'strict':
    GATE_DAICC_REL_MAX       = -0.01
    GATE_MI_REL_IMPROVE_MIN  = 0.00
    GATE_WINRATE_MIN         = 0.70
elif GATE_PROFILE == 'lenient':
    GATE_DAICC_REL_MAX       = 0.00
    GATE_MI_REL_IMPROVE_MIN  = 0.05
    GATE_WINRATE_MIN         = 0.50
elif GATE_PROFILE == 'super_lenient':
    GATE_DAICC_REL_MAX       = 0.02
    GATE_MI_REL_IMPROVE_MIN  = -0.10
    GATE_WINRATE_MIN         = 0.10
else:
    raise ValueError("Unknown GATE_PROFILE")

best_single_row = sum_tbl[sum_tbl['Model'] == best_single_name].iloc[0]
pair_tbl = sum_tbl[sum_tbl['Model'].str.match(r'^SEM_add_2_')]

if len(pair_tbl) > 0:
    best_pair_row = pair_tbl.nsmallest(1, 'pred_AICc_mean').iloc[0]

    # ΔAICc (absolute & relative)
    dAICc = (best_pair_row['pred_AICc_mean'] -
             best_single_row['pred_AICc_mean'])
    AICc_single = best_single_row['pred_AICc_mean']
    dAICc_rel = dAICc / max(abs(AICc_single), 1e-9)

    # Relative improvement in Moran's I
    absMI_single = best_single_row['absMI_mean']
    absMI_pair   = best_pair_row['absMI_mean']
    rel_mi_gain  = (
        (absMI_single - absMI_pair) /
        max(absMI_single, 1e-9)
    )

    # State-level win_rate: proportion of states where the pair has smaller AICc
    A_ = cv_loso[cv_loso['Model'] == best_pair_row['Model']][
        ['STATE','pred_AICc']
    ].rename(columns={'pred_AICc':'pair'})
    B_ = cv_loso[cv_loso['Model'] == best_single_name][
        ['STATE','pred_AICc']
    ].rename(columns={'pred_AICc':'single'})
    J = A_.merge(B_, on='STATE', how='inner')
    win_rate = float((J['pair'] < J['single']).mean()) if len(J) > 0 else 0.0

    # Gate decision
    pass_gate = (
        (dAICc_rel <= GATE_DAICC_REL_MAX) and
        (rel_mi_gain >= GATE_MI_REL_IMPROVE_MIN) and
        (win_rate >= GATE_WINRATE_MIN)
    )

    if pass_gate:
        final_model_main = best_pair_row['Model']
        gate_note = (
            f"[{GATE_PROFILE}] Accepted 2-channel by gates: "
            f"ΔAICc={dAICc:.2f}, ΔAICc_rel={dAICc_rel:.3f}<= {GATE_DAICC_REL_MAX}, "
            f"rel|I|↓={rel_mi_gain:.3f}>= {GATE_MI_REL_IMPROVE_MIN}, "
            f"win_rate={win_rate:.2f}>= {GATE_WINRATE_MIN}."
        )
    else:
        # If the pair fails the gate but ΔAICc_rel is very small, trigger tie-preference
        if (GATE_PROFILE == 'super_lenient' and
            PREFER_PAIR_IF_TIED and
            (dAICc_rel <= TIE_DELTA_REL) and
            (rel_mi_gain >= GATE_MI_REL_IMPROVE_MIN) and
            (win_rate >= GATE_WINRATE_MIN/2)):
            final_model_main = best_pair_row['Model']
            gate_note = (
                f"[{GATE_PROFILE}] Tie-preference to 2-channel "
                f"(ΔAICc_rel={dAICc_rel:.3f}<= {TIE_DELTA_REL}); "
                f"rel|I|↓={rel_mi_gain:.3f}, win_rate={win_rate:.2f}."
            )
        else:
            final_model_main = best_single_name
            gate_note = (
                f"[{GATE_PROFILE}] Kept best single: pair failed gates "
                f"and tie-preference not triggered. "
                f"ΔAICc={dAICc:.2f}, ΔAICc_rel={dAICc_rel:.3f}, "
                f"rel|I|↓={rel_mi_gain:.3f}, win_rate={win_rate:.2f}."
            )
else:
    final_model_main = best_single_name
    dAICc = np.nan
    dAICc_rel = np.nan
    rel_mi_gain = np.nan
    win_rate = np.nan
    gate_note = f"[{GATE_PROFILE}] No eligible two-channel; kept best single."

print(f"\n[Final model for {outcome_tag} 2019, US] {final_model_main}")
print("[Gate note] ", gate_note)
print("[Best-by-AICc on shortlist] ", best_by_AICc_name)

# ============================================
# 9. Add vs Prod sensitivity (US, current STD outcome)
#    — On the same W set as the final add model, compare additive vs multiplicative A(λ) structure
# ============================================

RUN_PROD_SENS = True  # Enable/disable add vs prod sensitivity analysis

if RUN_PROD_SENS:
    # 9.1 Parse channel keys from the model name (WP / WS / WQ ...)
    def parse_W_keys_from_model(model_name, W_dict_local):
        """
        For example:
          'SEM_add_1_WS'      -> ['WS']
          'SEM_add_2_WS_WQ'   -> ['WS','WQ']
        Keep only keys that exist in W_dict_local.
        """
        parts = model_name.split('_')
        chans = [p for p in parts[3:] if p in W_dict_local]
        if len(chans) == 0:
            raise ValueError(f"Cannot parse channels from model name: {model_name}")
        return chans

    W_keys_final = parse_W_keys_from_model(final_model_main, W_dict)
    Wset_final   = [W_dict[k] for k in W_keys_final]

    # 9.2 Fit add and prod structures using the same W set

    # add structure (main spec; refit full-sample for consistent output)
    res_add_full = fit_sem_multi(y, X, Wset_final, combo='add')
    I_add_full, p_add_full = moran_I_val_from_array(
        res_add_full['resid'], WQ, ids=IDs
    )

    # prod structure (sensitivity check)
    res_prod_full = fit_sem_multi(y, X, Wset_final, combo='prod')
    I_prod_full, p_prod_full = moran_I_val_from_array(
        res_prod_full['resid'], WQ, ids=IDs
    )

    # 9.3 Assemble results table & save
    sens_df = pd.DataFrame([
        {
            'Outcome_tag': outcome_tag,
            'Model_name': final_model_main,
            'Combo': 'add',
            'W_mix': ';'.join(W_keys_final),
            'logL': res_add_full['logL'],
            'AICc': res_add_full['AICc'],
            'MoranI': I_add_full,
            'absMoranI': abs(I_add_full),
            'p_norm': p_add_full
        },
        {
            'Outcome_tag': outcome_tag,
            'Model_name': final_model_main.replace('SEM_add', 'SEM_prod'),
            'Combo': 'prod',
            'W_mix': ';'.join(W_keys_final),
            'logL': res_prod_full['logL'],
            'AICc': res_prod_full['AICc'],
            'MoranI': I_prod_full,
            'absMoranI': abs(I_prod_full),
            'p_norm': p_prod_full
        }
    ])

    print("\n=== Add vs Prod (sensitivity on final W-combo, US) ===")
    print(sens_df)

    suffix_gate = f"_{GATE_PROFILE}" if 'GATE_PROFILE' in globals() else ""
    out_name = f"APPENDIX_add_vs_prod_sensitivity_US_{outcome_tag}{suffix_gate}.csv"
    sens_df.to_csv(out_name, index=False)

    print(f"\n[Add vs Prod sensitivity] Saved: {out_name}")

else:
    print("\n[Add vs Prod sensitivity] RUN_PROD_SENS = False, skipped.")


### Covariate sensitivity

In [ ]:
# ============================================================
#  SEM (multi-W, US) - Fast Covariate Robustness Check
#
#  Goal:
#    Covariate sensitivity (US-wide), mirroring the Deep South logic:
#      - 9 baseline covariates
#      - 4 "socio-sensitive" vars: Poverty, Noinsuranc, crime16to1, Dissimilar
#      - 11 covariate profiles:
#          * full (all 9)
#          * drop each of the 4 sensitive vars (x4)
#          * drop pairs of the 4 sensitive vars (C(4,2) = 6)
#
#  Data & W-matrices:
#    - US_HIV_Merged_total_final.shp (same as main US analysis)
#    - US_County_PCI_2019.csv      -> WP (Twitter PCI)
#    - processed_sci_summary.csv   -> WS (Facebook SCI)
#    - Queen contiguity            -> WQ (adjacency)
#
#  Channel search space (consistent with US main SEM analysis):
#    - Singles: WP, WS, WQ
#    - Pairs:   WP+WQ, WS+WQ    (NO WP+WS)
#
#  Output (CSV):
#    - COV_US_in_sample_add_<cov_profile>_<GATE_PROFILE>.csv
#    - COV_US_final_decision_allProfiles_<outcome>_<GATE_PROFILE>.csv
#    - COV_US_model_frequency_<outcome>_<GATE_PROFILE>.csv
#    - COV_US_Wmix_frequency_<outcome>_<GATE_PROFILE>.csv
#    - COV_US_lambda_table_<outcome>_<GATE_PROFILE>.csv
#
#  Note:
#    - No LOSO / cross-validation here (too expensive for 11 profiles).
#    - Only full-sample fits per profile for speed.
# ============================================================

import numpy as np
import pandas as pd
import geopandas as gpd
from itertools import combinations
from numpy.linalg import slogdet
from scipy.optimize import minimize
from esda.moran import Moran
from libpysal.weights import util, W, Queen

np.seterr(all="ignore")
RANDOM_SEED = 7
rng = np.random.default_rng(RANDOM_SEED)

# ----------------------------
# Mode / gate configs
# ----------------------------
RUN_PROD = False  # keep for compatibility; not really used here

# Gate profile for pair vs single (in-sample only)
GATE_PROFILE = "lenient"          # options: "strict" or "lenient"
PREFER_PAIR_IF_TIED = True
TIE_DELTA = 2.0                   # AICc "tie" width (pair - single <= TIE_DELTA)

if GATE_PROFILE == "strict":
    # Pair must clearly outperform single (in-sample)
    GATE_DAICC_THRESH = -4.0      # ΔAICc = pair - single (negative = better)
    GATE_MI_REL_IMPROVE_MIN = 0.00  # |I| at least not worse
else:
    # Lenient: allow small AICc disadvantage if Moran's I improves
    GATE_DAICC_THRESH = +1.0      # allow pair up to +1 AICc worse
    GATE_MI_REL_IMPROVE_MIN = 0.02  # ≥ 2% reduction in |I|


# ============================================================
# A. Read US shapefile & basic preprocessing
# ============================================================

shp_path = "US_HIV_Merged_total_final.shp"
gdf = gpd.read_file(shp_path)

if "FIPS" not in gdf.columns:
    raise KeyError("Shapefile needs a 'FIPS' field named 'FIPS'.")

# Clean FIPS: strip leading zeros, drop duplicates
gdf["FIPS"] = gdf["FIPS"].astype(str).str.lstrip("0")
gdf = gdf.drop_duplicates(subset=["FIPS"])

# Fix possibly truncated 2018 variable names (for consistency; not directly used here)
rename_map = {
    "RateChlamy": "RateChlamydia2018",
    "RateGonorr": "RateGonorrhea2018",
    "RateHIV201": "RateHIV2018"
}
for short, long_name in rename_map.items():
    if (short in gdf.columns) and (long_name not in gdf.columns):
        gdf = gdf.rename(columns={short: long_name})

# Baseline covariates (same 9 as in main US analysis)
X_cols = [
    "Population", "UrbanRural", "Female", "Old", "Black",
    "Noinsuranc", "Poverty", "crime16to1", "Dissimilar"
]

cols_keep = [
    "FIPS",
    "GonorrheaR", "ChlamydiaR", "PercenHIVp",
    *X_cols,
    "RateChlamydia2018", "RateGonorrhea2018", "RateHIV2018",
    "geometry"
]

df_geo = gdf[cols_keep].copy()
df_geo = gpd.GeoDataFrame(df_geo, geometry="geometry", crs=gdf.crs)

# STATE code from FIPS
def fips2state(x):
    return str(x).zfill(5)[:2]

df_geo["STATE"] = df_geo["FIPS"].apply(fips2state)


# ============================================================
# B. Build spatial weights (WP / WS / WQ) — US-wide
# ============================================================

def subset_W(w: W, keep_ids):
    """
    Subset a libpysal W to a given id set, preserving the given ID order.
    """
    keep_ids = list(dict.fromkeys([str(x) for x in keep_ids]))
    keep = set(keep_ids)
    new_neighbors, new_weights = {}, {}
    for i in w.id_order:
        if i not in keep:
            continue
        nbs = w.neighbors.get(i, [])
        wts = w.weights.get(i, [])
        kn, kw = [], []
        for nb, wt in zip(nbs, wts):
            if nb in keep:
                kn.append(nb)
                kw.append(wt)
        new_neighbors[i] = kn
        new_weights[i] = kw
    return W(new_neighbors, new_weights, ids=keep_ids)


# 1) WP: Twitter PCI mobility
pci = pd.read_csv("US_County_PCI_2019.csv")
pci["place_i"] = pci["place_i"].astype(str).str.lstrip("0")
pci["place_j"] = pci["place_j"].astype(str).str.lstrip("0")

pci_f = pci[
    pci["place_i"].isin(df_geo["FIPS"])
    & pci["place_j"].isin(df_geo["FIPS"])
].copy()

A_pci = pci_f.pivot_table(
    index="place_i", columns="place_j",
    values="pci", fill_value=0
)
np.fill_diagonal(A_pci.values, 0.0)
A = A_pci.values.astype(float)
A = A + A.T - np.diag(np.diag(A))  # symmetrize
row_sum = A.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WP_full = A / row_sum
wP = util.full2W(WP_full, ids=list(A_pci.index))

# 2) WS: Facebook SCI ties
sci = pd.read_csv("processed_sci_summary.csv")
sci["user_loc"] = sci["user_loc"].astype(str).str.lstrip("0")
sci["tfr_loc"]  = sci["tfr_loc"].astype(str).str.lstrip("0")

sci_f = sci[
    sci["user_loc"].isin(df_geo["FIPS"])
    & sci["tfr_loc"].isin(df_geo["FIPS"])
].copy()

A_sci = sci_f.pivot_table(
    index="user_loc", columns="tfr_loc",
    values="tscaled_sci", fill_value=0
)
np.fill_diagonal(A_sci.values, 0.0)
B = A_sci.values.astype(float)
B = B + B.T - np.diag(np.diag(B))
row_sum = B.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WS_full = B / row_sum
wS = util.full2W(WS_full, ids=list(A_sci.index))

# 3) WQ: queen contiguity (row-standardized)
g3857 = df_geo.to_crs(epsg=3857)
wQ = Queen.from_dataframe(g3857, ids=list(df_geo["FIPS"]))
wQ.transform = "R"  # row-standardize

# Harmonize ID set & ordering across df_geo, WP, WS, WQ
common = (
    set(df_geo["FIPS"])
    & set(wP.id_order)
    & set(wS.id_order)
    & set(wQ.id_order)
)

df_geo = df_geo[df_geo["FIPS"].isin(common)].copy()
wP = subset_W(wP, common)
wS = subset_W(wS, common)
wQ = subset_W(wQ, common)

df_geo = df_geo.set_index("FIPS").loc[wQ.id_order].reset_index()
IDs = list(wQ.id_order)
STATE = df_geo["STATE"].values

WP, _ = wP.full()
WS, _ = wS.full()
WQ, _ = wQ.full()


# ============================================================
# C. Outcome & covariate profiles (US-wide)
# ============================================================

# Outcome: HIV prevalence (%)
y_col = "PercenHIVp"   # can be switched to 'GonorrheaR' / 'ChlamydiaR' if needed
outcome_tag = {
    "PercenHIVp": "HIV",
    "GonorrheaR": "Gonorrhea",
    "ChlamydiaR": "Chlamydia"
}.get(y_col, y_col)

# Full 9 covariates (same as X_cols above)
X_cols_full = X_cols.copy()

# Socioeconomic vars that reviewers are most concerned about
SOCIO_SENSITIVE = ["Poverty", "Noinsuranc", "crime16to1", "Dissimilar"]


def build_profile(base_vars, drop_vars):
    drop_set = set(drop_vars)
    return [v for v in base_vars if v not in drop_set]


# Generate 11 covariate profiles: full + drop 1 + drop 2 (Deep South logic)
COV_PROFILES = {}

# 1) full model (9 covariates)
COV_PROFILES["full"] = X_cols_full

# 2) drop single sensitive variable
for v in SOCIO_SENSITIVE:
    name = f"drop_{v}"
    COV_PROFILES[name] = build_profile(X_cols_full, [v])

# 3) drop pairs of sensitive variables
for v1, v2 in combinations(SOCIO_SENSITIVE, 2):
    name = f"drop_{v1}_{v2}"
    COV_PROFILES[name] = build_profile(X_cols_full, [v1, v2])


# ============================================================
# D. OLS / SEM / Moran helpers
# ============================================================

def fit_ols(y, X):
    """
    Standard OLS fit.
    Returns beta, residuals, log-likelihood, and AICc.
    """
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X.dot(beta)
    sse = float(e.T.dot(e))
    N, K = X.shape
    df_ = N - K
    sigma2 = sse / df_
    logL = (
        -(N / 2) * np.log(2 * np.pi * sigma2)
        - sse / (2 * sigma2)
    )
    AIC = -2 * logL + 2 * K
    AICc = (
        AIC + (2 * K * (K + 1)) / (N - K - 1)
        if (N - K - 1) > 0 else np.nan
    )
    return {
        "model": "OLS",
        "beta": beta,
        "logL": logL,
        "AICc": AICc,
        "resid": e
    }


def build_A_and_logdet(lam, W_list, combo="add"):
    """
    Build A and log|A| for SEM:
      y = Xβ + u, (I - Σ λ_k W_k) u = ε.
    """
    N = W_list[0].shape[0]
    I = np.eye(N)

    if combo == "add":
        A = I.copy()
        for i, Wk in enumerate(W_list):
            A -= lam[i] * Wk
        sign, logdetA = slogdet(A)
        if sign <= 0:
            return None, None
        return A, logdetA

    elif combo == "prod":
        # Not used here, kept for compatibility
        A = I.copy()
        logdet_sum = 0.0
        for i, Wk in enumerate(W_list):
            Mi = I - lam[i] * Wk
            sign_i, logdet_i = slogdet(Mi)
            if sign_i <= 0:
                return None, None
            logdet_sum += logdet_i
            A = Mi @ A
        return A, logdet_sum

    else:
        raise ValueError("combo must be 'add' or 'prod'")


def neglog_sem(params, y, X, W_list, combo="add"):
    """
    Negative log-likelihood of SEM for optimization.
    params = [λ_1,...,λ_m, β_0,...,β_{K-1}]
    """
    N, K = X.shape
    m = len(W_list)
    lam = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(lam, W_list, combo=combo)
    if A is None:
        return 1e12

    e = y - X.dot(beta)
    v = A.dot(e)
    sse = float(v.T.dot(v))

    df_ = N - (K + m)
    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_
    if sigma2 <= 0:
        return 1e12

    logL = (
        logdetA
        - (N / 2) * np.log(2 * np.pi * sigma2)
        - sse / (2 * sigma2)
    )
    return -logL


def fit_sem(y, X, W_list, method="L-BFGS-B", combo="add"):
    """
    Single-start ML SEM fit (sufficient for covariate robustness).
    """
    N, K = X.shape
    m = len(W_list)

    # Starting point: OLS beta, λ = 0
    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
    x0 = np.r_[np.zeros(m), beta0]

    bounds = [(-0.99, 0.99)] * m + [(-np.inf, np.inf)] * K

    res = minimize(
        neglog_sem, x0,
        args=(y, X, W_list, combo),
        method=method,
        bounds=bounds,
        options={"maxiter": 400, "ftol": 1e-5, "gtol": 1e-5},
    )

    params = res.x
    lam = params[:m]
    beta = params[m:]
    logL = -res.fun
    p = m + K

    AIC = -2 * logL + 2 * p
    AICc = (
        AIC + (2 * p * (p + 1)) / (N - p - 1)
        if (N - p - 1) > 0 else np.nan
    )

    A, _ = build_A_and_logdet(lam, W_list, combo=combo)
    v = A.dot(y - X.dot(beta))

    return {
        "model": f"SEM_{combo}_{m}W",
        "lam": lam,
        "beta": beta,
        "logL": logL,
        "AICc": AICc,
        "resid": v,
        "combo": combo,
        "success": res.success,
    }


def moran_I_val_from_array(resid, W_array, ids):
    """
    Compute Moran's I using a numpy W-array and ID order.
    """
    w_obj = util.full2W(W_array, ids=ids)
    mi = Moran(resid, w_obj, permutations=0)
    return mi.I, mi.p_norm


# ============================================================
# E. Candidate SEM tasks (single + pair, add) — US channel set
# ============================================================

CHANNEL_KEYS = ["WP", "WS", "WQ"]


def make_tasks_add_single_pair_US(W_dict_local):
    """
    Candidate SEM models (consistent with main US analysis):
      - Singles: WP, WS, WQ
      - Pairs:   WP+WQ, WS+WQ
      (No WP+WS pair)
    """
    tasks = []
    # singles
    for k in ["WP", "WS", "WQ"]:
        if k in W_dict_local:
            tasks.append((f"SEM_add_1_{k}", [W_dict_local[k]], "add"))

    # pairs: WP+WQ / WS+WQ if both exist
    if "WP" in W_dict_local and "WQ" in W_dict_local:
        tasks.append(("SEM_add_2_WP_WQ",
                      [W_dict_local["WP"], W_dict_local["WQ"]], "add"))
    if "WS" in W_dict_local and "WQ" in W_dict_local:
        tasks.append(("SEM_add_2_WS_WQ",
                      [W_dict_local["WS"], W_dict_local["WQ"]], "add"))

    return tasks


def extract_W_mix_from_model(name: str) -> str:
    """
    Extract W-mix part from a model name, e.g.
      'SEM_add_2_WP_WQ' -> 'WP_WQ'
    """
    parts = name.split("_")
    if len(parts) >= 4:
        return "_".join(parts[3:])
    return ""


# ============================================================
# F. Main loop over covariate profiles (US-wide)
# ============================================================

all_decisions = []
lambda_rows = []

for cov_name, X_cols_profile in COV_PROFILES.items():
    print("\n" + "=" * 80)
    print(f"Running US covariate profile: {cov_name}  "
          f"(#{len(X_cols_profile)} covariates)")
    print("X_cols =", X_cols_profile)
    print("=" * 80)

    # 1) Subset data for this profile: require outcome + covariates not NA
    df_sub = df_geo[["FIPS", y_col] + X_cols_profile + ["STATE", "geometry"]].dropna().copy()
    fips_sub = df_sub["FIPS"].tolist()
    keep_set = set(fips_sub)

    # indices in original ID order
    idx_sub = [i for i, idd in enumerate(IDs) if idd in keep_set]
    if len(idx_sub) == 0:
        print(f"[WARN] cov_profile={cov_name} -> empty sample; skip.")
        continue

    IDs_sub = [IDs[i] for i in idx_sub]

    def submatrix(M, idx):
        return M[np.ix_(idx, idx)]

    WP_sub = submatrix(WP, idx_sub)
    WS_sub = submatrix(WS, idx_sub)
    WQ_sub = submatrix(WQ, idx_sub)

    # Reorder df_sub to match IDs_sub
    df_sub = df_sub.set_index("FIPS").loc[IDs_sub].reset_index()

    y = df_sub[y_col].values
    X_raw = df_sub[X_cols_profile].values
    N = X_raw.shape[0]
    X = np.hstack([np.ones((N, 1)), X_raw])
    K = X.shape[1]

    W_dict_local = {
        "WP": WP_sub,
        "WS": WS_sub,
        "WQ": WQ_sub,
    }
    TASKS_MAIN_ADD = make_tasks_add_single_pair_US(W_dict_local)

    # ----------------------------
    # Stage-1: in-sample OLS + SEM
    # ----------------------------
    base_rows = []

    # OLS baseline
    ols = fit_ols(y, X)
    I0, p0 = moran_I_val_from_array(ols["resid"], WQ_sub, ids=IDs_sub)
    base_rows.append(
        ["OLS", "-", "OLS", ols["logL"], ols["AICc"], I0, p0]
    )

    # SEM: all single + pair tasks
    for name, Wset, combo in TASKS_MAIN_ADD:
        res = fit_sem(y, X, Wset, combo=combo)
        I, pI = moran_I_val_from_array(res["resid"], WQ_sub, ids=IDs_sub)
        wmix = ";".join(name.split("_")[3:])
        base_rows.append(
            [name, wmix, combo, res["logL"], res["AICc"], I, pI]
        )

    base_df = pd.DataFrame(
        base_rows,
        columns=[
            "Model", "W_mix", "Combo",
            "logL", "AICc",
            "MoranI", "p_norm"
        ],
    )
    base_df = base_df.sort_values(["Combo", "AICc"])

    print(f"\n=== In-sample ranking (AICc) [US, cov={cov_name}, outcome={outcome_tag}] ===")
    print(base_df.head(10))

    # Save per-profile in-sample table
    base_path = f"COV_US_in_sample_add_{cov_name}_{outcome_tag}_{GATE_PROFILE}.csv"
    base_df.to_csv(base_path, index=False)
    print(f"Saved in-sample table for cov_profile={cov_name} -> {base_path}")

    # ----------------------------
    # Simple gate: best single vs best pair (in-sample)
    # ----------------------------
    stage1_sem = base_df[(base_df["Model"] != "OLS")].copy()
    single_pool = stage1_sem[stage1_sem["Model"].str.match(r"^SEM_add_1_")]
    pair_pool   = stage1_sem[stage1_sem["Model"].str.match(r"^SEM_add_2_")]

    if len(single_pool) == 0:
        print(f"[WARN] No single-channel SEM for cov_profile={cov_name}.")
        continue

    best_single_row = single_pool.nsmallest(1, "AICc").iloc[0]
    best_single_name = best_single_row["Model"]

    if len(pair_pool) > 0:
        best_pair_row = pair_pool.nsmallest(1, "AICc").iloc[0]
        best_pair_name = best_pair_row["Model"]

        dAICc = best_pair_row["AICc"] - best_single_row["AICc"]  # pair - single

        absMI_single = abs(best_single_row["MoranI"])
        absMI_pair   = abs(best_pair_row["MoranI"])
        rel_mi_gain  = (
            (absMI_single - absMI_pair) / max(absMI_single, 1e-9)
        )

        # In cov-robustness, "win_rate" is degenerate: 1 if pair has smaller AICc, else 0.
        win_rate = 1.0 if best_pair_row["AICc"] < best_single_row["AICc"] else 0.0

        pass_gate = (
            (dAICc <= GATE_DAICC_THRESH)
            and (rel_mi_gain >= GATE_MI_REL_IMPROVE_MIN)
        )

        if pass_gate:
            final_model = best_pair_name
            gate_note = (
                f"[{GATE_PROFILE}] Accepted 2-channel by in-sample gate "
                f"(cov={cov_name}, outcome={outcome_tag}): "
                f"ΔAICc={dAICc:.2f} <= {GATE_DAICC_THRESH}, "
                f"rel|I|↓={rel_mi_gain:.3f} >= {GATE_MI_REL_IMPROVE_MIN}."
            )
        else:
            if PREFER_PAIR_IF_TIED and (dAICc <= TIE_DELTA):
                final_model = best_pair_name
                gate_note = (
                    f"[{GATE_PROFILE}] Tie-preference to 2-channel "
                    f"(cov={cov_name}, outcome={outcome_tag}; "
                    f"ΔAICc={dAICc:.2f} <= {TIE_DELTA}); "
                    f"rel|I|↓={rel_mi_gain:.3f}."
                )
            else:
                final_model = best_single_name
                gate_note = (
                    f"[{GATE_PROFILE}] Kept best single (cov={cov_name}, outcome={outcome_tag}): "
                    f"pair failed gate / tie condition. "
                    f"ΔAICc={dAICc:.2f}, rel|I|↓={rel_mi_gain:.3f}."
                )
    else:
        final_model = best_single_name
        dAICc = np.nan
        rel_mi_gain = np.nan
        win_rate = np.nan
        gate_note = (
            f"[{GATE_PROFILE}] No 2-channel candidate (cov={cov_name}, outcome={outcome_tag}); "
            f"kept best single."
        )

    print(f"\n[Final (US)] cov_profile={cov_name}, outcome={outcome_tag} -> {final_model}")
    print("[Gate note] ", gate_note)

    # Record decision line for this profile
    decision_df = pd.DataFrame({
        "Outcome_tag": [outcome_tag],
        "Outcome_col": [y_col],
        "Cov_profile": [cov_name],
        "n_covariates": [len(X_cols_profile)],
        "Profile": [GATE_PROFILE],
        "Final_model": [final_model],
        "Final_W_mix": [extract_W_mix_from_model(final_model)],
        "Best_single": [best_single_name],
        "Decision": [gate_note],
        "AICc_final": [
            float(
                base_df.loc[base_df["Model"] == final_model, "AICc"].iloc[0]
            )
        ],
        "MoranI_final": [
            float(
                base_df.loc[base_df["Model"] == final_model, "MoranI"].iloc[0]
            )
        ],
        "AICc_best_single": [float(best_single_row["AICc"])],
        "MoranI_best_single": [float(best_single_row["MoranI"])],
        "dAICc_pair_vs_single": [dAICc],
        "Rel_MI_gain(pair_vs_single)": [rel_mi_gain],
        "Win_rate_pair_vs_single": [win_rate],
        "GATE_DAICC_THRESH": [GATE_DAICC_THRESH],
        "GATE_MI_REL_IMPROVE_MIN": [GATE_MI_REL_IMPROVE_MIN],
        "PREFER_PAIR_IF_TIED": [PREFER_PAIR_IF_TIED],
        "TIE_DELTA": [TIE_DELTA],
    })
    all_decisions.append(decision_df)

    # Record lambda for the final model (for robustness narrative)
    if final_model.startswith("SEM_add_1_"):
        _, Wkey = final_model.split("SEM_add_1_")
        Wset_final = [W_dict_local[Wkey]]
    elif final_model.startswith("SEM_add_2_"):
        parts = final_model.split("_")
        Wkey1, Wkey2 = parts[3], parts[4]
        Wset_final = [W_dict_local[Wkey1], W_dict_local[Wkey2]]
    else:
        Wset_final = []

    if len(Wset_final) > 0:
        res_final = fit_sem(y, X, Wset_final, combo="add")
        lam = res_final["lam"]
        lam_row = {
            "Outcome_tag": outcome_tag,
            "Outcome_col": y_col,
            "Cov_profile": cov_name,
            "Final_model": final_model,
            "Final_W_mix": extract_W_mix_from_model(final_model),
        }
        for i, val in enumerate(lam):
            lam_row[f"lambda_{i+1}"] = float(val)
        lambda_rows.append(lam_row)


# ============================================================
# G. Cross-profile summaries (US)
# ============================================================

if all_decisions:
    meta_df = pd.concat(all_decisions, ignore_index=True)
    meta_path = (
        f"COV_US_final_decision_allProfiles_{outcome_tag}_{GATE_PROFILE}.csv"
    )
    meta_df.to_csv(meta_path, index=False)

    # Frequency of each specific final model
    model_freq = (
        meta_df.groupby("Final_model", as_index=False)
        .agg(
            n_profiles=("Cov_profile", "nunique"),
            mean_n_covariates=("n_covariates", "mean"),
            mean_AICc_final=("AICc_final", "mean"),
        )
        .sort_values("n_profiles", ascending=False)
    )
    model_freq_path = (
        f"COV_US_model_frequency_{outcome_tag}_{GATE_PROFILE}.csv"
    )
    model_freq.to_csv(model_freq_path, index=False)

    # Frequency of each W-mix (aggregated across profiles)
    mix_freq = (
        meta_df.groupby("Final_W_mix", as_index=False)
        .agg(n_profiles=("Cov_profile", "nunique"))
        .sort_values("n_profiles", ascending=False)
    )
    mix_freq_path = (
        f"COV_US_Wmix_frequency_{outcome_tag}_{GATE_PROFILE}.csv"
    )
    mix_freq.to_csv(mix_freq_path, index=False)

    # Lambda table (for robustness narrative)
    if lambda_rows:
        lam_df = pd.DataFrame(lambda_rows)
        lam_path = f"COV_US_lambda_table_{outcome_tag}_{GATE_PROFILE}.csv"
        lam_df.to_csv(lam_path, index=False)
    else:
        lam_path = "(none)"

    print("\n[Summary across covariate profiles, US]")
    print(f"  - Decisions per profile: {meta_path}")
    print(f"  - Model frequency:       {model_freq_path}")
    print(f"  - W-mix frequency:       {mix_freq_path}")
    print(f"  - Lambda table:          {lam_path}")
else:
    print("\n[INFO] No valid covariate profiles were estimated (US).")

###SLM vs SEM hero

In [ ]:
# ============================================
# OLS vs hero SEM vs hero-structure SLM (US)
# ============================================
import numpy as np
import pandas as pd
import geopandas as gpd

from numpy.linalg import slogdet
from scipy.optimize import minimize
from esda.moran import Moran
from libpysal.weights import util, W, Queen

np.seterr(all='ignore')

# --------------------------------------------
# 0. Global config
# --------------------------------------------

# Outcome column: HIV / Gonorrhea / Chlamydia
#   HIV:         y_col = 'PercenHIVp'
#   Gonorrhea:   y_col = 'GonorrheaR'
#   Chlamydia:   y_col = 'ChlamydiaR'
y_col = 'PercenHIVp'

# Corresponding hero SEM structure (use your finalized choice)
# Example:
#   HIV        : 'SEM_add_2_WP_WQ'
#   Gonorrhea  : 'SEM_add_1_WQ'
#   Chlamydia  : 'SEM_add_2_WS_WQ'
hero_model = 'SEM_add_2_WP_WQ'

OUTCOME_TAG = {
    'PercenHIVp': 'HIV',
    'GonorrheaR': 'Gonorrhea',
    'ChlamydiaR': 'Chlamydia'
}
outcome_tag = OUTCOME_TAG.get(y_col, y_col)

# File paths (change to match your project paths)
shp_path = "US_HIV_Merged_total_final.shp"
pci_path = "US_County_PCI_2019.csv"
sci_path = "processed_sci_summary.csv"

RANDOM_SEED = 7

# Covariates (keep consistent with your main analysis)
X_cols = [
    'Population', 'UrbanRural', 'Female', 'Old', 'Black',
    'Noinsuranc', 'Poverty', 'crime16to1', 'Dissimilar'
]

# --------------------------------------------
# 1. Read shapefile & basic cleaning
# --------------------------------------------
gdf = gpd.read_file(shp_path)

if 'FIPS' not in gdf.columns:
    raise KeyError("Shapefile must contain a 'FIPS' field.")

# Strip leading zeros from FIPS and drop duplicates
gdf['FIPS'] = gdf['FIPS'].astype(str).str.lstrip('0')
gdf = gdf.drop_duplicates(subset=['FIPS'])

# Check that outcome and covariate columns exist
cols_need = ['FIPS', y_col] + X_cols + ['geometry']
missing_cols = set(cols_need) - set(gdf.columns)
if missing_cols:
    raise KeyError(f"Shapefile is missing required fields: {missing_cols}")

df_geo = gdf[cols_need].copy()

# Drop missing values for the current outcome + covariates
df_geo = df_geo.dropna(subset=[y_col] + X_cols)

# State code (first two digits of FIPS)
def fips2state(x):
    return str(x).zfill(5)[:2]

df_geo['STATE'] = df_geo['FIPS'].apply(fips2state)
df_geo = gpd.GeoDataFrame(df_geo, geometry='geometry', crs=gdf.crs)

# --------------------------------------------
# 2. Construct three spatial weight matrices: WP / WS / WQ (US)
# --------------------------------------------

# 2.1 WP: Twitter PCI mobility
pci = pd.read_csv(pci_path)
pci['place_i'] = pci['place_i'].astype(str).str.lstrip('0')
pci['place_j'] = pci['place_j'].astype(str).str.lstrip('0')

pci_f = pci[
    pci['place_i'].isin(df_geo['FIPS']) &
    pci['place_j'].isin(df_geo['FIPS'])
].copy()

A_pci = pci_f.pivot_table(
    index='place_i', columns='place_j',
    values='pci', fill_value=0.0
)

# Zero diagonal, row-standardize
np.fill_diagonal(A_pci.values, 0.0)
A = A_pci.values.astype(float)
A = A + A.T - np.diag(np.diag(A))     # Symmetrize
row_sum = A.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WP_full = A / row_sum                 # Row-standardize
wP = util.full2W(WP_full, ids=list(A_pci.index))

# 2.2 WS: Facebook SCI social ties
sci = pd.read_csv(sci_path)
sci['user_loc'] = sci['user_loc'].astype(str).str.lstrip('0')
sci['tfr_loc']  = sci['tfr_loc'].astype(str).str.lstrip('0')

sci_f = sci[
    sci['user_loc'].isin(df_geo['FIPS']) &
    sci['tfr_loc'].isin(df_geo['FIPS'])
].copy()

A_sci = sci_f.pivot_table(
    index='user_loc', columns='tfr_loc',
    values='tscaled_sci', fill_value=0.0
)

np.fill_diagonal(A_sci.values, 0.0)
B = A_sci.values.astype(float)
B = B + B.T - np.diag(np.diag(B))
row_sum = B.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WS_full = B / row_sum
wS = util.full2W(WS_full, ids=list(A_sci.index))

# 2.3 WQ: queen contiguity (US counties with data)
g3857 = df_geo.to_crs(epsg=3857)
wQ = Queen.from_dataframe(g3857, ids=list(df_geo['FIPS']))
wQ.transform = 'R'   # Row-standardized
WQ_full, _ = wQ.full()

# --------------------------------------------
# 3. Align ID set & ordering
#    —— ensure df_geo / WP / WS / WQ use the same FIPS ordering
# --------------------------------------------

common = (
    set(df_geo['FIPS']) &
    set(wP.id_order) &
    set(wS.id_order) &
    set(wQ.id_order)
)

def subset_W(w: W, keep_ids):
    """
    Trim W to contain only keep_ids, and preserve row-standardization.
    All W matrices share the same keep_ids order to guarantee consistency.
    """
    keep_ids = list(dict.fromkeys([str(x) for x in keep_ids]))  # fixed order
    keep = set(keep_ids)
    new_neighbors, new_weights = {}, {}

    for i in w.id_order:
        if i not in keep:
            continue
        nbs = w.neighbors.get(i, [])
        wts = w.weights.get(i, [])
        kn, kw = [], []
        for nb, wt in zip(nbs, wts):
            if nb in keep:
                kn.append(nb)
                kw.append(wt)
        new_neighbors[i] = kn
        new_weights[i] = kw

    return W(new_neighbors, new_weights, ids=keep_ids)

# Trim to common IDs
df_geo = df_geo[df_geo['FIPS'].isin(common)].copy()
wP = subset_W(wP, common)
wS = subset_W(wS, common)
wQ = subset_W(wQ, common)

# Align df_geo ordering to wQ.id_order
df_geo = df_geo.set_index('FIPS').loc[wQ.id_order].reset_index()
IDs = list(wQ.id_order)

# Convert back to numpy matrices
WP, _       = wP.full()
WS, _       = wS.full()
WQ_array, _ = wQ.full()

# --------------------------------------------
# 4. Build y & X
# --------------------------------------------
y = df_geo[y_col].values
X_raw = df_geo[X_cols].values
N = X_raw.shape[0]

# Add intercept
X = np.hstack([np.ones((N, 1)), X_raw])
K = X.shape[1]

# --------------------------------------------
# 5. Core helper functions: OLS / SEM / SLM
# --------------------------------------------

def moran_I_val_from_array(resid, W_array, ids):
    """
    Given residuals and full W (numpy array), compute Moran's I.
    Use WQ as the unified evaluation weight matrix here.
    """
    w_obj = util.full2W(W_array, ids=ids)
    mi = Moran(resid, w_obj, permutations=0)
    return mi.I, mi.p_norm


def fit_ols(y, X, W_eval, ids):
    """
    Standard OLS fit + residual Moran's I
    """
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X.dot(beta)
    sse = float(e.T.dot(e))
    N_, K_ = X.shape
    df_ = N_ - K_
    sigma2 = sse / df_
    logL = -(N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    AIC = -2*logL + 2*K_
    AICc = (AIC + (2*K_*(K_+1))/(N_-K_-1)
            if (N_-K_-1) > 0 else np.nan)

    I, pI = moran_I_val_from_array(e, W_eval, ids)
    return {
        'model_type': 'OLS',
        'W_mix': '-',
        'logL': logL,
        'AICc': AICc,
        'MoranI': I,
        'p_norm': pI
    }


def build_A_and_logdet(lam, W_list):
    """
    A = I - Σ λ_k W_k, return A and log|A|
    """
    N_ = W_list[0].shape[0]
    I = np.eye(N_)

    A = I.copy()
    for i, Wk in enumerate(W_list):
        A -= lam[i] * Wk

    sign, logdetA = slogdet(A)
    if sign <= 0:
        return None, None
    return A, logdetA


def neglog_sem(params, y, X, W_list):
    """
    Negative log-likelihood for multi-channel SEM (error model):
      y = Xβ + μ,  μ = (I - Σ λ_k W_k)^(-1) ε
      ⇒ ε = A (y - Xβ), logL includes log|A|
    params = [λ_1,...,λ_m, β_0,...,β_{K-1}]
    """
    N_, K_ = X.shape
    m = len(W_list)
    lam = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(lam, W_list)
    if A is None:
        return 1e12

    e = y - X.dot(beta)
    v = A.dot(e)
    sse = float(v.T.dot(v))

    df_ = N_ - (K_ + m)
    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_
    if sigma2 <= 0:
        return 1e12

    logL = logdetA - (N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    return -logL


def fit_sem_multi(y, X, W_list):
    """
    Multi-network SEM fit (for the hero model).
    """
    N_, K_ = X.shape
    m = len(W_list)

    # Start: OLS beta + lambda=0
    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
    x0 = np.r_[np.zeros(m), beta0]
    bounds = [(-0.99, 0.99)]*m + [(-np.inf, np.inf)]*K_

    res = minimize(
        neglog_sem, x0,
        args=(y, X, W_list),
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': 200, 'ftol': 1e-4, 'gtol': 1e-4}
    )

    params = res.x
    lam = params[:m]
    beta = params[m:]
    logL = -res.fun
    p = m + K_
    AIC = -2*logL + 2*p
    AICc = AIC + (2*p*(p+1))/(N_-p-1) if (N_-p-1) > 0 else np.nan

    A, _ = build_A_and_logdet(lam, W_list)
    v = A.dot(y - X.dot(beta))  # whitened residual

    return lam, beta, logL, AICc, v


def neglog_slm(params, y, X, W_list):
    """
    Negative log-likelihood for multi-channel SLM (lag model):
      y = Σ ρ_k W_k y + Xβ + u, u ~ N(0, σ^2 I)
      Let A = I - Σ ρ_k W_k, then u = A y - Xβ
      logL = log|A| - (n/2)log(2πσ^2) - (u'u)/(2σ^2)
    params = [ρ_1,...,ρ_m, β_0,...,β_{K-1}]
    """
    N_, K_ = X.shape
    m = len(W_list)
    rho = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(rho, W_list)
    if A is None:
        return 1e12

    u = A.dot(y) - X.dot(beta)
    sse = float(u.T.dot(u))

    df_ = N_ - (K_ + m)
    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_
    if sigma2 <= 0:
        return 1e12

    logL = logdetA - (N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    return -logL


def fit_slm_multi(y, X, W_list):
    """
    Multi-network SLM fit (hero lag benchmark).
    """
    N_, K_ = X.shape
    m = len(W_list)

    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
    x0 = np.r_[np.zeros(m), beta0]
    bounds = [(-0.99, 0.99)]*m + [(-np.inf, np.inf)]*K_

    res = minimize(
        neglog_slm, x0,
        args=(y, X, W_list),
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': 200, 'ftol': 1e-4, 'gtol': 1e-4}
    )

    params = res.x
    rho = params[:m]
    beta = params[m:]
    logL = -res.fun
    p = m + K_
    AIC = -2*logL + 2*p
    AICc = AIC + (2*p*(p+1))/(N_-p-1) if (N_-p-1) > 0 else np.nan

    A, _ = build_A_and_logdet(rho, W_list)
    u = A.dot(y) - X.dot(beta)  # model residual u

    return rho, beta, logL, AICc, u


# --------------------------------------------
# 6. Parse hero model name and extract W channels for SEM/SLM
# --------------------------------------------

W_dict = {
    'WP': WP,
    'WS': WS,
    'WQ': WQ_array
}

def get_W_list_from_model_name(model_name, W_dict_local):
    """
    Examples:
      'SEM_add_1_WS'    -> channels = ['WS']
      'SEM_add_2_WP_WQ' -> channels = ['WP','WQ']
    Extract 'WP'/'WS'/'WQ' in the order they appear in the name.
    """
    parts = model_name.split('_')
    chans = [p for p in parts if p in W_dict_local]
    if len(chans) == 0:
        raise ValueError(f"Cannot parse channels from model name: {model_name}")
    W_list = [W_dict_local[ch] for ch in chans]
    return W_list, chans

W_list_hero, hero_channels = get_W_list_from_model_name(hero_model, W_dict)
W_mix_str = '+'.join(hero_channels)

print(f"[Config] Outcome = {outcome_tag}, y_col = {y_col}")
print(f"[Config] Hero SEM structure = {hero_model} (channels: {W_mix_str})")
print(f"[Sample] N = {N} counties")

# --------------------------------------------
# 7. Fit three models: OLS / hero SEM / hero-structure SLM
# --------------------------------------------

# 7.1 OLS
ols_res = fit_ols(y, X, W_eval=WQ_array, ids=IDs)

# 7.2 hero SEM (error)
lam_sem, beta_sem, logL_sem, AICc_sem, resid_sem = fit_sem_multi(
    y, X, W_list_hero
)
I_sem, p_sem = moran_I_val_from_array(resid_sem, WQ_array, IDs)
sem_res = {
    'model_type': 'SEM_error',
    'W_mix': W_mix_str,
    'logL': logL_sem,
    'AICc': AICc_sem,
    'MoranI': I_sem,
    'p_norm': p_sem
}

# 7.3 hero-structure SLM (lag)
rho_slm, beta_slm, logL_slm, AICc_slm, resid_slm = fit_slm_multi(
    y, X, W_list_hero
)
I_slm, p_slm = moran_I_val_from_array(resid_slm, WQ_array, IDs)
slm_res = {
    'model_type': 'SLM_lag',
    'W_mix': W_mix_str,
    'logL': logL_slm,
    'AICc': AICc_slm,
    'MoranI': I_slm,
    'p_norm': p_slm
}

# --------------------------------------------
# 8. Assemble appendix table & save
# --------------------------------------------

res_table = pd.DataFrame([ols_res, sem_res, slm_res])

# Sort: AICc first, then |MoranI|
res_table = res_table.sort_values(
    by=['AICc', res_table['MoranI'].abs().name],
    ascending=[True, True]
).reset_index(drop=True)

print(f"\n=== OLS vs hero SEM vs hero-structure SLM (US, {outcome_tag}, 2019) ===")
print(res_table)

out_csv = f"APPENDIX_SEM_vs_SLM_US_{outcome_tag}.csv"
res_table.to_csv(out_csv, index=False)
print(f"\n[Saved] {out_csv}")

#Results

##Figure 1

Figure 1. Analytical workflow showing data pipeline

No code needed

##Figure 2

In [ ]:
###############################################################################
# Figure 1 (4x3 maps) + Global Moran's I (with p-value) shown on each panel
###############################################################################
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from libpysal.weights import Queen
from esda.moran import Moran

# -----------------------------
# 1) Load data
# -----------------------------
shp_path = "US_HIV_Merged_total.shp"   # <-- update path
gdf = gpd.read_file(shp_path)
gdf["FIPS"] = gdf["FIPS"].astype(str).str.lstrip("0")

fields = [
    "FIPS", "GonorrheaR", "ChlamydiaR", "PercenHIVp",
    "Population", "UrbanRural", "Female", "Old", "Black",
    "Noinsuranc", "Poverty", "crime16to1", "Dissimilar", "geometry"
]
gdf = gdf[fields].dropna(subset=["geometry"]).copy()

# Dissolve to one geometry per FIPS (recommended if your shp has duplicates)
vis_gdf = gdf.dissolve(by="FIPS").reset_index()

# -----------------------------
# 2) Project to a suitable UTM (avoid distortion)
#    Use lon from EPSG:4326 centroid, then project to UTM
# -----------------------------
tmp_ll = vis_gdf.to_crs("EPSG:4326")
lon = tmp_ll.geometry.union_all().centroid.x
utm_zone = int((lon + 180) // 6) + 1
utm_crs = f"EPSG:326{utm_zone:02d}"
vis_gdf = vis_gdf.to_crs(utm_crs)

# -----------------------------
# 3) Variable mapping
# -----------------------------
field_mapping = {
    "GonorrheaR": "Gonorrhoea",
    "ChlamydiaR": "Chlamydia",
    "PercenHIVp": "HIV",
    "Population": "Population Density",
    "UrbanRural": "Urban–Rural",
    "Female": "Female",
    "Old": "Age 60+",
    "Black": "Black or African American",
    "Noinsuranc": "No Insurance",
    "Poverty": "Poverty",
    "crime16to1": "Crime Rate",
    "Dissimilar": "Dissimilarity"
}
fields_to_plot = list(field_mapping.keys())  # 12 variables

# -----------------------------
# 4) Global Moran's I helper
#    - compute on non-missing
#    - drop islands (no neighbors) for stability/interpretability
# -----------------------------
def global_moran_i(gdf_proj, var, permutations=999):
    gsub = gdf_proj[["geometry", var]].dropna().reset_index(drop=True)

    # build Queen contiguity
    w = Queen.from_dataframe(gsub, use_index=False)

    # drop islands for Moran's I (counties with no neighbors under contiguity)
    islands = getattr(w, "islands", [])
    if islands:
        gsub = gsub.drop(index=islands).reset_index(drop=True)
        w = Queen.from_dataframe(gsub, use_index=False)

    # row-standardize
    w.transform = "R"

    y = gsub[var].to_numpy(dtype=float)
    mi = Moran(y, w, permutations=permutations)
    return mi.I, mi.p_sim, len(y), len(islands)

# pre-compute Moran stats
moran_stats = {}
for f in fields_to_plot:
    I, p, n_used, n_islands = global_moran_i(vis_gdf, f, permutations=999)
    moran_stats[f] = (I, p, n_used, n_islands)

# -----------------------------
# 5) Plot 4x3 maps + annotate Moran's I and p
# -----------------------------
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(18, 22))
axes = axes.flatten()

missing_specs = {"color": "grey"}

for i, field in enumerate(fields_to_plot):
    ax = axes[i]
    vis_gdf.plot(
        column=field,
        ax=ax,
        cmap="viridis",
        legend=True,
        missing_kwds=missing_specs,
        legend_kwds={"shrink": 0.6}
    )

    I, p, n_used, n_islands = moran_stats[field]
    p_txt = "<0.001" if p < 0.001 else f"{p:.3f}"

    # Title with Moran's I and p-value
    ax.set_title(f"{field_mapping[field]}\nMoran's I={I:.3f}, p={p_txt}")
    ax.set_axis_off()

# hide unused axes (if any)
for j in range(len(fields_to_plot), len(axes)):
    axes[j].set_visible(False)

# Missing legend patch (figure-level)
missing_patch = Patch(facecolor="grey", edgecolor="grey", label="Missing")
fig.legend(handles=[missing_patch], loc="lower center", ncol=1, frameon=False, fontsize="large")

plt.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig("US_Figure1_12maps_MoranI.png", dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved to US_Figure1_12maps_MoranI.png")

##Figure 3

In [ ]:
# ============================================================
# Figure 3. Top bridging counties by rank increase
#
# Goal:
#   Plot top bridging counties for three STI outcomes:
#   HIV / Chlamydia / Gonorrhoea
#
# Map design:
#   - Grey: counties not included in the corresponding hero SEM sample
#   - Light blue: all bridging counties
#   - Red gradient: top bridging counties with delta_rank >= THRESH
#
# Output:
#   FIG3_top_bridging_delta_rank_1x3.png
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch

# -----------------------------
# 0. Global config
# -----------------------------

ROOT = os.getcwd()
OUTDIR = ROOT
os.makedirs(OUTDIR, exist_ok=True)

print("[INFO] Working directory:", ROOT)

# Base county-level shapefile
base_shp_path = os.path.join(OUTDIR, "US_HIV_Merged_total.shp")

# Hero SEM spillover outputs
spillover_files = {
    "HIV": os.path.join(
        OUTDIR, "US_spillover_HIV_SEM_add_2_WP_WQ.shp"
    ),
    "Chlamydia": os.path.join(
        OUTDIR, "US_spillover_Chlamydia_SEM_add_1_WP.shp"
    ),
    "Gonorrhea": os.path.join(
        OUTDIR, "US_spillover_Gonorrhea_SEM_add_1_WS.shp"
    ),
}

# Outcome order and display labels
outcome_order = ["HIV", "Chlamydia", "Gonorrhea"]

outcome_labels = {
    "HIV": "HIV",
    "Chlamydia": "Chlamydia",
    "Gonorrhea": "Gonorrhoea",
}

# Top bridging threshold
THRESH = 10

# Plot colors
missing_color = "#d9d9d9"
bridge_color = "#c6dbef"

# Figure style
plt.rcParams["font.size"] = 11
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"


# -----------------------------
# 1. Helper functions
# -----------------------------

def clean_fips(gdf, fips_col="FIPS"):
    """
    Clean county FIPS codes as strings.

    """
    if fips_col not in gdf.columns:
        raise KeyError(f"'{fips_col}' field not found.")

    gdf = gdf.copy()
    gdf[fips_col] = gdf[fips_col].astype(str).str.strip().str.lstrip("0")
    return gdf


def get_utm_crs(gdf):
    """
    Choose a UTM CRS based on the longitude of the study-area centroid.
    """
    if gdf.crs is None:
        raise ValueError("Input GeoDataFrame has no CRS. Please define CRS first.")

    gdf_ll = gdf.to_crs("EPSG:4326")

    # Compatible with both newer and older GeoPandas versions
    if hasattr(gdf_ll.geometry, "union_all"):
        centroid = gdf_ll.geometry.union_all().centroid
    else:
        centroid = gdf_ll.unary_union.centroid

    lon = centroid.x
    utm_zone = int((lon + 180) // 6) + 1
    return f"EPSG:326{utm_zone:02d}"


def load_base_map(shp_path):
    """
    Load the national county-level base map and create a dissolved boundary.

    Returns
    -------
    base_gdf : GeoDataFrame
        Projected county-level geometries.
    utm_crs : str
        Selected UTM CRS.
    boundary : GeoSeries
        Dissolved outer boundary for plotting.
    """
    if not os.path.exists(shp_path):
        raise FileNotFoundError(f"Base shapefile not found:\n{shp_path}")

    base_gdf = gpd.read_file(shp_path)
    base_gdf = clean_fips(base_gdf, "FIPS")
    base_gdf = base_gdf.dropna(subset=["geometry"]).copy()

    # Dissolve duplicates if any
    base_gdf = base_gdf.dissolve(by="FIPS").reset_index()

    utm_crs = get_utm_crs(base_gdf)
    base_gdf = base_gdf.to_crs(utm_crs)

    boundary = base_gdf.dissolve().boundary

    return base_gdf, utm_crs, boundary


def find_delta_rank_column(gdf):
    """
    Find the delta-rank column in a spillover shapefile.
    """
    if "delta_rank" in gdf.columns:
        return "delta_rank"

    candidates = [c for c in gdf.columns if "delta" in c.lower()]
    if len(candidates) == 0:
        raise KeyError("Cannot find a delta-rank column.")

    return candidates[0]


def identify_bridging_counties(gdf, threshold=10):
    """
    Identify bridging counties.

    Priority:
    1. Use 'is_bridging' if available.
    2. Use truncated shapefile field 'is_bridgin' if available.
    3. Fall back to spill_type == 'LowQ_HighB' and delta_rank >= threshold.
    """
    gdf = gdf.copy()

    if "is_bridging" in gdf.columns:
        bridge_col = "is_bridging"

    elif "is_bridgin" in gdf.columns:
        bridge_col = "is_bridgin"

    else:
        spill_candidates = [
            c for c in gdf.columns
            if c.lower().startswith("spill")
        ]

        if len(spill_candidates) == 0:
            raise KeyError(
                "Cannot find 'is_bridging', 'is_bridgin', or spill-type field."
            )

        spill_col = spill_candidates[0]

        # Harmonize older naming if needed
        gdf[spill_col] = gdf[spill_col].replace(
            {
                "HighQ_HighP": "HighQ_HighB",
                "HighQ_LowP": "HighQ_LowB",
                "LowQ_HighP": "LowQ_HighB",
                "LowQ_LowP": "LowQ_LowB",
            }
        )

        bridge_col = "is_bridge_fallback"
        gdf[bridge_col] = (
            (gdf[spill_col] == "LowQ_HighB")
            & (gdf["delta_rank"] >= threshold)
        )

    gdf["is_bridge"] = (
        gdf[bridge_col]
        .astype(str)
        .str.lower()
        .isin(["1", "true", "t", "yes"])
    )

    return gdf


def load_spillover_map(shp_path, utm_crs, threshold=10):
    """
    Load one pathogen-specific spillover shapefile and standardize fields.
    """
    if not os.path.exists(shp_path):
        raise FileNotFoundError(f"Spillover shapefile not found:\n{shp_path}")

    gdf = gpd.read_file(shp_path)
    gdf = clean_fips(gdf, "FIPS")
    gdf = gdf.dropna(subset=["geometry"]).copy()
    gdf = gdf.to_crs(utm_crs)

    delta_col = find_delta_rank_column(gdf)
    gdf["delta_rank"] = pd.to_numeric(gdf[delta_col], errors="coerce")

    gdf = identify_bridging_counties(gdf, threshold=threshold)

    return gdf


def plot_background(ax, base_gdf, boundary, model_fips):
    """
    Plot the shared map background for one panel.
    """
    # Counties outside the corresponding hero SEM sample
    missing_gdf = base_gdf[~base_gdf["FIPS"].isin(model_fips)].copy()

    if not missing_gdf.empty:
        missing_gdf.plot(
            ax=ax,
            color=missing_color,
            edgecolor="none",
            linewidth=0.0,
            zorder=1.0,
        )

    # Outer boundary only; do not draw county grids
    boundary.plot(
        ax=ax,
        edgecolor="black",
        linewidth=0.8,
        zorder=2.5,
    )


# -----------------------------
# 2. Load base map
# -----------------------------

base_gdf, utm_crs, us_boundary = load_base_map(base_shp_path)

print("[INFO] Base map loaded.")
print("[INFO] Number of counties in base map:", len(base_gdf))
print("[INFO] Plot CRS:", utm_crs)


# -----------------------------
# 3. Load spillover maps
# -----------------------------

spillover_gdfs = {}
top_delta_values = []

for outcome in outcome_order:
    shp_path = spillover_files[outcome]
    print(f"[INFO] Reading spillover shapefile for {outcome}:")
    print("       ", shp_path)

    gdf_spill = load_spillover_map(
        shp_path=shp_path,
        utm_crs=utm_crs,
        threshold=THRESH,
    )

    top_mask = (
        gdf_spill["is_bridge"]
        & (gdf_spill["delta_rank"] >= THRESH)
    )

    if top_mask.any():
        top_delta_values.append(
            gdf_spill.loc[top_mask, "delta_rank"].dropna().to_numpy()
        )

    spillover_gdfs[outcome] = gdf_spill

    print(
        f"[INFO] {outcome}: "
        f"n_model={len(gdf_spill)}, "
        f"n_bridge={gdf_spill['is_bridge'].sum()}, "
        f"n_top_bridge={top_mask.sum()}"
    )


# -----------------------------
# 4. Define global color scale
# -----------------------------

if len(top_delta_values) == 0:
    top_delta_values = [np.array([THRESH + 1.0])]

top_delta_values = np.concatenate(top_delta_values)

# Use the 98th percentile to avoid color compression by extreme values
vmax_global = np.nanquantile(top_delta_values, 0.98)
vmax_global = max(vmax_global, THRESH + 1.0)

norm = mpl.colors.Normalize(vmin=THRESH, vmax=vmax_global)
cmap = mpl.colormaps.get_cmap("Reds")

print(
    "[INFO] Global 98th percentile of delta_rank among top bridging counties:",
    f"{vmax_global:.1f}"
)


# -----------------------------
# 5. Plot 1 x 3 maps
# -----------------------------

fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(12, 4),
    sharex=True,
    sharey=True,
)

fig.subplots_adjust(
    left=0.03,
    right=0.88,
    top=0.96,
    bottom=0.18,
    wspace=0.02,
)

for ax, outcome in zip(axes, outcome_order):
    gdf_spill = spillover_gdfs[outcome]

    model_fips = set(gdf_spill["FIPS"])

    gdf_bridge = gdf_spill[gdf_spill["is_bridge"]].copy()
    gdf_top_bridge = gdf_bridge[
        gdf_bridge["delta_rank"] >= THRESH
    ].copy()

    # Background
    plot_background(
        ax=ax,
        base_gdf=base_gdf,
        boundary=us_boundary,
        model_fips=model_fips,
    )

    # All bridging counties
    if not gdf_bridge.empty:
        gdf_bridge.plot(
            ax=ax,
            color=bridge_color,
            edgecolor="none",
            linewidth=0.0,
            zorder=1.8,
        )

    # Top bridging counties
    if not gdf_top_bridge.empty:
        gdf_top_bridge.plot(
            ax=ax,
            column="delta_rank",
            cmap=cmap,
            norm=norm,
            edgecolor="none",
            linewidth=0.0,
            legend=False,
            zorder=2.2,
        )

    ax.set_title(outcome_labels[outcome], fontsize=12)
    ax.set_axis_off()
    ax.set_aspect("equal")


# -----------------------------
# 6. Colorbar and legend
# -----------------------------

# Colorbar
cax = fig.add_axes([0.90, 0.24, 0.015, 0.52])

cbar = mpl.colorbar.ColorbarBase(
    cax,
    cmap=cmap,
    norm=norm,
    orientation="vertical",
)

cbar.set_label(
    r"Rank increase $\Delta$rank (hero − $W_Q$)",
    fontsize=10,
)

cbar.ax.tick_params(labelsize=9)

# Figure-level legend
mid_value = THRESH + 0.5 * (vmax_global - THRESH)
top_bridge_color = cmap(norm(mid_value))

legend_handles = [
    Patch(
        facecolor=top_bridge_color,
        edgecolor="none",
        label=f"Top bridging counties (Δrank ≥ {THRESH})",
    ),
    Patch(
        facecolor=bridge_color,
        edgecolor="none",
        label="Bridging counties (all)",
    ),
    Patch(
        facecolor=missing_color,
        edgecolor="none",
        label="Missing counties",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=True,
    fontsize=9,
    bbox_to_anchor=(0.45, 0.03),
)


# -----------------------------
# 7. Save figure
# -----------------------------

fig_out = os.path.join(
    OUTDIR,
    "FIG3_top_bridging_delta_rank_1x3.png",
)

plt.savefig(
    fig_out,
    dpi=300,
    bbox_inches="tight",
)

print("[INFO] Saved Figure 3 to:")
print("       ", fig_out)

plt.show()

##Figure S1

In [ ]:
# ============================================================
# Supplementary Figure S1. Temporal robustness: 2018 vs 2019
#
# Goal:
#   Examine the year-to-year consistency of county-level STI rates
#   by comparing 2018 and 2019 values for:
#   HIV / Gonorrhoea / Chlamydia
#
# Output:
#   FIG_S1_STI_2018_vs_2019_scatter.png
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


# -----------------------------
# 0. Global config
# -----------------------------

ROOT = os.getcwd()
OUTDIR = ROOT
os.makedirs(OUTDIR, exist_ok=True)

print("[INFO] Working directory:", ROOT)

# Input shapefile
shp_path = os.path.join(ROOT, "US_HIV_Merged_total_final.shp")

# Outcome and covariate fields
outcome_cols_2019 = {
    "HIV": "PercenHIVp",
    "Gonorrhoea": "GonorrheaR",
    "Chlamydia": "ChlamydiaR",
}

outcome_cols_2018 = {
    "HIV": "RateHIV2018",
    "Gonorrhoea": "RateGonorrhea2018",
    "Chlamydia": "RateChlamydia2018",
}

# Truncated shapefile field-name repair
rename_map = {
    "RateChlamy": "RateChlamydia2018",
    "RateGonorr": "RateGonorrhea2018",
    "RateHIV201": "RateHIV2018",
}

# Optional covariates used elsewhere in the IJGIS workflow
x_cols = [
    "Population",
    "UrbanRural",
    "Female",
    "Old",
    "Black",
    "Noinsuranc",
    "Poverty",
    "crime16to1",
    "Dissimilar",
]

# Colors for each outcome
colors = {
    "HIV": "#1f77b4",
    "Gonorrhoea": "#ff7f0e",
    "Chlamydia": "#2ca02c",
}

# Figure style
plt.rcParams["font.size"] = 11
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"


# -----------------------------
# 1. Helper functions
# -----------------------------

def clean_fips(gdf, fips_col="FIPS"):
    """
    Clean county FIPS codes as strings.
    """
    if fips_col not in gdf.columns:
        raise KeyError(f"'{fips_col}' field not found in the shapefile.")

    gdf = gdf.copy()
    gdf[fips_col] = gdf[fips_col].astype(str).str.strip().str.lstrip("0")
    return gdf


def repair_truncated_fields(gdf, field_map):
    """
    Repair truncated shapefile field names if the full names are absent.
    """
    gdf = gdf.copy()

    for short_name, full_name in field_map.items():
        if short_name in gdf.columns and full_name not in gdf.columns:
            gdf = gdf.rename(columns={short_name: full_name})
            print(f"[INFO] Renamed truncated field: {short_name} -> {full_name}")

    return gdf


def get_state_fips(fips):
    """
    Extract two-digit state FIPS from county FIPS.
    """
    return str(fips).zfill(5)[:2]


def check_required_columns(gdf, required_cols):
    """
    Check whether required columns are present.
    """
    missing_cols = [c for c in required_cols if c not in gdf.columns]

    if len(missing_cols) > 0:
        raise KeyError(
            "The following required columns are missing:\n"
            + "\n".join(missing_cols)
        )


def print_temporal_correlations(df, pairs):
    """
    Print Pearson correlations between 2018 and 2019 values.
    """
    for label, col_2019, col_2018 in pairs:
        tmp = df[[col_2019, col_2018]].dropna()

        if len(tmp) >= 5:
            r = tmp[col_2019].corr(tmp[col_2018])
            print(
                f"[Temporal] Pearson r ({label}: 2018 vs 2019) "
                f"= {r:.3f} (n = {len(tmp)})"
            )
        else:
            print(
                f"[WARNING] {label}: fewer than 5 valid observations; "
                "correlation not reported."
            )


def get_common_axis_limits(df, pairs, padding_ratio=0.02):
    """
    Derive common x/y axis limits across all outcomes.
    """
    all_values = []

    for _, col_2019, col_2018 in pairs:
        tmp = df[[col_2019, col_2018]].dropna()

        if len(tmp) > 0:
            all_values.append(tmp[[col_2018, col_2019]].to_numpy())

    if len(all_values) == 0:
        raise ValueError("No valid 2018-2019 outcome pairs found for plotting.")

    all_values = np.vstack(all_values)

    vmin = np.nanmin(all_values)
    vmax = np.nanmax(all_values)

    padding = padding_ratio * (vmax - vmin)

    return vmin - padding, vmax + padding


# -----------------------------
# 2. Read and preprocess data
# -----------------------------

if not os.path.exists(shp_path):
    raise FileNotFoundError(f"Input shapefile not found:\n{shp_path}")

gdf = gpd.read_file(shp_path)
gdf = clean_fips(gdf, "FIPS")
gdf = repair_truncated_fields(gdf, rename_map)

# Remove duplicated counties if any
gdf = gdf.drop_duplicates(subset=["FIPS"]).copy()

# Build required field list
required_cols = (
    ["FIPS", "geometry"]
    + list(outcome_cols_2019.values())
    + list(outcome_cols_2018.values())
)

# Keep covariates only if they exist
available_x_cols = [c for c in x_cols if c in gdf.columns]

check_required_columns(gdf, required_cols)

cols_keep = required_cols + available_x_cols
df_geo = gdf[cols_keep].copy()

# Add state FIPS
df_geo["STATE"] = df_geo["FIPS"].apply(get_state_fips)

df_geo = gpd.GeoDataFrame(
    df_geo,
    geometry="geometry",
    crs=gdf.crs,
)

print("[INFO] Data loaded.")
print("[INFO] Number of counties:", len(df_geo))


# -----------------------------
# 3. Prepare temporal pairs
# -----------------------------

temporal_pairs = []

for label in ["HIV", "Gonorrhoea", "Chlamydia"]:
    col_2019 = outcome_cols_2019[label]
    col_2018 = outcome_cols_2018[label]
    temporal_pairs.append((label, col_2019, col_2018))

print_temporal_correlations(df_geo, temporal_pairs)

axis_min, axis_max = get_common_axis_limits(
    df=df_geo,
    pairs=temporal_pairs,
    padding_ratio=0.02,
)

print(
    f"[INFO] Common axis range: "
    f"{axis_min:.3f} to {axis_max:.3f}"
)


# -----------------------------
# 4. Plot temporal robustness figure
# -----------------------------

fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(12, 4),
    sharex=True,
    sharey=True,
)

for ax, (label, col_2019, col_2018) in zip(axes, temporal_pairs):
    tmp = df_geo[[col_2019, col_2018]].dropna().copy()

    x = tmp[col_2018].to_numpy()
    y = tmp[col_2019].to_numpy()
    n = len(tmp)

    r = tmp[col_2019].corr(tmp[col_2018])

    # Scatter points
    ax.scatter(
        x,
        y,
        s=10,
        alpha=0.35,
        color=colors[label],
        edgecolors="none",
    )

    # 45-degree reference line
    ax.plot(
        [axis_min, axis_max],
        [axis_min, axis_max],
        color="black",
        linestyle="--",
        linewidth=1.0,
    )

    # Common axis limits
    ax.set_xlim(axis_min, axis_max)
    ax.set_ylim(axis_min, axis_max)

    # In-panel annotation
    ax.text(
        0.04,
        0.96,
        f"r = {r:.3f}\nn = {n}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            alpha=0.75,
            edgecolor="none",
        ),
    )

    ax.set_title(label, fontsize=12)
    ax.set_xlabel("2018 rate")

    if ax is axes[0]:
        ax.set_ylabel("2019 rate")


plt.tight_layout()


# -----------------------------
# 5. Save figure
# -----------------------------

fig_out = os.path.join(
    OUTDIR,
    "FIG_S1_STI_2018_vs_2019_scatter.png",
)

plt.savefig(
    fig_out,
    dpi=300,
    bbox_inches="tight",
)

print("[INFO] Saved temporal robustness figure to:")
print("       ", fig_out)

plt.show()

##Table 1

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from libpysal.weights import Queen

###############################################################################
# Helpers
###############################################################################
def _row_standardize(mat: np.ndarray) -> np.ndarray:
    """Row-standardize with zero-row protection."""
    mat = mat.astype(float, copy=True)
    row_sums = mat.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return mat / row_sums

def _symmetrize(mat: np.ndarray, mode: str = "mean") -> np.ndarray:
    """Symmetrize an NxN matrix; diagonal preserved for now (we will zero it later)."""
    if mode == "mean":
        return 0.5 * (mat + mat.T)
    if mode == "sum":
        return mat + mat.T
    if mode == "max":
        return np.maximum(mat, mat.T)
    raise ValueError("mode must be one of {'mean','sum','max'}")

def summarize_matrix(mat: np.ndarray) -> dict:
    """Summary stats computed on the full matrix (including zeros), consistent with your Table 1 style."""
    flat = mat.ravel()
    return {
        "Mean": float(np.mean(flat)),
        "SD": float(np.std(flat, ddof=1)),
        "Min": float(np.min(flat)),
        "Max": float(np.max(flat)),
        "NonzeroShare": float(np.mean(flat != 0.0)),
    }

###############################################################################
# 1. Read and preprocess shapefile (unique FIPS, project to UTM)
###############################################################################
shp_path = "US_HIV_Merged_total.shp"  # <-- update
gdf = gpd.read_file(shp_path)

gdf["FIPS"] = gdf["FIPS"].astype(str).str.lstrip("0")
vis_gdf = gdf.dissolve(by="FIPS").reset_index()

centroid = vis_gdf.unary_union.centroid
utm_zone = int((centroid.x + 180) // 6) + 1
utm_crs = f"EPSG:326{utm_zone:02d}"
vis_gdf = vis_gdf.to_crs(utm_crs)

ids = sorted(vis_gdf["FIPS"].tolist())
n = len(ids)

###############################################################################
# 2. PCI-based matrix (RAW + R)
###############################################################################
pci_df = pd.read_csv("US_County_PCI_2019.csv")  # <-- update
pci_df["place_i"] = pci_df["place_i"].astype(str).str.lstrip("0")
pci_df["place_j"] = pci_df["place_j"].astype(str).str.lstrip("0")

mask = pci_df["place_i"].isin(ids) & pci_df["place_j"].isin(ids)
tmp = pci_df[mask].pivot_table(index="place_i", columns="place_j", values="pci", fill_value=0.0)
tmp = tmp.reindex(index=ids, columns=ids, fill_value=0.0)
mat_pci_raw = tmp.values.astype(float)

# Symmetrize and zero diagonal (so "raw" matches your method description)
mat_pci_raw = _symmetrize(mat_pci_raw, mode="mean")
np.fill_diagonal(mat_pci_raw, 0.0)

mat_pci_R = _row_standardize(mat_pci_raw)

###############################################################################
# 3. SCI-based matrix (RAW + R)
###############################################################################
sci_df = pd.read_csv("processed_sci_summary.csv")  # <-- update
sci_df["user_loc"] = sci_df["user_loc"].astype(str).str.lstrip("0")
sci_df["tfr_loc"] = sci_df["tfr_loc"].astype(str).str.lstrip("0")

tmp = sci_df.pivot_table(index="user_loc", columns="tfr_loc", values="tscaled_sci", fill_value=0.0)
tmp = tmp.reindex(index=ids, columns=ids, fill_value=0.0)
mat_sci_raw = tmp.values.astype(float)

# Symmetrize and zero diagonal
mat_sci_raw = _symmetrize(mat_sci_raw, mode="mean")
np.fill_diagonal(mat_sci_raw, 0.0)

mat_sci_R = _row_standardize(mat_sci_raw)

###############################################################################
# 4. Distance-based matrix (<=200 km): RAW (1/d) + R
###############################################################################
coords = np.vstack(vis_gdf.geometry.centroid.apply(lambda p: (p.x, p.y)))
dist = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=2)

mask_d = (dist > 0) & (dist <= 200_000)  # UTM meters: 200 km
mat_dist_raw = np.where(mask_d, 1.0 / dist, 0.0)
np.fill_diagonal(mat_dist_raw, 0.0)

mat_dist_R = _row_standardize(mat_dist_raw)

###############################################################################
# 5. Queen contiguity: RAW (binary) + R
###############################################################################
wq = Queen.from_dataframe(vis_gdf, use_index=False)

# RAW: binary adjacency (pre-row-standardization)
wq.transform = "B"
mat_queen_raw = wq.full()[0].astype(float)
np.fill_diagonal(mat_queen_raw, 0.0)

# R: row-standardized adjacency
wq.transform = "R"
mat_queen_R = wq.full()[0].astype(float)
np.fill_diagonal(mat_queen_R, 0.0)

###############################################################################
# 6. Summary tables: BEFORE standardization vs AFTER row-standardization
###############################################################################
mats_raw = {
    "PCI-based (raw)": mat_pci_raw,
    "SCI-based (raw)": mat_sci_raw,
    "Distance-based (raw)": mat_dist_raw,
    "Queen-based (raw, binary)": mat_queen_raw,
}

mats_R = {
    "PCI-based (R)": mat_pci_R,
    "SCI-based (R)": mat_sci_R,
    "Distance-based (R)": mat_dist_R,
    "Queen-based (R)": mat_queen_R,
}

raw_rows = []
for k, m in mats_raw.items():
    s = summarize_matrix(m)
    s["Measure"] = k
    raw_rows.append(s)

R_rows = []
for k, m in mats_R.items():
    s = summarize_matrix(m)
    s["Measure"] = k
    R_rows.append(s)

summary_raw_df = pd.DataFrame(raw_rows).set_index("Measure")
summary_R_df = pd.DataFrame(R_rows).set_index("Measure")

print("\n=== Summary statistics BEFORE row-standardization (raw / unstandardized) ===")
print(summary_raw_df)

print("\n=== Summary statistics AFTER row-standardization (R) ===")
print(summary_R_df)

In [ ]:
# ============================================================
# Supplementary analysis. Correlation and VIF among spatial
# weight matrices
#
# Goal:
#   Compare four spatial weight matrices:
#   W_PCI, W_SCI, W_Distance, and W_Queen
#
# Outputs:
#   - Correlation matrix among flattened off-diagonal weights
#   - VIF values for the four weight vectors
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

from libpysal.weights import util, W, Queen
from statsmodels.stats.outliers_influence import variance_inflation_factor


# -----------------------------
# 0. Global config
# -----------------------------

ROOT = os.getcwd()
OUTDIR = ROOT
os.makedirs(OUTDIR, exist_ok=True)

print("[INFO] Working directory:", ROOT)

# Input files
shp_path = os.path.join(ROOT, "US_HIV_Merged_total.shp")
pci_file_path = os.path.join(ROOT, "US_County_PCI_2019.csv")
sci_file_path = os.path.join(ROOT, "processed_sci_summary.csv")

# Required fields
outcome_cols = [
    "GonorrheaR",
    "ChlamydiaR",
    "PercenHIVp",
]

x_cols = [
    "Population",
    "UrbanRural",
    "Female",
    "Old",
    "Black",
    "Noinsuranc",
    "Poverty",
    "crime16to1",
    "Dissimilar",
]

coord_cols = ["X", "Y"]

required_cols = [
    "FIPS",
    *outcome_cols,
    *x_cols,
    *coord_cols,
    "geometry",
]

EPSILON = 1e-10


# -----------------------------
# 1. Helper functions
# -----------------------------

def clean_fips_series(s):
    """
    Clean FIPS codes as strings and remove leading zeros.
    """
    return s.astype(str).str.strip().str.lstrip("0")


def check_required_columns(df, required):
    """
    Check whether all required columns are present.
    """
    missing = [c for c in required if c not in df.columns]

    if len(missing) > 0:
        raise KeyError(
            "The following required columns are missing:\n"
            + "\n".join(missing)
        )


def subset_W(w_obj, subset_ids):
    """
    Subset a libpysal W object to a given set of IDs.
    """
    subset_ids = set(subset_ids)

    new_neighbors = {}
    new_weights = {}

    for i in w_obj.id_order:
        if i not in subset_ids:
            continue

        old_neighbors = w_obj.neighbors[i]
        old_weights = w_obj.weights[i]

        filtered_neighbors = []
        filtered_weights = []

        for nb, wt in zip(old_neighbors, old_weights):
            if nb in subset_ids:
                filtered_neighbors.append(nb)
                filtered_weights.append(wt)

        new_neighbors[i] = filtered_neighbors
        new_weights[i] = filtered_weights

    return W(
        new_neighbors,
        new_weights,
        ids=list(subset_ids),
    )


def row_normalize_matrix(mat):
    """
    Row-normalize a dense matrix.
    """
    mat = np.asarray(mat, dtype=float)

    row_sum = mat.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0

    mat_norm = mat / row_sum
    mat_norm = np.nan_to_num(
        mat_norm,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    return mat_norm


def dense_matrix_to_W(mat, ids):
    """
    Convert a dense matrix to a libpysal W object.
    """
    return util.full2W(
        np.asarray(mat, dtype=float),
        ids=list(ids),
    )


def flatten_off_diagonal(mat):
    """
    Extract off-diagonal values from a square matrix.
    """
    mat = np.asarray(mat, dtype=float)

    if mat.shape[0] != mat.shape[1]:
        raise ValueError("Input matrix must be square.")

    mask = ~np.eye(mat.shape[0], dtype=bool)
    return mat[mask]


def build_pci_weight(hiv_df, pci_path):
    """
    Build row-normalized PCI-based spatial weight matrix.
    """
    if not os.path.exists(pci_path):
        raise FileNotFoundError(f"PCI file not found:\n{pci_path}")

    pci_df = pd.read_csv(pci_path)

    check_required_columns(pci_df, ["place_i", "place_j", "pci"])

    pci_df["place_i"] = clean_fips_series(pci_df["place_i"])
    pci_df["place_j"] = clean_fips_series(pci_df["place_j"])

    valid_fips = set(hiv_df["FIPS"])

    pci_df = pci_df[
        pci_df["place_i"].isin(valid_fips)
        & pci_df["place_j"].isin(valid_fips)
    ].copy()

    adj = pci_df.pivot_table(
        index="place_i",
        columns="place_j",
        values="pci",
        fill_value=0,
    )

    np.fill_diagonal(adj.values, 0)

    common_ids = sorted(set(hiv_df["FIPS"]).intersection(adj.index))

    adj = adj.reindex(
        index=common_ids,
        columns=common_ids,
        fill_value=0,
    )

    mat_norm = row_normalize_matrix(adj.values)

    return dense_matrix_to_W(mat_norm, adj.index)


def build_sci_weight(hiv_df, sci_path):
    """
    Build row-normalized SCI-based spatial weight matrix.
    """
    if not os.path.exists(sci_path):
        raise FileNotFoundError(f"SCI file not found:\n{sci_path}")

    sci_df = pd.read_csv(sci_path)

    check_required_columns(sci_df, ["user_loc", "tfr_loc", "tscaled_sci"])

    sci_df["user_loc"] = clean_fips_series(sci_df["user_loc"])
    sci_df["tfr_loc"] = clean_fips_series(sci_df["tfr_loc"])

    valid_fips = set(hiv_df["FIPS"])

    sci_df = sci_df[
        sci_df["user_loc"].isin(valid_fips)
        & sci_df["tfr_loc"].isin(valid_fips)
    ].copy()

    adj = sci_df.pivot_table(
        index="user_loc",
        columns="tfr_loc",
        values="tscaled_sci",
        fill_value=0,
    )

    # Make the SCI matrix symmetric if needed
    sci_np = adj.values
    sci_np = sci_np + sci_np.T - np.diag(np.diag(sci_np))

    adj = pd.DataFrame(
        sci_np,
        index=adj.index,
        columns=adj.columns,
    )

    np.fill_diagonal(adj.values, 0)

    common_ids = sorted(set(hiv_df["FIPS"]).intersection(adj.index))

    adj = adj.reindex(
        index=common_ids,
        columns=common_ids,
        fill_value=0,
    )

    mat_norm = row_normalize_matrix(adj.values)

    return dense_matrix_to_W(mat_norm, adj.index)


def build_distance_weight(hiv_df, normalized=True):
    """
    Build inverse-distance spatial weight matrix.

    Parameters
    ----------
    normalized : bool
        If True, return row-normalized inverse-distance weights.
        If False, return raw inverse-distance weights.
    """
    coords = hiv_df[["X", "Y"]].to_numpy(dtype=float)

    distance_matrix = np.linalg.norm(
        coords[:, np.newaxis, :] - coords[np.newaxis, :, :],
        axis=2,
    )

    weight_matrix = 1.0 / (distance_matrix + EPSILON)
    np.fill_diagonal(weight_matrix, 0)

    if normalized:
        weight_matrix = row_normalize_matrix(weight_matrix)
    else:
        weight_matrix = np.nan_to_num(
            weight_matrix,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

    return pd.DataFrame(
        weight_matrix,
        index=hiv_df["FIPS"],
        columns=hiv_df["FIPS"],
    )


def build_queen_weight(hiv_df):
    """
    Build row-standardized Queen contiguity weights.
    """
    hiv_df_proj = hiv_df.to_crs(epsg=3857)

    w_queen = Queen.from_dataframe(
        hiv_df_proj,
        ids=hiv_df_proj["FIPS"],
    )

    w_queen.transform = "R"

    return w_queen


# -----------------------------
# 2. Read and preprocess shapefile
# -----------------------------

if not os.path.exists(shp_path):
    raise FileNotFoundError(f"Input shapefile not found:\n{shp_path}")

gdf = gpd.read_file(shp_path)

check_required_columns(gdf, required_cols)

gdf["FIPS"] = clean_fips_series(gdf["FIPS"])
gdf = gdf.drop_duplicates(subset=["FIPS"]).copy()

hiv_df = gdf[required_cols].copy()
hiv_df = gpd.GeoDataFrame(
    hiv_df,
    geometry="geometry",
    crs=gdf.crs,
)

print("[INFO] Input data loaded.")
print("[INFO] Number of unique counties:", len(hiv_df))
print(hiv_df.head())


# -----------------------------
# 3. Build four spatial weight matrices
# -----------------------------

# PCI-based W
w_pci = build_pci_weight(
    hiv_df=hiv_df,
    pci_path=pci_file_path,
)

print("[INFO] PCI-based W_PCI.n =", w_pci.n)

# SCI-based W
w_sci = build_sci_weight(
    hiv_df=hiv_df,
    sci_path=sci_file_path,
)

print("[INFO] SCI-based W_SCI.n =", w_sci.n)

# Distance-based W
distance_norm_df = build_distance_weight(
    hiv_df=hiv_df,
    normalized=True,
)

w_distance = dense_matrix_to_W(
    distance_norm_df.values,
    distance_norm_df.index,
)

print("[INFO] Distance-based W_Distance.n =", w_distance.n)

# Queen-based W
w_queen = build_queen_weight(hiv_df)

print("[INFO] Queen-based W_Queen.n =", w_queen.n)


# -----------------------------
# 4. Intersect IDs across matrices
# -----------------------------

common_ids = (
    set(w_pci.id_order)
    & set(w_sci.id_order)
    & set(w_distance.id_order)
    & set(w_queen.id_order)
    & set(hiv_df["FIPS"])
)

common_ids = sorted(common_ids)

print("[INFO] Common ID size:", len(common_ids))

hiv_df = hiv_df[hiv_df["FIPS"].isin(common_ids)].copy()

# Keep a stable FIPS order
hiv_df["FIPS"] = pd.Categorical(
    hiv_df["FIPS"],
    categories=common_ids,
    ordered=True,
)

hiv_df = hiv_df.sort_values("FIPS").copy()
hiv_df["FIPS"] = hiv_df["FIPS"].astype(str)

w_pci = subset_W(w_pci, common_ids)
w_sci = subset_W(w_sci, common_ids)
w_distance = subset_W(w_distance, common_ids)
w_queen = subset_W(w_queen, common_ids)

print("[INFO] Final data size:", len(hiv_df))
print("[INFO] Final W_PCI.n =", w_pci.n)
print("[INFO] Final W_SCI.n =", w_sci.n)
print("[INFO] Final W_Distance.n =", w_distance.n)
print("[INFO] Final W_Queen.n =", w_queen.n)


# -----------------------------
# 5. Extract comparable weight vectors
# -----------------------------

arr_pci, ids_pci = w_pci.full()
arr_sci, ids_sci = w_sci.full()
arr_queen, ids_queen = w_queen.full()

# For distance, use raw inverse-distance weights as in the original code
raw_distance_df = build_distance_weight(
    hiv_df=hiv_df,
    normalized=False,
)

arr_distance = raw_distance_df.reindex(
    index=ids_pci,
    columns=ids_pci,
).fillna(0).values

weights_df = pd.DataFrame(
    {
        "PCI": flatten_off_diagonal(arr_pci),
        "SCI": flatten_off_diagonal(arr_sci),
        "Distance": flatten_off_diagonal(arr_distance),
        "Queen": flatten_off_diagonal(arr_queen),
    }
)


# -----------------------------
# 6. Correlation matrix and VIF
# -----------------------------

corr_matrix = weights_df.corr()

print("\n[RESULT] Correlation matrix among spatial weight matrices:")
print(corr_matrix)

X_vif = weights_df.to_numpy(dtype=float)

vif_df = pd.DataFrame(
    {
        "variable": weights_df.columns,
        "VIF": [
            variance_inflation_factor(X_vif, i)
            for i in range(X_vif.shape[1])
        ],
    }
)

print("\n[RESULT] VIF values among spatial weight matrices:")
print(vif_df)


# -----------------------------
# 7. Save outputs
# -----------------------------

corr_out = os.path.join(
    OUTDIR,
    "TABLE_S_weight_matrix_correlation.csv",
)

vif_out = os.path.join(
    OUTDIR,
    "TABLE_S_weight_matrix_vif.csv",
)

corr_matrix.to_csv(corr_out)
vif_df.to_csv(vif_out, index=False)

print("\n[INFO] Saved correlation matrix to:")
print("       ", corr_out)

print("[INFO] Saved VIF table to:")
print("       ", vif_out)

##Key Table 2 & 3 & 4

###KEY Multi-network spatial error workflow for three STI outcomes

In [ ]:
# ============================================================
# Multi-network spatial error workflow for three STI outcomes
#
# This script combines the original outcome-specific workflows
# into one shared workflow. It runs the same analysis for:
#   1) HIV        -> PercenHIVp
#   2) Gonorrhea  -> GonorrheaR
#   3) Chlamydia  -> ChlamydiaR
#
# To run only selected outcomes, edit OUTCOMES_TO_RUN below.
# ============================================================

# ============================================
# 0. Imports & Global configs
# ============================================
import numpy as np
import pandas as pd
import geopandas as gpd

from itertools import combinations
from numpy.linalg import slogdet
from scipy.optimize import minimize

from esda.moran import Moran
from libpysal.weights import util, W, Queen
from joblib import Parallel, delayed

import os

np.seterr(all='ignore')

RANDOM_SEED = 7
rng_global = np.random.default_rng(RANDOM_SEED)

# Parallel and production-mode settings
N_JOBS = -1        # -1 uses all cores; set to 1 if parallel execution is unstable
RUN_PROD = False   # Set True for larger iterations / B_NULL / B_PERT in production runs

# ------------- Outcome configuration: switch STI outcome in one line ----------------
# Options: 'PercenHIVp', 'GonorrheaR', 'ChlamydiaR'

# ------------- Outcome configuration ----------------
# Available options:
#   'PercenHIVp'  -> HIV
#   'GonorrheaR'  -> Gonorrhea
#   'ChlamydiaR'  -> Chlamydia
OUTCOMES_TO_RUN = [
    'PercenHIVp',
    'GonorrheaR',
    'ChlamydiaR',
]

# Output folder for shared reproducible results
OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Keep False for the shared script unless geospatial files are explicitly needed.
# The core shared outputs are CSV tables only; no figures are generated.
EXPORT_SPILLOVER_SHAPEFILE = False


def outpath(filename):
    """Return a file path inside OUTPUT_DIR."""
    return os.path.join(OUTPUT_DIR, filename)


def run_one_outcome(y_col):
    """
    Run the full IJGIS multi-network SEM workflow for one STI outcome.

    Parameters
    ----------
    y_col : str
        Outcome field to analyze. Expected values are:
        'PercenHIVp', 'GonorrheaR', or 'ChlamydiaR'.
    """
    print("\n" + "=" * 80)
    print(f"[RUN] Starting workflow for outcome column: {y_col}")
    print("=" * 80)


    OUTCOME_TAG = {
        'PercenHIVp': 'HIV',
        'GonorrheaR': 'Gonorrhea',
        'ChlamydiaR': 'Chlamydia'
    }
    outcome_tag = OUTCOME_TAG.get(y_col, y_col)

    # Mapping from 2019 outcome fields to the corresponding 2018 fields for temporal robustness
    TEMPORAL_2018_COL = {
        'PercenHIVp': 'RateHIV2018',
        'GonorrheaR': 'RateGonorrhea2018',
        'ChlamydiaR': 'RateChlamydia2018'
    }
    outcome_2018_col = TEMPORAL_2018_COL.get(y_col, None)

    # ============================================
    # 1. Read US shapefile & basic preprocessing
    # ============================================

    # 1.1 Read the US shapefile; field structure follows the DeepSouth workflow
    shp_path = "US_HIV_Merged_total_final.shp"

    gdf = gpd.read_file(shp_path)
    if 'FIPS' not in gdf.columns:
        raise KeyError("Shapefile needs a 'FIPS' field.")

    # Remove leading zeros from FIPS codes and drop duplicate county records
    gdf['FIPS'] = gdf['FIPS'].astype(str).str.lstrip('0')
    gdf = gdf.drop_duplicates(subset=['FIPS'])

    # Repair potentially truncated 2018 field names
    rename_map = {
        'RateChlamy': 'RateChlamydia2018',
        'RateGonorr': 'RateGonorrhea2018',
        'RateHIV201': 'RateHIV2018'
    }
    for short, long_name in rename_map.items():
        if (short in gdf.columns) and (long_name not in gdf.columns):
            gdf = gdf.rename(columns={short: long_name})

    # Covariates
    X_cols = [
        'Population','UrbanRural','Female','Old','Black',
        'Noinsuranc','Poverty','crime16to1','Dissimilar'
    ]

    cols_keep = [
        'FIPS',
        'GonorrheaR','ChlamydiaR','PercenHIVp',
        *X_cols,
        'RateChlamydia2018','RateGonorrhea2018','RateHIV2018',
        'geometry'
    ]

    df_geo = gdf[cols_keep].copy()

    # Drop rows with missing current 2019 outcome or covariates
    df_geo = df_geo.dropna(subset=[y_col] + X_cols)

    # State code
    def fips2state(x):
        return str(x).zfill(5)[:2]

    df_geo['STATE'] = df_geo['FIPS'].apply(fips2state)
    df_geo = gpd.GeoDataFrame(df_geo, geometry='geometry', crs=gdf.crs)

    # 1.2 Simple year-to-year correlation between 2018 and 2019 outcomes (US)
    def _print_temporal_corr(col_2019, col_2018, label):
        if {col_2019, col_2018}.issubset(df_geo.columns):
            tmp = df_geo[[col_2019, col_2018]].dropna()
            if len(tmp) >= 5:
                r = tmp[col_2019].corr(tmp[col_2018])
                print(f"[Temporal] Pearson r ({label} 2018 vs 2019) = {r:.3f} (n={len(tmp)})")

    _print_temporal_corr('PercenHIVp',        'RateHIV2018',        'HIV')
    _print_temporal_corr('GonorrheaR',        'RateGonorrhea2018', 'Gonorrhea')
    _print_temporal_corr('ChlamydiaR',        'RateChlamydia2018', 'Chlamydia')


    # ============================================
    # 2. Build spatial weights & design matrix (US)
    # ============================================

    # 2.1 WP：Twitter PCI mobility
    pci_path = "US_County_PCI_2019.csv"
    pci = pd.read_csv(pci_path)

    pci['place_i'] = pci['place_i'].astype(str).str.lstrip('0')
    pci['place_j'] = pci['place_j'].astype(str).str.lstrip('0')

    pci_f = pci[
        pci['place_i'].isin(df_geo['FIPS']) &
        pci['place_j'].isin(df_geo['FIPS'])
    ].copy()

    A_pci = pci_f.pivot_table(
        index='place_i', columns='place_j',
        values='pci', fill_value=0
    )
    np.fill_diagonal(A_pci.values, 0.0)
    A = A_pci.values.astype(float)
    A = A + A.T - np.diag(np.diag(A))        # Symmetrize
    row_sum = A.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    WP = A / row_sum                         # Row-standardize
    wP = util.full2W(WP, ids=list(A_pci.index))

    # 2.2 WS：Facebook SCI social ties
    sci_path = "processed_sci_summary.csv"
    sci = pd.read_csv(sci_path)

    sci['user_loc'] = sci['user_loc'].astype(str).str.lstrip('0')
    sci['tfr_loc']  = sci['tfr_loc'].astype(str).str.lstrip('0')

    sci_f = sci[
        sci['user_loc'].isin(df_geo['FIPS']) &
        sci['tfr_loc'].isin(df_geo['FIPS'])
    ].copy()

    A_sci = sci_f.pivot_table(
        index='user_loc', columns='tfr_loc',
        values='tscaled_sci', fill_value=0
    )
    np.fill_diagonal(A_sci.values, 0.0)
    B = A_sci.values.astype(float)
    B = B + B.T - np.diag(np.diag(B))
    row_sum = B.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    WS = B / row_sum
    wS = util.full2W(WS, ids=list(A_sci.index))

    # 2.3 WQ: Queen contiguity (US)
    g3857 = df_geo.to_crs(epsg=3857)
    wQ = Queen.from_dataframe(g3857, ids=list(df_geo['FIPS']))
    wQ.transform = 'R'
    WQ, _ = wQ.full()

    # 2.4 Align ID sets and ordering across WP / WS / WQ / df_geo
    common = (
        set(df_geo['FIPS']) &
        set(wP.id_order) &
        set(wS.id_order) &
        set(wQ.id_order)
    )

    def subset_W(w: W, keep_ids):
        keep_ids = list(dict.fromkeys([str(x) for x in keep_ids]))
        keep = set(keep_ids)
        new_neighbors, new_weights = {}, {}
        for i in w.id_order:
            if i not in keep:
                continue
            nbs = w.neighbors.get(i, [])
            wts = w.weights.get(i, [])
            kn, kw = [], []
            for nb, wt in zip(nbs, wts):
                if nb in keep:
                    kn.append(nb); kw.append(wt)
            new_neighbors[i] = kn
            new_weights[i] = kw
        return W(new_neighbors, new_weights, ids=keep_ids)

    df_geo = df_geo[df_geo['FIPS'].isin(common)].copy()
    wP = subset_W(wP, common)
    wS = subset_W(wS, common)
    wQ = subset_W(wQ, common)

    df_geo = df_geo.set_index('FIPS').loc[wQ.id_order].reset_index()
    IDs   = list(wQ.id_order)
    STATE = df_geo['STATE'].values

    # Extract aligned dense matrices
    WP, _ = wP.full()
    WS, _ = wS.full()
    WQ, _ = wQ.full()

    # Aligned projected coordinates (EPSG:3857)
    XY_all = np.vstack([
        g3857.set_index('FIPS').loc[IDs].geometry.centroid.x.values,
        g3857.set_index('FIPS').loc[IDs].geometry.centroid.y.values
    ]).T

    # 2.5 Build outcome and covariate matrices for the current 2019 STI outcome
    df = df_geo[['FIPS', y_col] + X_cols + ['STATE','geometry']].dropna().copy()

    y = df[y_col].values
    X_raw = df[X_cols].values
    N = X_raw.shape[0]
    X = np.hstack([np.ones((N, 1)), X_raw])   # Add intercept
    K = X.shape[1]

    # ============================================
    # 3. SEM core helpers + Stage-1 & Stage-2 + Gate
    # ============================================

    # 3.1 Core helper functions --------------------------------------------------------------

    def fit_ols(y, X):
        """
        Fit standard OLS as the baseline model.
        Returns beta, residuals, log-likelihood, and AICc.
        """
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        e = y - X.dot(beta)
        sse = float(e.T.dot(e))
        N_, K_ = X.shape
        df_ = N_ - K_
        sigma2 = sse / df_
        logL = -(N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
        AIC  = -2*logL + 2*K_
        AICc = (AIC + (2*K_*(K_+1)) / (N_-K_-1)
                if (N_-K_-1) > 0 else np.nan)
        return {
            'model': 'OLS',
            'beta': beta,
            'logL': logL,
            'AICc': AICc,
            'resid': e
        }


    def build_A_and_logdet(lam, W_list, combo='add'):
        """
        Build the SEM transformation matrix A = I - sum(lambda_k W_k) for add,
        or the product transformation for prod.
        """
        N_ = W_list[0].shape[0]
        I = np.eye(N_)

        if combo == 'add':
            A = I.copy()
            for i, Wk in enumerate(W_list):
                A -= lam[i] * Wk
            sign, logdetA = slogdet(A)
            if sign <= 0:
                return None, None
            return A, logdetA

        elif combo == 'prod':
            A = I.copy()
            logdet_sum = 0.0
            for i, Wk in enumerate(W_list):
                Mi = I - lam[i] * Wk
                sign_i, logdet_i = slogdet(Mi)
                if sign_i <= 0:
                    return None, None
                logdet_sum += logdet_i
                A = Mi @ A
            return A, logdet_sum

        else:
            raise ValueError("combo must be 'add' or 'prod'")


    def neglog_sem(params, y, X, W_list, combo='add'):
        """
        Negative log-likelihood for SEM optimization.
        params = [λ_1,...,λ_m, β_0,...,β_{K-1}]
        """
        N_, K_ = X.shape
        m = len(W_list)
        lam = params[:m]
        beta = params[m:]

        A, logdetA = build_A_and_logdet(lam, W_list, combo=combo)
        if A is None:
            return 1e12

        e = y - X.dot(beta)
        v = A.dot(e)
        sse = float(v.T.dot(v))

        df_ = N_ - (K_ + m)
        if df_ <= 0:
            return 1e12

        sigma2 = sse / df_
        if sigma2 <= 0:
            return 1e12

        logL = logdetA - (N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
        return -logL


    def fit_sem_multi(y, X, W_list, method='L-BFGS-B', combo='add'):
        """
        Fit a full multi-weight SEM for Stage-1 and LOSO validation.
        """
        N_, K_ = X.shape
        m = len(W_list)

        # Initial values: OLS beta and lambda = 0
        beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
        x0 = np.r_[np.zeros(m), beta0]
        bounds = [(-0.99, 0.99)]*m + [(-np.inf, np.inf)]*K_

        res = minimize(
            neglog_sem, x0,
            args=(y, X, W_list, combo),
            method=method,
            bounds=bounds,
            options={'maxiter': 200, 'ftol': 1e-4, 'gtol': 1e-4}
        )

        params = res.x
        lam = params[:m]
        beta = params[m:]
        logL = -res.fun
        p = m + K_
        AIC  = -2*logL + 2*p
        AICc = AIC + (2*p*(p+1))/(N_-p-1) if (N_-p-1) > 0 else np.nan

        A, _ = build_A_and_logdet(lam, W_list, combo=combo)
        v = A.dot(y - X.dot(beta))

        # Approximate standard errors from the Hessian inverse, if available
        if hasattr(res, 'hess_inv'):
            try:
                Hin = res.hess_inv.todense()
            except Exception:
                Hin = np.asarray(res.hess_inv)
            se = np.sqrt(np.diag(Hin)) if Hin.shape == (p, p) else np.full(p, np.nan)
        else:
            se = np.full(p, np.nan)

        return {
            'model': f'SEM_{combo}_{len(W_list)}W',
            'lam': lam,
            'beta': beta,
            'logL': logL,
            'AICc': AICc,
            'resid': v,
            'combo': combo,
            'success': res.success,
            'se': se
        }


    def moran_I_val_from_array(resid, W_array, ids):
        """
        Compute Moran's I from residuals and a dense W matrix.
        """
        w_obj = util.full2W(W_array, ids=ids)
        mi = Moran(resid, w_obj, permutations=0)
        return mi.I, mi.p_norm


    def row_standardize(A):
        """
        Row-standardize a spatial weight matrix.
        """
        R = A.astype(float)
        rs = R.sum(axis=1, keepdims=True)
        rs[rs == 0] = 1.0
        return R / rs


    def ensure_connected_within_state(W_sub, XY_sub):
        """
        If the within-state WQ submatrix contains isolated nodes, connect each isolated node
        to its nearest neighbor once and then row-standardize.
        """
        R = W_sub.copy()
        deg = R.sum(axis=1)

        if np.any(deg == 0):
            D = np.linalg.norm(
                XY_sub[:, None, :] - XY_sub[None, :, :], axis=2
            )
            np.fill_diagonal(D, np.inf)
            lonely = np.where(deg == 0)[0]
            for i in lonely:
                j = int(D[i].argmin())
                if np.isfinite(D[i, j]):
                    R[i, j] = 1.0
                    R[j, i] = 1.0
            R = row_standardize(R)

        return R


    def submatrix(M, idx):
        """
        Extract the square submatrix M[idx, idx].
        """
        return M[np.ix_(idx, idx)]


    # 3.2 Candidate weight combinations: WP / WS / WQ singles plus WQ-anchored pairs only -----------------------

    W_dict = {'WP': WP, 'WS': WS, 'WQ': WQ}

    def make_tasks_add_single_pair(W_dict_local):
        """
        Single-channel candidates: WP, WS, WQ.
        Pair candidates: retain WQ+WP and WQ+WS only; exclude WP+WS.
        """
        keys = list(W_dict_local.keys())
        tasks = []
        # Single-channel models
        for k in keys:
            tasks.append((f'SEM_add_1_{k}', [W_dict_local[k]], 'add'))

        # Two-channel models: allow only WQ+WP / WQ+WS
        if ('WP' in keys) and ('WQ' in keys):
            tasks.append(('SEM_add_2_WP_WQ',
                          [W_dict_local['WP'], W_dict_local['WQ']],
                          'add'))
        if ('WS' in keys) and ('WQ' in keys):
            tasks.append(('SEM_add_2_WS_WQ',
                          [W_dict_local['WS'], W_dict_local['WQ']],
                          'add'))
        return tasks

    TASKS_MAIN_ADD = make_tasks_add_single_pair(W_dict)

    # 3.3 Stage-1: In-sample ranking for 2019 US models --------------------------------------

    K = X.shape[1]
    base_rows = []

    # OLS baseline; Moran's I is also computed for reference
    ols = fit_ols(y, X)
    I0, p0 = moran_I_val_from_array(ols['resid'], WQ, ids=IDs)
    base_rows.append(['OLS', '-', 'OLS',
                      ols['logL'], ols['AICc'], I0, p0])

    # All single- and two-channel SEMs
    for name, Wset, combo in TASKS_MAIN_ADD:
        res = fit_sem_multi(y, X, Wset, combo=combo)
        I, pI = moran_I_val_from_array(res['resid'], WQ, ids=IDs)
        wmix = ';'.join(name.split('_')[3:])
        base_rows.append([name, wmix, combo,
                          res['logL'], res['AICc'], I, pI])

    base_df = pd.DataFrame(
        base_rows,
        columns=['Model', 'W_mix', 'Combo', 'logL', 'AICc', 'MoranI', 'p_norm']
    )
    base_df = base_df.sort_values(['Combo', 'AICc'])

    print(f"\n=== Stage-1: In-sample ranking for {outcome_tag} 2019 (US) ===")
    print(base_df.head(10))

    # 3.4 Stage-2: Leave-one-state-out validation for shortlisted models ----------------------------

    stage1_sem = base_df[(base_df['Model'] != 'OLS') &
                         (base_df['Combo'] == 'add')].copy()

    single_pool = stage1_sem[stage1_sem['Model'].str.match(r'^SEM_add_1_')]
    best_single_name = single_pool.nsmallest(1, 'AICc')['Model'].iloc[0]

    TOP_K_PAIR = 5  # There are few pair candidates, so 5 is sufficient here
    pair_pool = stage1_sem[stage1_sem['Model'].str.match(r'^SEM_add_2_')]
    pair_shortlist = pair_pool.nsmallest(TOP_K_PAIR, 'AICc')['Model'].tolist()

    stage2_models = [best_single_name] + pair_shortlist

    print(f"\n[Stage-1] Shortlist for LOSO (best single + TOP_{TOP_K_PAIR} pairs):")
    print(stage2_models)


    def _loso_state_worker(st, IDs_all, df_all, y_all, X_all, tasks,
                           WQ_array, XY_all_, K_):
        """
        Run one leave-one-state-out fold.
        Fit the model on the training set and compute logL, AICc,
        and Moran's I on the holdout state.
        """
        hold_ids = df_all.loc[df_all['STATE'] == st, 'FIPS'].tolist()
        hold_set = set(hold_ids)

        hold_idx = np.array(
            [i for i, idd in enumerate(IDs_all) if idd in hold_set],
            dtype=int
        )
        train_idx = np.array(
            [i for i in range(len(IDs_all)) if i not in hold_set],
            dtype=int
        )

        out_rows = []

        # Skip states with too few counties
        if len(hold_idx) < 5 or len(train_idx) < (K_ + 1 + 2):
            return out_rows

        # OLS baseline; record logL only as a sanity check
        ols_tr = fit_ols(y_all[train_idx], X_all[train_idx, :])
        out_rows.append([st, 'OLS', '-', 'OLS',
                         ols_tr['logL'], np.nan, np.nan, np.nan])

        for name, Wset, combo in tasks:
            # Training set
            y_tr = y_all[train_idx]
            X_tr = X_all[train_idx, :]
            W_tr = [submatrix(Wk, train_idx) for Wk in Wset]
            res_tr = fit_sem_multi(y_tr, X_tr, W_tr, combo=combo)

            # holdout
            y_te = y_all[hold_idx]
            X_te = X_all[hold_idx, :]
            W_te = [submatrix(Wk, hold_idx) for Wk in Wset]
            N_te, K_te = X_te.shape
            m = len(W_te)

            A_te, logdetA_te = build_A_and_logdet(res_tr['lam'], W_te, combo=combo)
            if A_te is None:
                out_rows.append([
                    st, name, ';'.join(name.split('_')[3:]), combo,
                    -np.inf, np.inf, np.nan, np.nan
                ])
                continue

            e_te = y_te - X_te.dot(res_tr['beta'])
            v_te = A_te.dot(e_te)
            sse = float(v_te.T.dot(v_te))

            df_ = N_te - (K_te + m)
            sigma2 = sse/df_ if df_ > 0 else sse / max(N_te, 1)
            logL = (logdetA_te
                    - (N_te/2)*np.log(2*np.pi*sigma2)
                    - sse/(2*sigma2))
            AIC  = -2*logL + 2*(K_te + m)
            AICc = (AIC + (2*(K_te+m)*((K_te+m)+1)) /
                    (N_te-(K_te+m)-1) if (N_te-(K_te+m)-1) > 0 else np.nan)

            # Moran's I: repair within-holdout WQ connectivity before evaluation
            WQ_te = submatrix(WQ_array, hold_idx)
            XY_te = XY_all_[hold_idx, :]
            WQ_eval = ensure_connected_within_state(WQ_te, XY_te)
            mi = Moran(
                v_te,
                util.full2W(WQ_eval, ids=[IDs_all[i] for i in hold_idx]),
                permutations=0
            )

            out_rows.append([
                st, name, ';'.join(name.split('_')[3:]), combo,
                logL, AICc, mi.I, mi.p_norm
            ])

        return out_rows


    states = sorted(df['STATE'].unique())
    tasks_for_loso = [t for t in TASKS_MAIN_ADD if t[0] in stage2_models]

    res_lists = Parallel(
        n_jobs=N_JOBS, backend='loky', verbose=0
    )(
        delayed(_loso_state_worker)(
            st, IDs, df, y, X, tasks_for_loso, WQ, XY_all, K
        )
        for st in states
    )

    cv_loso = pd.DataFrame(
        [row for rows in res_lists for row in rows],
        columns=['STATE','Model','W_mix','Combo',
                 'pred_logL','pred_AICc','pred_MoranI','pred_p']
    )

    print(f"\n=== Stage-2: LOSO (shortlist, US, outcome={outcome_tag}) — preview ===")
    print(cv_loso.head(10))


    # 3.5 Summary, Pareto front, and forward gate with a super-lenient option -----------------------

    def summarize_models(cv_df, prefix='SEM_add_'):
        sub = cv_df[cv_df['Model'].str.startswith(prefix)].copy()
        g = (sub.groupby('Model', as_index=False)
               .agg(pred_AICc_mean=('pred_AICc','mean'),
                    pred_AICc_sd  =('pred_AICc','std'),
                    absMI_mean    =('pred_MoranI',
                                     lambda s: s.abs().mean()),
                    absMI_sd      =('pred_MoranI',
                                     lambda s: s.abs().std())))
        return g.dropna(subset=['pred_AICc_mean','absMI_mean'])


    def pareto_front(df_, x='pred_AICc_mean', y='absMI_mean'):
        rows = []
        for i, ri in df_.iterrows():
            dominated = False
            for j, rj in df_.iterrows():
                if j == i:
                    continue
                # Two objectives: lower AICc and lower absolute Moran's I are preferred
                if ((rj[x] <= ri[x] and rj[y] <= ri[y]) and
                    (rj[x] < ri[x] or rj[y] < ri[y])):
                    dominated = True
                    break
            if not dominated:
                rows.append(ri)
        return pd.DataFrame(rows)


    sum_tbl = summarize_models(cv_loso, prefix='SEM_add_')
    sum_tbl['dAICc_vs_best'] = (
        sum_tbl['pred_AICc_mean'] - sum_tbl['pred_AICc_mean'].min()
    )

    best_by_AICc_row  = sum_tbl.loc[sum_tbl['pred_AICc_mean'].idxmin()]
    best_by_AICc_name = best_by_AICc_row['Model']

    # ---- Gate settings: strict / lenient / super_lenient ----

    GATE_PROFILE        = 'super_lenient'   # Recommended setting for the US main analysis
    PREFER_PAIR_IF_TIED = True
    TIE_DELTA_REL       = 0.01             # Allow tie-preference when relative AICc difference is <= 1%

    if GATE_PROFILE == 'strict':
        # Emphasize strict cross-validation advantage
        GATE_DAICC_REL_MAX       = -0.01   # Require the pair model to improve AICc by at least 1%
        GATE_MI_REL_IMPROVE_MIN  = 0.00    # No strong improvement required; avoid worsening
        GATE_WINRATE_MIN         = 0.70    # Require lower AICc in at least 70% of states
    elif GATE_PROFILE == 'lenient':
        # Balance statistical performance and interpretability
        GATE_DAICC_REL_MAX       = 0.00    # Pair model mean AICc must not be worse than the single model
        GATE_MI_REL_IMPROVE_MIN  = 0.05    # Require at least 5% reduction in |Moran's I|
        GATE_WINRATE_MIN         = 0.50    # Require lower AICc in at least half of states
    elif GATE_PROFILE == 'super_lenient':
        # Structure-oriented option: prefer pair models unless clearly worse
        GATE_DAICC_REL_MAX       = 0.02    # Allow mean AICc to be up to 2% worse
        GATE_MI_REL_IMPROVE_MIN  = -0.10   # Allow |Moran's I| to worsen by at most 10%
        GATE_WINRATE_MIN         = 0.10    # Require lower AICc in at least 10% of states
    else:
        raise ValueError("Unknown GATE_PROFILE")

    best_single_row = sum_tbl[sum_tbl['Model'] == best_single_name].iloc[0]
    pair_tbl = sum_tbl[sum_tbl['Model'].str.match(r'^SEM_add_2_')]

    if len(pair_tbl) > 0:
        best_pair_row = pair_tbl.nsmallest(1, 'pred_AICc_mean').iloc[0]

        # Delta AICc: absolute and relative
        dAICc = (best_pair_row['pred_AICc_mean'] -
                 best_single_row['pred_AICc_mean'])
        AICc_single = best_single_row['pred_AICc_mean']
        dAICc_rel = dAICc / max(abs(AICc_single), 1e-9)

        # Relative Moran's I improvement; >0 means lower |MI| for the pair model
        absMI_single = best_single_row['absMI_mean']
        absMI_pair   = best_pair_row['absMI_mean']
        rel_mi_gain  = (
            (absMI_single - absMI_pair) /
            max(absMI_single, 1e-9)
        )

        # State-level win rate: proportion of states where the pair model has lower AICc
        A_ = cv_loso[cv_loso['Model'] == best_pair_row['Model']][
            ['STATE','pred_AICc']
        ].rename(columns={'pred_AICc':'pair'})
        B_ = cv_loso[cv_loso['Model'] == best_single_name][
            ['STATE','pred_AICc']
        ].rename(columns={'pred_AICc':'single'})
        J = A_.merge(B_, on='STATE', how='inner')
        win_rate = float((J['pair'] < J['single']).mean()) if len(J) > 0 else 0.0

        # Gate decision
        pass_gate = (
            (dAICc_rel <= GATE_DAICC_REL_MAX) and
            (rel_mi_gain >= GATE_MI_REL_IMPROVE_MIN) and
            (win_rate >= GATE_WINRATE_MIN)
        )

        if pass_gate:
            final_model_main = best_pair_row['Model']
            gate_note = (
                f"[{GATE_PROFILE}] Accepted 2-channel by gates: "
                f"ΔAICc={dAICc:.2f}, ΔAICc_rel={dAICc_rel:.3f}<= {GATE_DAICC_REL_MAX}, "
                f"rel|I|↓={rel_mi_gain:.3f}>= {GATE_MI_REL_IMPROVE_MIN}, "
                f"win_rate={win_rate:.2f}>= {GATE_WINRATE_MIN}."
            )
        else:
            # If gates fail but relative delta AICc is very small, allow tie-preference for the pair model
            if (GATE_PROFILE == 'super_lenient' and
                PREFER_PAIR_IF_TIED and
                (dAICc_rel <= TIE_DELTA_REL) and
                (rel_mi_gain >= GATE_MI_REL_IMPROVE_MIN) and
                (win_rate >= GATE_WINRATE_MIN/2)):
                final_model_main = best_pair_row['Model']
                gate_note = (
                    f"[{GATE_PROFILE}] Tie-preference to 2-channel "
                    f"(ΔAICc_rel={dAICc_rel:.3f}<= {TIE_DELTA_REL}); "
                    f"rel|I|↓={rel_mi_gain:.3f}, win_rate={win_rate:.2f}."
                )
            else:
                final_model_main = best_single_name
                gate_note = (
                    f"[{GATE_PROFILE}] Kept best single: pair failed gates "
                    f"and tie-preference not triggered. "
                    f"ΔAICc={dAICc:.2f}, ΔAICc_rel={dAICc_rel:.3f}, "
                    f"rel|I|↓={rel_mi_gain:.3f}, win_rate={win_rate:.2f}."
                )
    else:
        final_model_main = best_single_name
        dAICc = np.nan
        dAICc_rel = np.nan
        rel_mi_gain = np.nan
        win_rate = np.nan
        gate_note = f"[{GATE_PROFILE}] No eligible two-channel; kept best single."

    final_combo_main = 'add'
    pf = pareto_front(sum_tbl[['Model','pred_AICc_mean','absMI_mean']])

    print(f"\n[Final model for {outcome_tag} 2019, US] {final_model_main}")
    print("[Gate note] ", gate_note)
    print("[Best-by-AICc on shortlist] ", best_by_AICc_name)
    print("\n[Pareto front]")
    print(pf)


    # 3.6 Channel importance based on LOSO wAICc weights ----------------------------------

    def channel_importance_from_weights(aicc_series):
        """
        Compute wAICc weights from AICc values.
        """
        m_ = aicc_series.min()
        wi = np.exp(-0.5*(aicc_series - m_))
        return wi / wi.sum()


    def channel_importance_loso(cv_df, eligible_prefix='SEM_add_'):
        """
        Compute average channel importance from LOSO wAICc weights
        across all candidate SEM models.
        """
        sub = cv_df[cv_df['Model'].str.startswith(eligible_prefix)].copy()
        out_rows = []

        for st, g in sub.groupby('STATE'):
            g = g.dropna(subset=['pred_AICc'])
            if len(g) == 0:
                continue
            wi = channel_importance_from_weights(g['pred_AICc'])
            tmp = pd.DataFrame({
                'Model': g['Model'].values,
                'weight': wi.values
            })
            tmp['STATE'] = st
            out_rows.append(tmp)

        if len(out_rows) == 0:
            return pd.DataFrame(), pd.DataFrame()

        Wtbl = pd.concat(out_rows, ignore_index=True)

        ch_rows = []
        for key in W_dict.keys():
            mask = Wtbl['Model'].str.contains(f"_{key}")
            ch_rows.append([
                key,
                Wtbl.loc[mask, 'weight'].mean() if mask.any() else 0.0
            ])

        imp = (pd.DataFrame(ch_rows, columns=['Channel','Mean_wAICc'])
                 .sort_values('Mean_wAICc', ascending=False))
        return imp, Wtbl


    chan_imp, wtbl = channel_importance_loso(cv_loso)


    # 3.7 Save main-analysis outputs --------------------------------------------------------

    suffix = f"_{GATE_PROFILE}"

    base_df.to_csv(
        outpath(f"APPENDIX_in_sample_add_{outcome_tag}{suffix}.csv"),
        index=False
    )
    cv_loso.to_csv(
        outpath(f"MAIN_loso_add_shortlist_{outcome_tag}{suffix}.csv"),
        index=False
    )
    chan_imp.to_csv(
        outpath(f"MAIN_channel_importance_shortlist_{outcome_tag}{suffix}.csv"),
        index=False
    )

    pd.DataFrame({
        'Profile':[GATE_PROFILE],
        'Model':[final_model_main],
        'Combo':[final_combo_main],
        'Best_single':[best_single_name],
        'Best_by_AICc_on_shortlist':[best_by_AICc_name],
        'Decision':[gate_note],
        'dAICc_pair_vs_single':[dAICc],
        'dAICc_rel_pair_vs_single':[dAICc_rel],
        'Rel_MI_gain(pair_vs_single)':[rel_mi_gain],
        'Win_rate(pair_vs_single)':[win_rate],
        'GATE_DAICC_REL_MAX':[GATE_DAICC_REL_MAX],
        'GATE_WINRATE_MIN':[GATE_WINRATE_MIN],
        'GATE_MI_REL_IMPROVE_MIN':[GATE_MI_REL_IMPROVE_MIN],
        'PREFER_PAIR_IF_TIED':[PREFER_PAIR_IF_TIED],
        'TIE_DELTA_REL':[TIE_DELTA_REL],
        'TOP_K_PAIR':[TOP_K_PAIR],
        'Outcome_col':[y_col],
        'Outcome_tag':[outcome_tag]
    }).to_csv(
        outpath(f"MAIN_final_decision_forward_gate_{outcome_tag}{suffix}.csv"),
        index=False
    )

    pf.to_csv(
        outpath(f"MAIN_pareto_shortlist_{outcome_tag}{suffix}.csv"),
        index=False
    )

    print("\n[Main analysis] Saved:")
    print(f"  - APPENDIX_in_sample_add_{outcome_tag}{suffix}.csv")
    print(f"  - MAIN_loso_add_shortlist_{outcome_tag}{suffix}.csv")
    print(f"  - MAIN_channel_importance_shortlist_{outcome_tag}{suffix}.csv")
    print(f"  - MAIN_final_decision_forward_gate_{outcome_tag}{suffix}.csv")
    print(f"  - MAIN_pareto_shortlist_{outcome_tag}{suffix}.csv")

    # Hero model: keep the same logic as the DeepSouth workflow
    hero_model    = final_model_main
    hero_channels = hero_model.split("_")[3:]

    # ============================================
    # 4. Spillover & rank shift for hero model (US)
    #    Automatically supports single- or two-channel hero models
    # ============================================

    # 4.1 Parse W_list from the model name; automatically detect WP / WS / WQ
    def get_W_list_from_model_name(model_name, W_dict_local):
        """
        Examples:
          'SEM_add_1_WS'      -> ['WS']
          'SEM_add_2_WS_WQ'   -> ['WS','WQ']
        """
        parts = model_name.split('_')
        chans = [p for p in parts[3:] if p in W_dict_local]
        if len(chans) == 0:
            raise ValueError(f"Cannot parse channels from model name: {model_name}")
        return [W_dict_local[c] for c in chans], chans

    # The hero model has already been selected by the forward gate
    W_list_hero, hero_channels = get_W_list_from_model_name(hero_model, W_dict)
    print(f"[Hero] model = {hero_model}, channels = {hero_channels}")

    # 4.2 Fit the hero model and the WQ-only model using the full data
    res_hero = fit_sem_multi(y, X, W_list_hero, combo='add')

    # Baseline: WQ-only SEM; if WQ is unavailable, fall back to the OLS linear predictor
    has_WQ = ('WQ' in W_dict)

    if has_WQ:
        res_WQ = fit_sem_multi(y, X, [W_dict['WQ']], combo='add')
    else:
        res_WQ = None
        print("[Spillover] Warning: WQ not in W_dict; using OLS linear predictor as baseline.")


    def sem_fitted_mean(lam, beta, W_list, X_, combo='add'):
        """
        Compute SEM fitted values:
          (I - Σ λ_k W_k) y = Xβ  =>  ŷ = A^{-1} Xβ
        """
        A_, _ = build_A_and_logdet(lam, W_list, combo=combo)
        mu_lin = X_.dot(beta)
        y_hat = np.linalg.solve(A_, mu_lin)
        return y_hat

    # Hero fitted values; same handling for single- and two-channel models
    fit_hero = sem_fitted_mean(
        res_hero['lam'], res_hero['beta'], W_list_hero, X, combo='add'
    )

    # Baseline fitted values: use WQ-only SEM if available; otherwise use the OLS linear predictor
    if res_WQ is not None:
        fit_WQ = sem_fitted_mean(
            res_WQ['lam'], res_WQ['beta'], [W_dict['WQ']], X, combo='add'
        )
    else:
        fit_WQ = X @ ols['beta']

    # 4.3 Compute rank shifts: hero model vs WQ baseline
    g_rank = df[['FIPS','STATE']].copy()
    g_rank['fit_WQ']   = fit_WQ
    g_rank['fit_hero'] = fit_hero

    g_rank['rank_WQ']   = g_rank['fit_WQ'].rank(method='first')
    g_rank['rank_hero'] = g_rank['fit_hero'].rank(method='first')
    g_rank['delta_rank'] = g_rank['rank_hero'] - g_rank['rank_WQ']

    # 4.4 Construct spill_type: Q = WQ and B = non-WQ bridging channel(s) in the hero model
    N_all_model = len(IDs)

    # Q strength
    if 'WQ' in W_dict:
        q_strength = W_dict['WQ'].sum(axis=1)
    else:
        q_strength = np.zeros(N_all_model)

    # Bridging channels are hero_channels excluding WQ; possible values include [WP], [WS], [WP, WS], or empty
    bridging_channels = [ch for ch in hero_channels if ch != 'WQ']

    if len(bridging_channels) > 0:
        print(f"[Spillover] Bridging channels in hero: {bridging_channels}")
        # Sum all bridging-channel matrices to form a composite B matrix
        B_mat = None
        for ch in bridging_channels:
            if B_mat is None:
                B_mat = W_dict[ch].copy()
            else:
                B_mat = B_mat + W_dict[ch]
        b_strength = B_mat.sum(axis=1)

        # Use medians to define High / Low groups
        q_thr = np.median(q_strength)
        b_thr = np.median(b_strength)

        g_rank['Q_level'] = np.where(q_strength >= q_thr, 'HighQ', 'LowQ')
        g_rank['B_level'] = np.where(b_strength >= b_thr, 'HighB', 'LowB')
        g_rank['spill_type'] = g_rank['Q_level'] + '_' + g_rank['B_level']

        # Generalized bridging definition: LowQ & HighB regardless of whether B is WP, WS, or both
        g_rank['is_bridging'] = g_rank['spill_type'].eq('LowQ_HighB')

        # Record which bridging channel(s) are used
        g_rank['bridging_channels'] = '+'.join(bridging_channels)

    else:
        # Hero model uses WQ only, for example SEM_add_1_WQ
        print("[Spillover] Hero uses only WQ (no bridging channel); "
              "spillover classification degenerates to 'OnlyQ'.")
        if 'WQ' in W_dict:
            q_thr = np.median(q_strength)
            g_rank['Q_level'] = np.where(q_strength >= q_thr, 'HighQ', 'LowQ')
        else:
            g_rank['Q_level'] = 'NA'
        g_rank['B_level'] = 'NoB'
        g_rank['spill_type'] = 'OnlyQ'
        g_rank['is_bridging'] = False
        g_rank['bridging_channels'] = ''

    print("\n[Delta rank by spillover type] (US)")
    print(
        g_rank.groupby('spill_type')['delta_rank']
              .agg(['count','mean','std','min','median','max'])
    )

    if g_rank['is_bridging'].any():
        print("\n[Top bridging counties whose risk rank increases under hero model] (US)")
        print(
            g_rank[g_rank['is_bridging']]
                  .sort_values('delta_rank', ascending=False)
                  .head(15)
        )
    else:
        print("\n[Bridging] No counties flagged as 'LowQ_HighB' under current hero model.")

    # 4.5 Save rank-shift table
    # The shared workflow exports CSV tables only by default.
    csv_name = outpath(f"US_{outcome_tag}_SEM_WQ_vs_{hero_model}_ranks.csv")
    g_rank.to_csv(csv_name, index=False)
    print(f"[Rank shift] Saved CSV: {csv_name}")

    # Optional geospatial output for users who want to reproduce maps separately.
    if EXPORT_SPILLOVER_SHAPEFILE:
        g_out = df[['FIPS', 'STATE', 'geometry']].copy()
        g_out = g_out.merge(
            g_rank[['FIPS', 'fit_WQ', 'fit_hero', 'rank_WQ', 'rank_hero',
                    'delta_rank', 'spill_type', 'is_bridging', 'bridging_channels']],
            on='FIPS', how='left'
        )
        g_out = gpd.GeoDataFrame(g_out, geometry='geometry', crs=df_geo.crs)
        shp_name = outpath(f"US_spillover_{outcome_tag}_{hero_model}.shp")
        g_out.to_file(shp_name)
        print(f"[Spillover] Saved shapefile: {shp_name}")



    # ============================================
    # 5. Rewiring nulls for hero model (US, fast, bridging-aware)
    # ============================================

    RUN_REWIRING = True

    if RUN_REWIRING:
        print("\n[Rewiring nulls] Start ...")
        print(f"[Rewiring] Hero model = {hero_model}")
        print(f"[Rewiring] Hero channels = {hero_channels}")

        # 5.1 Select the target channel for rewiring:
        #     Prefer non-WQ bridging channels in the hero model; otherwise rewire WQ or the single channel
        bridging_channels = [ch for ch in hero_channels if ch != 'WQ']

        if len(bridging_channels) > 0:
            target_ch = bridging_channels[0]
            print(f"[Rewiring] Bridging channels in hero = {bridging_channels}")
            print(f"[Rewiring] Target channel = {target_ch} (bridging)")
        else:
            target_ch = hero_channels[0]
            print("[Rewiring] Hero has no bridging channel (only WQ or single).")
            print(f"[Rewiring] Target channel = {target_ch}")

        # Original hero-model AICc and |Moran's I|
        I_hero, _ = moran_I_val_from_array(res_hero['resid'], WQ, ids=IDs)
        AICc_hero = res_hero['AICc']
        print(f"[Rewiring] Original hero AICc = {AICc_hero:.3f}, |MoranI| = {abs(I_hero):.3f}")

        # 5.2 Fast SEM fitting for null models
        def fit_sem_multi_quick(y_, X_, W_list_, combo='add'):
            N_, K_ = X_.shape
            m_ = len(W_list_)
            beta0, *_ = np.linalg.lstsq(X_, y_, rcond=None)
            x0 = np.r_[np.zeros(m_), beta0]
            bounds = [(-0.99, 0.99)]*m_ + [(-np.inf, np.inf)]*K_

            res_ = minimize(
                neglog_sem, x0,
                args=(y_, X_, W_list_, combo),
                method='L-BFGS-B', bounds=bounds,
                options={'maxiter': 100, 'ftol': 1e-4, 'gtol': 1e-4}
            )
            params_ = res_.x
            lam_ = params_[:m_]
            beta_ = params_[m_:]
            logL_ = -res_.fun
            p_ = m_ + K_
            AIC_  = -2*logL_ + 2*p_
            AICc_ = AIC_ + (2*p_*(p_+1))/(N_-p_-1) if (N_-p_-1) > 0 else np.nan

            A_, _ = build_A_and_logdet(lam_, W_list_, combo=combo)
            v_ = A_.dot(y_ - X_.dot(beta_))
            I_, _ = moran_I_val_from_array(v_, WQ, ids=IDs)
            return AICc_, abs(I_)

        # 5.3 Rewiring: shuffle nonzero weights within each row while preserving row degree
        def rewire_W_rowwise(W_base, rng=None):
            """
            Degree-preserving rewiring for a row-standardized matrix:
              - Shuffle weights only among nonzero entries within each row.
              - Keep island rows as all-zero rows.
            """
            if rng is None:
                rng = np.random.default_rng(RANDOM_SEED + 101)

            M = np.array(W_base, dtype=float).copy()
            N_ = M.shape[0]

            for i_ in range(N_):
                row = M[i_, :]
                if row.sum() <= 0:
                    continue
                nz_idx = np.where(row > 0)[0]
                if len(nz_idx) <= 1:
                    continue

                row_vals = row[nz_idx]
                perm = rng.permutation(len(nz_idx))
                row[nz_idx] = row_vals[perm]
                M[i_, :] = row

            rs = M.sum(axis=1, keepdims=True)
            rs[rs == 0] = 1.0
            M = M / rs
            return M

        # 5.4 Generate rewired null networks and fit the same hero structure
        B_NULL = 50 if not RUN_PROD else 150
        rng_rew = np.random.default_rng(RANDOM_SEED + 401)

        null_records = []
        for b in range(B_NULL):
            if (b % 20) == 0:
                print(f"[Rewiring] iter {b}/{B_NULL}")

            # Apply row-wise rewiring to the target channel; keep other channels unchanged
            W_dict_null = dict(W_dict)
            W_dict_null[target_ch] = rewire_W_rowwise(W_dict[target_ch], rng=rng_rew)

            # Use the same hero_model channel combination, with one channel rewired
            W_list_null, _ = get_W_list_from_model_name(hero_model, W_dict_null)
            AICc_null, absI_null = fit_sem_multi_quick(y, X, W_list_null, combo='add')

            null_records.append({
                'iter': b,
                'AICc_null': AICc_null,
                'absI_null': absI_null
            })

        rew_df = pd.DataFrame(null_records)
        q_AICc = rew_df['AICc_null'].quantile([0.025, 0.5, 0.975])
        q_MI   = rew_df['absI_null'].quantile([0.025, 0.5, 0.975])

        print("\n[Rewiring nulls] Summary for hero structure (US):")
        print("  Null AICc (median, 2.5%, 97.5%):",
              q_AICc[0.5], q_AICc[0.025], q_AICc[0.975])
        print("  Null |Moran I| (median, 2.5%, 97.5%):",
              q_MI[0.5], q_MI[0.025], q_MI[0.975])

        safe_name = hero_model.replace(" ", "").replace(":", "")
        rew_df.to_csv(outpath(f"US_{outcome_tag}_rewiring_nulls_{safe_name}.csv"), index=False)
        print(f"\n[Rewiring nulls] Saved to US_{outcome_tag}_rewiring_nulls_{safe_name}.csv")


    # ============================================
    # 6. FAST Network perturbation (perturb WP/WS, keep Q fixed, US)
    #    Follow the DeepSouth logic: perturb only WP/WS and keep Q fixed
    # ============================================

    RUN_PERTURB_FAST = True

    if RUN_PERTURB_FAST:
        print("\n[FAST Network perturbation] Start (US) ...")

        # 6.1 Pre-build the WQ Moran object; WQ is not perturbed
        wQ_obj_global = util.full2W(WQ, ids=IDs)

        def moran_I_from_w(resid, w_obj=wQ_obj_global):
            mi = Moran(resid, w_obj, permutations=0)
            return mi.I, mi.p_norm

        # 6.2 Fast SEM fitting with reduced optimization precision
        def fit_sem_multi_quick(y_, X_, W_list_, combo='add'):
            N_, K_ = X_.shape
            m_ = len(W_list_)
            beta0, *_ = np.linalg.lstsq(X_, y_, rcond=None)
            x0 = np.r_[np.zeros(m_), beta0]
            bounds = [(-0.99, 0.99)]*m_ + [(-np.inf, np.inf)]*K_

            res_ = minimize(
                neglog_sem, x0,
                args=(y_, X_, W_list_, combo),
                method='L-BFGS-B', bounds=bounds,
                options={'maxiter': 100, 'ftol': 1e-4, 'gtol': 1e-4}
            )
            params_ = res_.x
            lam_ = params_[:m_]
            beta_ = params_[m_:]
            logL_ = -res_.fun
            p_ = m_ + K_
            AIC_  = -2*logL_ + 2*p_
            AICc_ = AIC_ + (2*p_*(p_+1))/(N_-p_-1) if (N_-p_-1) > 0 else np.nan
            A_, _ = build_A_and_logdet(lam_, W_list_, combo=combo)
            v_ = A_.dot(y_ - X_.dot(beta_))
            return {
                'logL': logL_, 'AICc': AICc_,
                'lam': lam_, 'beta': beta_, 'resid': v_,
                'success': res_.success
            }

        # 6.3 FAST Stage-1: compare only WQ-only / WP+WQ / WS+WQ
        def stage1_selection_fast(y_, X_, IDs_, WQ_array_, W_dict_local):
            rows = []
            ols_local = fit_ols(y_, X_)
            I0, p0 = moran_I_from_w(ols_local['resid'])
            rows.append(['OLS', '-', 'OLS',
                         ols_local['logL'], ols_local['AICc'], I0, p0])

            tasks = []
            # WQ-only
            tasks.append(('SEM_add_1_WQ', [W_dict_local['WQ']], 'add'))
            # Add WP+WQ / WS+WQ candidates if the corresponding channels exist
            if 'WP' in W_dict_local:
                tasks.append(('SEM_add_2_WP_WQ',
                              [W_dict_local['WP'], W_dict_local['WQ']], 'add'))
            if 'WS' in W_dict_local:
                tasks.append(('SEM_add_2_WS_WQ',
                              [W_dict_local['WS'], W_dict_local['WQ']], 'add'))

            for name, Wset, combo in tasks:
                res_ = fit_sem_multi_quick(y_, X_, Wset, combo=combo)
                I_, pI = moran_I_from_w(res_['resid'])
                wmix = ';'.join(name.split('_')[3:]) if name != 'OLS' else '-'
                rows.append([name, wmix, combo,
                             res_['logL'], res_['AICc'], I_, pI])

            df_stage1 = pd.DataFrame(
                rows,
                columns=['Model','W_mix','Combo','logL','AICc','MoranI','p_norm']
            )
            df_sem_only = df_stage1[df_stage1['Model'] != 'OLS']
            best_model = df_sem_only.nsmallest(1, 'AICc')['Model'].iloc[0]
            return best_model, df_stage1

        # 6.4 Random perturbation of WP / WS; Q remains fixed
        def perturb_W_matrix(W_base, drop_frac=0.15, noise_sd=0.10, rng=None):
            if rng is None:
                rng = np.random.default_rng(RANDOM_SEED + 200)
            M = np.array(W_base, dtype=float).copy()

            # Random edge dropping, symmetrically
            iu, ju = np.where(np.triu(M > 0, k=1))
            n_edges = len(iu)
            if n_edges > 0 and drop_frac > 0:
                n_drop = int(np.floor(drop_frac * n_edges))
                if n_drop > 0:
                    sel = rng.choice(n_edges, size=n_drop, replace=False)
                    di = iu[sel]; dj = ju[sel]
                    M[di, dj] = 0.0
                    M[dj, di] = 0.0

            # Multiplicative noise on edge weights
            nz_i, nz_j = np.where(M > 0)
            if len(nz_i) > 0 and noise_sd > 0:
                eps = rng.normal(loc=0.0, scale=noise_sd, size=len(nz_i))
                factor = 1.0 + eps
                factor = np.clip(factor, 0.05, 3.0)
                M[nz_i, nz_j] *= factor

            # Row-standardize
            rs = M.sum(axis=1, keepdims=True)
            rs[rs == 0] = 1.0
            M = M / rs
            return M

        # 6.5 Main perturbation loop
        B_PERT   = 10 if not RUN_PROD else 30
        DROP_FRAC = 0.15
        NOISE_SD  = 0.10

        rng_local = np.random.default_rng(RANDOM_SEED + 300)

        pert_results = []
        for b in range(B_PERT):
            if (b % 5) == 0:
                print(f"[FAST Network perturbation] iter {b}/{B_PERT}")

            WP_p = perturb_W_matrix(WP, drop_frac=DROP_FRAC,
                                    noise_sd=NOISE_SD, rng=rng_local)
            WS_p = perturb_W_matrix(WS, drop_frac=DROP_FRAC,
                                    noise_sd=NOISE_SD, rng=rng_local)

            W_dict_p = {'WP': WP_p, 'WS': WS_p, 'WQ': WQ}

            best_model_b, df_stage1_b = stage1_selection_fast(
                y, X, IDs, WQ, W_dict_p
            )
            channels_b = best_model_b.split('_')[3:]
            pert_results.append({
                'iter': b,
                'best_model': best_model_b,
                'channels': ','.join(channels_b)
            })

        pert_df = pd.DataFrame(pert_results)

        # Model frequency
        model_counts = pert_df['best_model'].value_counts().sort_values(ascending=False)
        model_freq_df = pd.DataFrame({
            'Model': model_counts.index,
            'Count': model_counts.values,
            'Freq': model_counts.values / B_PERT
        })

        # Channel inclusion frequency
        for ch in ['WP','WS','WQ']:
            pert_df[ch] = pert_df['channels'].str.contains(ch)

        ch_freq = (
            pert_df[['WP','WS','WQ']]
            .mean()
            .rename('Inclusion_rate')
            .sort_values(ascending=False)
            .reset_index()
            .rename(columns={'index':'Channel'})
        )

        print("\n[FAST Network perturbation] Best model frequencies (US):")
        print(model_freq_df)
        print("\n[FAST Network perturbation] Channel inclusion rates (US):")
        print(ch_freq)

        suffix_gate = f"_{GATE_PROFILE}" if 'GATE_PROFILE' in globals() else ""
        model_freq_df.to_csv(
            outpath(f"APPENDIX_fast_network_perturb_best_models_US_{outcome_tag}{suffix_gate}.csv"),
            index=False
        )
        ch_freq.to_csv(
            outpath(f"APPENDIX_fast_network_perturb_channel_freq_US_{outcome_tag}{suffix_gate}.csv"),
            index=False
        )

        print(f"\n[FAST Network perturbation] Saved:")
        print(f"  - APPENDIX_fast_network_perturb_best_models_US_{outcome_tag}{suffix_gate}.csv")
        print(f"  - APPENDIX_fast_network_perturb_channel_freq_US_{outcome_tag}{suffix_gate}.csv")


    # ============================================
    # 7. Temporal robustness: 2018 vs 2019 (US)
    #    Automatically reuse the hero structure for the current outcome
    # ============================================

    if outcome_2018_col is not None and outcome_2018_col in df_geo.columns:
        # 7.1 Parse the hero channel structure from the 2019 final_model_main
        W_list_hero_full, hero_channels = get_W_list_from_model_name(final_model_main, W_dict)
        print(f"[Temporal robust] Hero model (2019, US, {outcome_tag}) = {final_model_main}")
        print(f"[Temporal robust] Channels = {hero_channels}")

        # 7.2 Common sample with complete 2018 and 2019 outcomes plus covariates
        need_cols = ['FIPS', 'STATE', y_col, outcome_2018_col] + X_cols
        df_tmp = df_geo[need_cols].copy().dropna()
        N_all_geo  = df_geo.shape[0]
        N_use      = df_tmp.shape[0]
        print(f"[Temporal robust] Counties with complete 2018 & 2019 & covariates (US): "
              f"{N_use} / {N_all_geo}")

        id_to_pos = {fid: i for i, fid in enumerate(IDs)}
        idx = np.array([id_to_pos[f] for f in df_tmp['FIPS']], dtype=int)

        WP_2 = WP[np.ix_(idx, idx)]
        WS_2 = WS[np.ix_(idx, idx)]
        WQ_2 = WQ[np.ix_(idx, idx)]
        XY_2 = XY_all[idx, :]
        IDs_2 = [IDs[i] for i in idx]

        W_dict_2 = {
            'WP': WP_2,
            'WS': WS_2,
            'WQ': WQ_2
        }

        # Hero W_list on the common sample
        W_list_hero_2, _ = get_W_list_from_model_name(final_model_main, W_dict_2)

        X_2 = np.hstack([
            np.ones((df_tmp.shape[0], 1)),
            df_tmp[X_cols].values
        ])
        y2019 = df_tmp[y_col].values
        y2018 = df_tmp[outcome_2018_col].values

        def fit_three_models_for_year(y_, X_, label_year,
                                      WQ_array_, IDs_local,
                                      W_dict_local, W_list_hero_local):
            rows = []

            # 1) OLS
            ols_y = fit_ols(y_, X_)
            I_ols, _ = moran_I_val_from_array(ols_y['resid'], WQ_array_, ids=IDs_local)
            rows.append({
                'Year': label_year,
                'Model': 'OLS',
                'W_mix': '-',
                'logL': ols_y['logL'],
                'AICc': ols_y['AICc'],
                'MoranI': I_ols,
                'absMoranI': abs(I_ols)
            })

            # 2) WQ-only SEM
            res_wq = fit_sem_multi(y_, X_, [W_dict_local['WQ']], combo='add')
            I_wq, _ = moran_I_val_from_array(res_wq['resid'], WQ_array_, ids=IDs_local)
            rows.append({
                'Year': label_year,
                'Model': 'SEM_add_1_WQ',
                'W_mix': 'WQ',
                'logL': res_wq['logL'],
                'AICc': res_wq['AICc'],
                'MoranI': I_wq,
                'absMoranI': abs(I_wq)
            })

            # 3) Hero SEM using the 2019 channel structure
            res_hero_y = fit_sem_multi(y_, X_, W_list_hero_local, combo='add')
            I_hero_y, _ = moran_I_val_from_array(res_hero_y['resid'], WQ_array_, ids=IDs_local)
            rows.append({
                'Year': label_year,
                'Model': final_model_main,
                'W_mix': ';'.join(hero_channels),
                'logL': res_hero_y['logL'],
                'AICc': res_hero_y['AICc'],
                'MoranI': I_hero_y,
                'absMoranI': abs(I_hero_y)
            })

            return rows, res_wq, res_hero_y

        perf_rows_2019, res_wq_2019, res_hero_2019 = fit_three_models_for_year(
            y2019, X_2, '2019', WQ_2, IDs_2, W_dict_2, W_list_hero_2
        )
        perf_rows_2018, res_wq_2018, res_hero_2018 = fit_three_models_for_year(
            y2018, X_2, '2018', WQ_2, IDs_2, W_dict_2, W_list_hero_2
        )

        perf_df = pd.DataFrame(perf_rows_2019 + perf_rows_2018)
        perf_df = perf_df.sort_values(['Year','AICc']).reset_index(drop=True)

        print(f"\n[Temporal robust] Performance comparison on common counties (US, {outcome_tag}):")
        print(perf_df)

        # 7.4 Year-to-year comparison of hero lambda estimates; also export hero lambdas
        lam_rows = []
        for year_label, res_hero_y in [('2019', res_hero_2019), ('2018', res_hero_2018)]:
            for ch, lam_val in zip(hero_channels, res_hero_y['lam']):
                lam_rows.append({
                    'Outcome_tag': outcome_tag,
                    'Outcome_col': y_col,
                    'Year': year_label,
                    'Channel': ch,
                    'lambda': lam_val
                })

        lam_df = pd.DataFrame(lam_rows)
        lam_wide = lam_df.pivot_table(index='Channel', columns='Year', values='lambda')

        print(f"\n[Temporal robust] Hero lambda by channel (2019 vs 2018, US, {outcome_tag}):")
        print(lam_wide)

        # 7.5 Save results; filenames include outcome_tag
        perf_df.to_csv(
            outpath(f"TEMPORAL_US_{outcome_tag}_2018_vs_2019_model_perf.csv"),
            index=False
        )
        lam_df.to_csv(
            outpath(f"TEMPORAL_US_{outcome_tag}_2018_vs_2019_hero_lambda.csv"),
            index=False
        )

        print("\n[Temporal robust] Saved:")
        print(f"  - TEMPORAL_US_{outcome_tag}_2018_vs_2019_model_perf.csv")
        print(f"  - TEMPORAL_US_{outcome_tag}_2018_vs_2019_hero_lambda.csv")
    else:
        print(f"[Temporal robust] Skip: No valid 2018 column for outcome {y_col}.")


    # ============================================
    # 8. No visualization in the shared script
    # ============================================

    print("[INFO] No figures are generated by this shared workflow.")

    print("\n" + "=" * 80)
    print(f"[DONE] Finished workflow for outcome column: {y_col}")
    print("=" * 80)


# ============================================================
# Run workflow
# ============================================================

for _y_col in OUTCOMES_TO_RUN:
    run_one_outcome(_y_col)

###Table 2

In [ ]:
# ============================================================
# Table 2. Model metrics of OLS and candidate SEMs
# for HIV, Chlamydia, and Gonorrhea
#
# Purpose:
#   Generate manuscript Table 2 from saved Stage-1 in-sample
#   candidate model results produced by the three-outcome
#   main workflow.
#
# Required input files:
#   outputs/APPENDIX_in_sample_add_HIV_super_lenient.csv
#   outputs/APPENDIX_in_sample_add_Chlamydia_super_lenient.csv
#   outputs/APPENDIX_in_sample_add_Gonorrhea_super_lenient.csv
#
# Required columns in each input CSV:
#   Model, AICc, MoranI
#
# Output:
#   outputs/TABLE2_model_metrics_OLS_candidate_SEMs.csv
#   outputs/TABLE2_model_metrics_OLS_candidate_SEMs_full_check.csv
# ============================================================

import os
import pandas as pd


# -----------------------------
# 0. Global config
# -----------------------------

ROOT = os.getcwd()
OUTDIR = os.path.join(ROOT, "outputs")
os.makedirs(OUTDIR, exist_ok=True)

print("[INFO] Working directory:", ROOT)
print("[INFO] Output directory:", OUTDIR)


# -----------------------------
# 1. Input specifications
# -----------------------------

GATE_PROFILE = "super_lenient"

TABLE2_SPECS = [
    {
        "Pathogen": "HIV",
        "file_tag": "HIV",
    },
    {
        "Pathogen": "Chlamydia",
        "file_tag": "Chlamydia",
    },
    {
        "Pathogen": "Gonorrhea",
        "file_tag": "Gonorrhea",
    },
]

MODEL_ORDER = [
    "OLS",
    "SEM_add_1_WP",
    "SEM_add_1_WQ",
    "SEM_add_1_WS",
    "SEM_add_2_WP_WQ",
    "SEM_add_2_WS_WQ",
]

MODEL_LABELS = {
    "OLS": "OLS",
    "SEM_add_1_WP": "SEM_add_1_W_P",
    "SEM_add_1_WQ": "SEM_add_1_W_Q",
    "SEM_add_1_WS": "SEM_add_1_W_S",
    "SEM_add_2_WP_WQ": "SEM_add_2_W_P_W_Q",
    "SEM_add_2_WS_WQ": "SEM_add_2_W_S_W_Q",
}


# -----------------------------
# 2. Helper functions
# -----------------------------

def check_required_columns(df, required_cols, file_path):
    """
    Check whether all required columns are present.
    """
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise KeyError(
            "The following required columns are missing from:\n"
            f"{file_path}\n"
            + "\n".join(missing_cols)
        )


def load_in_sample_result(pathogen, file_tag, gate_profile):
    """
    Load one pathogen-specific Stage-1 in-sample result file.
    """
    file_path = os.path.join(
        OUTDIR,
        f"APPENDIX_in_sample_add_{file_tag}_{gate_profile}.csv",
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Input file for {pathogen} not found:\n{file_path}\n\n"
            "Please first run the three-outcome main workflow."
        )

    df = pd.read_csv(file_path)

    check_required_columns(
        df=df,
        required_cols=["Model", "AICc", "MoranI"],
        file_path=file_path,
    )

    df = df.copy()
    df["Pathogen"] = pathogen
    df["AICc"] = pd.to_numeric(df["AICc"], errors="coerce")
    df["MoranI"] = pd.to_numeric(df["MoranI"], errors="coerce")

    df = df.dropna(subset=["Model", "AICc", "MoranI"]).copy()

    return df


def format_model_label(model_name):
    """
    Convert internal model names to manuscript-style labels.
    """
    return MODEL_LABELS.get(model_name, model_name)


# -----------------------------
# 3. Load all pathogen-specific results
# -----------------------------

all_results = []

for spec in TABLE2_SPECS:
    pathogen = spec["Pathogen"]
    file_tag = spec["file_tag"]

    print(f"[INFO] Loading Stage-1 in-sample results for {pathogen}")

    df_one = load_in_sample_result(
        pathogen=pathogen,
        file_tag=file_tag,
        gate_profile=GATE_PROFILE,
    )

    all_results.append(df_one)

all_results_df = pd.concat(all_results, ignore_index=True)


# -----------------------------
# 4. Generate manuscript-ready Table 2
# -----------------------------

table2_rows = []

for list_id, model_name in enumerate(MODEL_ORDER, start=1):
    row = {
        "List": list_id,
        "Combination": format_model_label(model_name),
    }

    for spec in TABLE2_SPECS:
        pathogen = spec["Pathogen"]

        tmp = all_results_df[
            (all_results_df["Pathogen"] == pathogen)
            & (all_results_df["Model"] == model_name)
        ]

        if len(tmp) != 1:
            raise ValueError(
                f"Expected exactly one row for {pathogen} / {model_name}, "
                f"but found {len(tmp)}."
            )

        row[f"{pathogen}_AICc"] = round(float(tmp["AICc"].iloc[0]), 1)
        row[f"{pathogen}_MoranI_residual"] = round(float(tmp["MoranI"].iloc[0]), 3)

    table2_rows.append(row)

table2_df = pd.DataFrame(table2_rows)


# -----------------------------
# 5. Save Table 2
# -----------------------------

table2_out = os.path.join(
    OUTDIR,
    "TABLE2_model_metrics_OLS_candidate_SEMs.csv",
)

table2_df.to_csv(
    table2_out,
    index=False,
    encoding="utf-8-sig",
)

# Full checking table, preserving original rows from all three input files
full_check_out = os.path.join(
    OUTDIR,
    "TABLE2_model_metrics_OLS_candidate_SEMs_full_check.csv",
)

all_results_df.to_csv(
    full_check_out,
    index=False,
    encoding="utf-8-sig",
)

print("\n[INFO] Saved manuscript-ready Table 2 to:")
print("       ", table2_out)

print("[INFO] Saved full checking version to:")
print("       ", full_check_out)

print("\n[RESULT] Table 2. Model metrics of OLS and candidate SEMs")
print(table2_df)

###Table 3

In [ ]:
# ============================================================
# Table 3. Hero spatial error models and estimated spatial
# error parameters for each pathogen
#
# Purpose:
#   Generate manuscript Table 3 from saved outputs produced by
#   the three-outcome main workflow.
#
# Required input files:
#   outputs/APPENDIX_in_sample_add_HIV_super_lenient.csv
#   outputs/APPENDIX_in_sample_add_Chlamydia_super_lenient.csv
#   outputs/APPENDIX_in_sample_add_Gonorrhea_super_lenient.csv
#
#   outputs/TEMPORAL_US_HIV_2018_vs_2019_hero_lambda.csv
#   outputs/TEMPORAL_US_Chlamydia_2018_vs_2019_hero_lambda.csv
#   outputs/TEMPORAL_US_Gonorrhea_2018_vs_2019_hero_lambda.csv
#
# Output:
#   outputs/TABLE3_hero_spatial_error_models.csv
#   outputs/TABLE3_hero_spatial_error_models_full_check.csv
# ============================================================

import os
import pandas as pd


# -----------------------------
# 0. Global config
# -----------------------------

ROOT = os.getcwd()
OUTDIR = os.path.join(ROOT, "outputs")
os.makedirs(OUTDIR, exist_ok=True)

print("[INFO] Working directory:", ROOT)
print("[INFO] Output directory:", OUTDIR)

GATE_PROFILE = "super_lenient"


# -----------------------------
# 1. Table 3 specifications
# -----------------------------

TABLE3_SPECS = [
    {
        "Pathogen": "HIV",
        "file_tag": "HIV",
        "hero_model": "SEM_add_2_WP_WQ",
        "hero_display": "SEM_add_2_W_P_W_Q",
    },
    {
        "Pathogen": "Chlamydia",
        "file_tag": "Chlamydia",
        "hero_model": "SEM_add_1_WP",
        "hero_display": "SEM_add_1_W_P",
    },
    {
        "Pathogen": "Gonorrhea",
        "file_tag": "Gonorrhea",
        "hero_model": "SEM_add_1_WS",
        "hero_display": "SEM_add_1_W_S",
    },
]


# -----------------------------
# 2. Helper functions
# -----------------------------

def check_required_columns(df, required_cols, file_path):
    """
    Check whether all required columns are present.
    """
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise KeyError(
            "The following required columns are missing from:\n"
            f"{file_path}\n"
            + "\n".join(missing_cols)
        )


def load_in_sample_result(pathogen, file_tag, gate_profile):
    """
    Load one pathogen-specific Stage-1 in-sample result file.
    """
    file_path = os.path.join(
        OUTDIR,
        f"APPENDIX_in_sample_add_{file_tag}_{gate_profile}.csv",
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"In-sample model file for {pathogen} not found:\n{file_path}\n\n"
            "Please first run the three-outcome main workflow."
        )

    df = pd.read_csv(file_path)

    check_required_columns(
        df=df,
        required_cols=["Model", "AICc", "MoranI"],
        file_path=file_path,
    )

    df = df.copy()
    df["Pathogen"] = pathogen
    df["AICc"] = pd.to_numeric(df["AICc"], errors="coerce")
    df["MoranI"] = pd.to_numeric(df["MoranI"], errors="coerce")

    return df


def load_hero_lambda_result(pathogen, file_tag):
    """
    Load one pathogen-specific hero-lambda file and retain 2019 estimates.
    """
    file_path = os.path.join(
        OUTDIR,
        f"TEMPORAL_US_{file_tag}_2018_vs_2019_hero_lambda.csv",
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Hero lambda file for {pathogen} not found:\n{file_path}\n\n"
            "Please run the temporal robustness section of the main workflow, "
            "or save hero-model lambda estimates from the main full-sample fit."
        )

    df = pd.read_csv(file_path)

    check_required_columns(
        df=df,
        required_cols=["Year", "Channel", "lambda"],
        file_path=file_path,
    )

    df = df.copy()
    df["Year"] = df["Year"].astype(str)
    df["lambda"] = pd.to_numeric(df["lambda"], errors="coerce")

    df_2019 = df[df["Year"] == "2019"].copy()

    if len(df_2019) == 0:
        raise ValueError(
            f"No 2019 lambda estimates found in:\n{file_path}"
        )

    return df_2019


def get_model_row(df, pathogen, model_name):
    """
    Extract one model row for one pathogen.
    """
    tmp = df[
        (df["Pathogen"] == pathogen)
        & (df["Model"].astype(str) == model_name)
    ]

    if len(tmp) != 1:
        raise ValueError(
            f"Expected exactly one row for {pathogen} / {model_name}, "
            f"but found {len(tmp)}."
        )

    return tmp.iloc[0]


def format_lambda_string(lambda_df):
    """
    Format spatial error coefficients for Table 3.
    """
    channel_order = ["WP", "WS", "WQ"]

    channel_labels = {
        "WP": "lambda_W_P",
        "WS": "lambda_W_S",
        "WQ": "lambda_W_Q",
    }

    parts = []

    for ch in channel_order:
        tmp = lambda_df[lambda_df["Channel"] == ch]

        if len(tmp) == 0:
            continue

        lam_value = float(tmp["lambda"].iloc[0])
        parts.append(f"{channel_labels[ch]} = {lam_value:.3f}")

    if not parts:
        return "–"

    return "; ".join(parts)


# -----------------------------
# 3. Load all required outputs
# -----------------------------

all_in_sample = []
all_lambda = []

for spec in TABLE3_SPECS:
    pathogen = spec["Pathogen"]
    file_tag = spec["file_tag"]

    print(f"[INFO] Loading in-sample model metrics for {pathogen}")
    in_sample_df = load_in_sample_result(
        pathogen=pathogen,
        file_tag=file_tag,
        gate_profile=GATE_PROFILE,
    )
    all_in_sample.append(in_sample_df)

    print(f"[INFO] Loading 2019 hero lambda estimates for {pathogen}")
    lambda_df = load_hero_lambda_result(
        pathogen=pathogen,
        file_tag=file_tag,
    )
    lambda_df["Pathogen"] = pathogen
    all_lambda.append(lambda_df)

all_in_sample_df = pd.concat(all_in_sample, ignore_index=True)
all_lambda_df = pd.concat(all_lambda, ignore_index=True)


# -----------------------------
# 4. Generate Table 3
# -----------------------------

table3_rows = []

for spec in TABLE3_SPECS:
    pathogen = spec["Pathogen"]
    hero_model = spec["hero_model"]
    hero_display = spec["hero_display"]

    # Baseline OLS row
    ols_row = get_model_row(
        df=all_in_sample_df,
        pathogen=pathogen,
        model_name="OLS",
    )

    table3_rows.append(
        {
            "Pathogen": pathogen,
            "Model type": "Baseline",
            "Model name": "OLS",
            "AICc": round(float(ols_row["AICc"]), 2),
            "Moran’s I (residuals)": round(float(ols_row["MoranI"]), 3),
            "Spatial error coefficient(s) lambda": "–",
        }
    )

    # Hero SEM row
    hero_row = get_model_row(
        df=all_in_sample_df,
        pathogen=pathogen,
        model_name=hero_model,
    )

    lambda_2019 = all_lambda_df[
        all_lambda_df["Pathogen"] == pathogen
    ].copy()

    lambda_string = format_lambda_string(lambda_2019)

    table3_rows.append(
        {
            "Pathogen": pathogen,
            "Model type": "Hero SEM",
            "Model name": hero_display,
            "AICc": round(float(hero_row["AICc"]), 2),
            "Moran’s I (residuals)": round(float(hero_row["MoranI"]), 3),
            "Spatial error coefficient(s) lambda": lambda_string,
        }
    )

table3_df = pd.DataFrame(table3_rows)


# -----------------------------
# 5. Save Table 3
# -----------------------------

table3_out = os.path.join(
    OUTDIR,
    "TABLE3_hero_spatial_error_models.csv",
)

table3_df.to_csv(
    table3_out,
    index=False,
    encoding="utf-8-sig",
)

# Full checking file
full_check_out = os.path.join(
    OUTDIR,
    "TABLE3_hero_spatial_error_models_full_check.csv",
)

full_check_df = all_in_sample_df.merge(
    all_lambda_df,
    on="Pathogen",
    how="left",
    suffixes=("_model", "_lambda"),
)

full_check_df.to_csv(
    full_check_out,
    index=False,
    encoding="utf-8-sig",
)

print("\n[INFO] Saved manuscript-ready Table 3 to:")
print("       ", table3_out)

print("[INFO] Saved full checking version to:")
print("       ", full_check_out)

print("\n[RESULT] Table 3. Hero spatial error models")
print(table3_df)

###Table 4

In [ ]:
# ============================================================
# Table 4. Distribution of rank shifts by connectivity regime
#
# Goal:
#   Summarize county-level rank shifts from the adjacency-only
#   SEM (W_Q) to each pathogen-specific hero model.
#
# Required input:
#   Three rank-shift CSV files generated by the main workflow:
#     - US_HIV_SEM_WQ_vs_SEM_add_2_WP_WQ_ranks.csv
#     - US_Chlamydia_SEM_WQ_vs_SEM_add_1_WP_ranks.csv
#     - US_Gonorrhea_SEM_WQ_vs_SEM_add_1_WS_ranks.csv
#
# Key fields required in each rank-shift CSV:
#   - delta_rank
#   - spill_type
#
# Output:
#   TABLE4_rank_shift_by_connectivity_regime.csv
# ============================================================

import os
import numpy as np
import pandas as pd


# -----------------------------
# 0. Global config
# -----------------------------

ROOT = os.getcwd()
OUTDIR = os.path.join(ROOT, "outputs")
os.makedirs(OUTDIR, exist_ok=True)

print("[INFO] Working directory:", ROOT)
print("[INFO] Output directory:", OUTDIR)


# -----------------------------
# 1. Rank-shift CSV inputs
# -----------------------------
# These files should be produced by the three-outcome shared workflow.
# If your workflow saves them inside outputs/, keep OUTDIR here.
# If it saves them in ROOT, replace OUTDIR with ROOT in the paths below.

RANK_SHIFT_FILES = {
    "HIV": os.path.join(
        OUTDIR,
        "US_HIV_SEM_WQ_vs_SEM_add_2_WP_WQ_ranks.csv",
    ),
    "Chlamydia": os.path.join(
        OUTDIR,
        "US_Chlamydia_SEM_WQ_vs_SEM_add_1_WP_ranks.csv",
    ),
    "Gonorrhoea": os.path.join(
        OUTDIR,
        "US_Gonorrhea_SEM_WQ_vs_SEM_add_1_WS_ranks.csv",
    ),
}

# Display order in the manuscript table
PATHOGEN_ORDER = [
    "HIV",
    "Chlamydia",
    "Gonorrhoea",
]

REGIME_ORDER = [
    "HighQ_HighB",
    "HighQ_LowB",
    "LowQ_HighB",
    "LowQ_LowB",
]

REGIME_LABELS = {
    "HighQ_HighB": "HighQ–HighB",
    "HighQ_LowB": "HighQ–LowB",
    "LowQ_HighB": "LowQ–HighB",
    "LowQ_LowB": "LowQ–LowB",
}


# -----------------------------
# 2. Helper functions
# -----------------------------

def check_required_columns(df, required_cols, file_path):
    """
    Check whether all required columns exist in a rank-shift file.
    """
    missing_cols = [c for c in required_cols if c not in df.columns]

    if len(missing_cols) > 0:
        raise KeyError(
            "The following required columns are missing from:\n"
            f"{file_path}\n"
            + "\n".join(missing_cols)
        )


def load_rank_shift_file(pathogen, file_path):
    """
    Load one pathogen-specific rank-shift CSV.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Rank-shift file for {pathogen} not found:\n{file_path}\n\n"
            "Please first run the three-outcome main workflow to generate "
            "the pathogen-specific rank-shift CSV files."
        )

    df = pd.read_csv(file_path)

    check_required_columns(
        df=df,
        required_cols=["delta_rank", "spill_type"],
        file_path=file_path,
    )

    df = df.copy()
    df["Pathogen"] = pathogen
    df["delta_rank"] = pd.to_numeric(df["delta_rank"], errors="coerce")

    # Harmonize possible older labels if needed
    df["spill_type"] = df["spill_type"].replace(
        {
            "HighQ_HighP": "HighQ_HighB",
            "HighQ_LowP": "HighQ_LowB",
            "LowQ_HighP": "LowQ_HighB",
            "LowQ_LowP": "LowQ_LowB",
            "HighQ–HighB": "HighQ_HighB",
            "HighQ–LowB": "HighQ_LowB",
            "LowQ–HighB": "LowQ_HighB",
            "LowQ–LowB": "LowQ_LowB",
        }
    )

    df = df.dropna(subset=["delta_rank", "spill_type"]).copy()

    return df


def summarize_rank_shift_by_regime(df):
    """
    Summarize delta_rank by connectivity regime for one pathogen.
    """
    summary = (
        df.groupby("spill_type", dropna=False)["delta_rank"]
        .agg(
            n="count",
            mean_delta_rank="mean",
            median_delta_rank="median",
            min_delta_rank="min",
            max_delta_rank="max",
        )
        .reset_index()
    )

    return summary


# -----------------------------
# 3. Load and summarize all pathogens
# -----------------------------

table4_rows = []

for pathogen in PATHOGEN_ORDER:
    rank_file = RANK_SHIFT_FILES[pathogen]

    print(f"[INFO] Loading rank-shift file for {pathogen}:")
    print("       ", rank_file)

    rank_df = load_rank_shift_file(
        pathogen=pathogen,
        file_path=rank_file,
    )

    summary_df = summarize_rank_shift_by_regime(rank_df)

    summary_df.insert(0, "Pathogen", pathogen)
    summary_df = summary_df.rename(
        columns={
            "spill_type": "Connectivity regime",
        }
    )

    table4_rows.append(summary_df)

table4_df = pd.concat(table4_rows, ignore_index=True)


# -----------------------------
# 4. Format table for manuscript
# -----------------------------

# Enforce order
table4_df["Pathogen"] = pd.Categorical(
    table4_df["Pathogen"],
    categories=PATHOGEN_ORDER,
    ordered=True,
)

table4_df["Connectivity regime"] = pd.Categorical(
    table4_df["Connectivity regime"],
    categories=REGIME_ORDER,
    ordered=True,
)

table4_df = (
    table4_df
    .sort_values(["Pathogen", "Connectivity regime"])
    .reset_index(drop=True)
)

# Convert internal labels to manuscript display labels
table4_df["Connectivity regime"] = (
    table4_df["Connectivity regime"]
    .astype(str)
    .map(REGIME_LABELS)
)

# Round values to match manuscript table style
table4_df["Mean Δrank"] = table4_df["mean_delta_rank"].round(1)
table4_df["Median Δrank"] = table4_df["median_delta_rank"].round(1)

# Min and max are ranks, so keep as integers where possible
table4_df["Min Δrank"] = table4_df["min_delta_rank"].round(0).astype(int)
table4_df["Max Δrank"] = table4_df["max_delta_rank"].round(0).astype(int)

# Keep only final manuscript columns
table4_df = table4_df[
    [
        "Pathogen",
        "Connectivity regime",
        "n",
        "Mean Δrank",
        "Median Δrank",
        "Min Δrank",
        "Max Δrank",
    ]
].copy()


# -----------------------------
# 5. Save Table 4
# -----------------------------

table4_out = os.path.join(
    OUTDIR,
    "TABLE4_rank_shift_by_connectivity_regime.csv",
)

table4_df.to_csv(table4_out, index=False, encoding="utf-8-sig")

print("\n[INFO] Saved Table 4 to:")
print("       ", table4_out)

print("\n[RESULT] Table 4. Distribution of rank shifts by connectivity regime")
print(table4_df)

##Table S1

In [ ]:
# ============================================================
# Table S1. Hero vs degree-preserving rewired null models
#
# Goal:
#   Summarize the performance of pathogen-specific hero models
#   against degree-preserving rewired null models.
#
# Required input:
#   1. Rewiring null CSVs generated by the main workflow:
#        - US_HIV_rewiring_nulls_SEM_add_2_WP_WQ.csv
#        - US_Chlamydia_rewiring_nulls_SEM_add_1_WP.csv
#        - US_Gonorrhea_rewiring_nulls_SEM_add_1_WS.csv
#
#   2. Hero model performance values.
#      These can be supplied in either of two ways:
#        Option A: directly define HERO_RESULTS below.
#        Option B: read from saved model-summary CSVs if your shared
#                  workflow outputs them.
#
# Output:
#   TABLE_S1_hero_vs_rewired_null_models.csv
# ============================================================

import os
import numpy as np
import pandas as pd


# -----------------------------
# 0. Global config
# -----------------------------

ROOT = os.getcwd()
OUTDIR = os.path.join(ROOT, "outputs")
os.makedirs(OUTDIR, exist_ok=True)

print("[INFO] Working directory:", ROOT)
print("[INFO] Output directory:", OUTDIR)


# -----------------------------
# 1. Input files from rewiring null module
# -----------------------------
# These files should be produced by the pathogen-specific rewiring-null
# section of the shared main workflow.

REWIRING_NULL_FILES = {
    "HIV": os.path.join(
        OUTDIR,
        "US_HIV_rewiring_nulls_SEM_add_2_WP_WQ.csv",
    ),
    "Chlamydia": os.path.join(
        OUTDIR,
        "US_Chlamydia_rewiring_nulls_SEM_add_1_WP.csv",
    ),
    "Gonorrhoea": os.path.join(
        OUTDIR,
        "US_Gonorrhea_rewiring_nulls_SEM_add_1_WS.csv",
    ),
}


# -----------------------------
# 2. Hero model values
# -----------------------------
# Option A: direct values from the final manuscript workflow.
# If these values are already saved in your main workflow output,
# you can replace this block with a CSV-reading function.

HERO_RESULTS = {
    "HIV": {
        "AICc_hero": 32424.7,
        "abs_MoranI_hero": 0.028,
    },
    "Chlamydia": {
        "AICc_hero": 33135.3,
        "abs_MoranI_hero": 0.090,
    },
    "Gonorrhoea": {
        "AICc_hero": 29590.5,
        "abs_MoranI_hero": 0.001,
    },
}

PATHOGEN_ORDER = [
    "HIV",
    "Chlamydia",
    "Gonorrhoea",
]


# -----------------------------
# 3. Helper functions
# -----------------------------

def check_required_columns(df, required_cols, file_path):
    """
    Check whether all required columns are present.
    """
    missing_cols = [c for c in required_cols if c not in df.columns]

    if len(missing_cols) > 0:
        raise KeyError(
            "The following required columns are missing from:\n"
            f"{file_path}\n"
            + "\n".join(missing_cols)
        )


def load_rewiring_null_file(pathogen, file_path):
    """
    Load one pathogen-specific rewiring-null CSV.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Rewiring-null file for {pathogen} not found:\n{file_path}\n\n"
            "Please first run the rewiring-null section of the main workflow."
        )

    df = pd.read_csv(file_path)

    check_required_columns(
        df=df,
        required_cols=["AICc_null", "absI_null"],
        file_path=file_path,
    )

    df = df.copy()
    df["AICc_null"] = pd.to_numeric(df["AICc_null"], errors="coerce")
    df["absI_null"] = pd.to_numeric(df["absI_null"], errors="coerce")

    df = df.dropna(subset=["AICc_null", "absI_null"]).copy()

    if len(df) == 0:
        raise ValueError(
            f"No valid rewiring-null rows found for {pathogen}:\n{file_path}"
        )

    return df


def summarize_null_distribution(values):
    """
    Return median and 2.5th–97.5th percentile interval.
    """
    values = pd.Series(values).dropna()

    return {
        "median": values.quantile(0.500),
        "q025": values.quantile(0.025),
        "q975": values.quantile(0.975),
    }


def format_median_interval(x, digits=1):
    """
    Format median and percentile interval for manuscript display.
    """
    return (
        f"{x['median']:.{digits}f} "
        f"({x['q025']:.{digits}f}–{x['q975']:.{digits}f})"
    )


def format_moran_interval(x, digits=4):
    """
    Format Moran's I median and percentile interval.
    """
    return (
        f"{x['median']:.{digits}f} "
        f"({x['q025']:.{digits}f}–{x['q975']:.{digits}f})"
    )


# -----------------------------
# 4. Generate Table S1
# -----------------------------

table_s1_rows = []

for pathogen in PATHOGEN_ORDER:
    rewiring_file = REWIRING_NULL_FILES[pathogen]

    print(f"[INFO] Loading rewiring-null results for {pathogen}:")
    print("       ", rewiring_file)

    null_df = load_rewiring_null_file(
        pathogen=pathogen,
        file_path=rewiring_file,
    )

    aicc_null_summary = summarize_null_distribution(
        null_df["AICc_null"]
    )

    abs_moran_null_summary = summarize_null_distribution(
        null_df["absI_null"]
    )

    if pathogen not in HERO_RESULTS:
        raise KeyError(
            f"Hero results for {pathogen} are not defined in HERO_RESULTS."
        )

    hero_aicc = HERO_RESULTS[pathogen]["AICc_hero"]
    hero_abs_moran = HERO_RESULTS[pathogen]["abs_MoranI_hero"]

    table_s1_rows.append(
        {
            "Pathogen": pathogen,
            "AICc_hero": round(hero_aicc, 1),
            "AICc_null_median (2.5–97.5%)": format_median_interval(
                aicc_null_summary,
                digits=1,
            ),
            "|MoranI|_hero": round(hero_abs_moran, 3),
            "|MoranI|_null_median (2.5–97.5%)": format_moran_interval(
                abs_moran_null_summary,
                digits=4,
            ),
            # Machine-readable columns retained for checking
            "AICc_null_median": aicc_null_summary["median"],
            "AICc_null_q025": aicc_null_summary["q025"],
            "AICc_null_q975": aicc_null_summary["q975"],
            "absMoranI_null_median": abs_moran_null_summary["median"],
            "absMoranI_null_q025": abs_moran_null_summary["q025"],
            "absMoranI_null_q975": abs_moran_null_summary["q975"],
            "n_null": len(null_df),
        }
    )

table_s1_df = pd.DataFrame(table_s1_rows)


# -----------------------------
# 5. Save manuscript-ready and full checking tables
# -----------------------------

# Manuscript-ready table
table_s1_manuscript = table_s1_df[
    [
        "Pathogen",
        "AICc_hero",
        "AICc_null_median (2.5–97.5%)",
        "|MoranI|_hero",
        "|MoranI|_null_median (2.5–97.5%)",
    ]
].copy()

table_s1_out = os.path.join(
    OUTDIR,
    "TABLE_S1_hero_vs_rewired_null_models.csv",
)

table_s1_manuscript.to_csv(
    table_s1_out,
    index=False,
    encoding="utf-8-sig",
)

# Full diagnostic version with raw percentile columns
table_s1_full_out = os.path.join(
    OUTDIR,
    "TABLE_S1_hero_vs_rewired_null_models_full_check.csv",
)

table_s1_df.to_csv(
    table_s1_full_out,
    index=False,
    encoding="utf-8-sig",
)

print("\n[INFO] Saved manuscript-ready Table S1 to:")
print("       ", table_s1_out)

print("[INFO] Saved full checking version to:")
print("       ", table_s1_full_out)

print("\n[RESULT] Table S1. Hero vs degree-preserving rewired null models")
print(table_s1_manuscript)

##Table S2

In [ ]:
# ============================================================
# Table S2. FAST perturbations: preferred non-adjacent channel
# among adjacency-anchored two-channel SEMs
#
# Goal:
#   From raw data, run FAST network perturbations for three STI
#   outcomes and generate Supplementary Table S2.
#
# Candidate structures under each perturbation:
#   - SEM_add_1_WQ
#   - SEM_add_2_WP_WQ
#   - SEM_add_2_WS_WQ
#
# Perturbation design:
#   - Perturb WP and WS
#   - Keep WQ fixed
#   - Select the AICc-best SEM structure under each perturbation
#
# Outputs:
#   outputs/APPENDIX_fast_network_perturb_best_models_US_*.csv
#   outputs/APPENDIX_fast_network_perturb_channel_freq_US_*.csv
#   outputs/TABLE_S2_fast_perturbation_channel_preference.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

from numpy.linalg import slogdet
from scipy.optimize import minimize
from esda.moran import Moran
from libpysal.weights import util, Queen


# -----------------------------
# 0. Global config
# -----------------------------

ROOT = os.getcwd()
OUTDIR = os.path.join(ROOT, "outputs")
os.makedirs(OUTDIR, exist_ok=True)

print("[INFO] Working directory:", ROOT)
print("[INFO] Output directory:", OUTDIR)

RANDOM_SEED = 7
np.seterr(all="ignore")

# Input files
SHP_PATH = os.path.join(ROOT, "US_HIV_Merged_total_final.shp")
PCI_PATH = os.path.join(ROOT, "US_County_PCI_2019.csv")
SCI_PATH = os.path.join(ROOT, "processed_sci_summary.csv")

# Set RUN_PROD=True for final production results
RUN_PROD = False

B_PERT = 10 if not RUN_PROD else 30
DROP_FRAC = 0.15
NOISE_SD = 0.10

# Outcomes
OUTCOME_SPECS = [
    {
        "pathogen": "HIV",
        "file_tag": "HIV",
        "y_col": "PercenHIVp",
    },
    {
        "pathogen": "Chlamydia",
        "file_tag": "Chlamydia",
        "y_col": "ChlamydiaR",
    },
    {
        "pathogen": "Gonorrhoea",
        "file_tag": "Gonorrhea",
        "y_col": "GonorrheaR",
    },
]

# Covariates used in the main SEM workflow
X_COLS = [
    "Population",
    "UrbanRural",
    "Female",
    "Old",
    "Black",
    "Noinsuranc",
    "Poverty",
    "crime16to1",
    "Dissimilar",
]

# Shapefile truncated field-name repair
RENAME_MAP = {
    "RateChlamy": "RateChlamydia2018",
    "RateGonorr": "RateGonorrhea2018",
    "RateHIV201": "RateHIV2018",
}


# -----------------------------
# 1. Helper functions
# -----------------------------

def clean_fips_series(s):
    """
    Clean county FIPS codes as strings and remove leading zeros.
    """
    return s.astype(str).str.strip().str.lstrip("0")


def fips_to_state(fips):
    """
    Extract two-digit state FIPS from county FIPS.
    """
    return str(fips).zfill(5)[:2]


def check_required_columns(df, required_cols, source_name):
    """
    Check whether all required columns exist.
    """
    missing = [c for c in required_cols if c not in df.columns]

    if len(missing) > 0:
        raise KeyError(
            f"The following required columns are missing from {source_name}:\n"
            + "\n".join(missing)
        )


def row_standardize(mat):
    """
    Row-standardize a dense matrix.
    """
    mat = np.asarray(mat, dtype=float)

    row_sum = mat.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0

    mat = mat / row_sum
    mat = np.nan_to_num(mat, nan=0.0, posinf=0.0, neginf=0.0)

    return mat


def make_square_adjacency(df, row_col, col_col, value_col, valid_ids):
    """
    Build a square adjacency matrix from an edge table.
    """
    check_required_columns(
        df,
        [row_col, col_col, value_col],
        f"edge table for {value_col}",
    )

    df = df.copy()
    df[row_col] = clean_fips_series(df[row_col])
    df[col_col] = clean_fips_series(df[col_col])

    df = df[
        df[row_col].isin(valid_ids)
        & df[col_col].isin(valid_ids)
    ].copy()

    adj = df.pivot_table(
        index=row_col,
        columns=col_col,
        values=value_col,
        fill_value=0,
    )

    ids = sorted(set(valid_ids).intersection(adj.index).intersection(adj.columns))

    adj = adj.reindex(
        index=ids,
        columns=ids,
        fill_value=0,
    )

    np.fill_diagonal(adj.values, 0.0)

    return adj


def build_wp_matrix(df_geo, pci_path):
    """
    Build row-standardized PCI-based mobility matrix W_P.
    """
    if not os.path.exists(pci_path):
        raise FileNotFoundError(f"PCI file not found:\n{pci_path}")

    pci = pd.read_csv(pci_path)
    valid_ids = set(df_geo["FIPS"])

    adj = make_square_adjacency(
        df=pci,
        row_col="place_i",
        col_col="place_j",
        value_col="pci",
        valid_ids=valid_ids,
    )

    A = adj.values.astype(float)

    # Symmetrize before row standardization, following the main workflow
    A = A + A.T - np.diag(np.diag(A))
    np.fill_diagonal(A, 0.0)

    WP = row_standardize(A)

    return pd.DataFrame(WP, index=adj.index, columns=adj.columns)


def build_ws_matrix(df_geo, sci_path):
    """
    Build row-standardized SCI-based social-tie matrix W_S.
    """
    if not os.path.exists(sci_path):
        raise FileNotFoundError(f"SCI file not found:\n{sci_path}")

    sci = pd.read_csv(sci_path)
    valid_ids = set(df_geo["FIPS"])

    adj = make_square_adjacency(
        df=sci,
        row_col="user_loc",
        col_col="tfr_loc",
        value_col="tscaled_sci",
        valid_ids=valid_ids,
    )

    B = adj.values.astype(float)

    # Symmetrize before row standardization, following the main workflow
    B = B + B.T - np.diag(np.diag(B))
    np.fill_diagonal(B, 0.0)

    WS = row_standardize(B)

    return pd.DataFrame(WS, index=adj.index, columns=adj.columns)


def build_wq_matrix(df_geo):
    """
    Build row-standardized Queen contiguity matrix W_Q.
    """
    if df_geo.crs is None:
        raise ValueError("Input GeoDataFrame has no CRS.")

    g3857 = df_geo.to_crs(epsg=3857)

    wq = Queen.from_dataframe(
        g3857,
        ids=list(g3857["FIPS"]),
    )

    wq.transform = "R"
    WQ, ids = wq.full()

    return pd.DataFrame(WQ, index=ids, columns=ids)


def read_and_prepare_outcome_data(gdf_raw, y_col):
    """
    Prepare outcome-specific county sample.
    """
    required = [
        "FIPS",
        y_col,
        *X_COLS,
        "geometry",
    ]

    check_required_columns(gdf_raw, required, "US shapefile")

    df_geo = gdf_raw[required].copy()
    df_geo = df_geo.dropna(subset=[y_col] + X_COLS).copy()

    df_geo["FIPS"] = clean_fips_series(df_geo["FIPS"])
    df_geo["STATE"] = df_geo["FIPS"].apply(fips_to_state)

    df_geo = gpd.GeoDataFrame(
        df_geo,
        geometry="geometry",
        crs=gdf_raw.crs,
    )

    return df_geo


def align_data_and_matrices(df_geo, WP_df, WS_df, WQ_df):
    """
    Align county sample and three spatial matrices to a common FIPS order.
    """
    common_ids = sorted(
        set(df_geo["FIPS"])
        & set(WP_df.index)
        & set(WS_df.index)
        & set(WQ_df.index)
    )

    if len(common_ids) == 0:
        raise ValueError("No common counties across df_geo, WP, WS, and WQ.")

    df_model = (
        df_geo
        .set_index("FIPS")
        .loc[common_ids]
        .reset_index()
        .copy()
    )

    WP = WP_df.loc[common_ids, common_ids].values.astype(float)
    WS = WS_df.loc[common_ids, common_ids].values.astype(float)
    WQ = WQ_df.loc[common_ids, common_ids].values.astype(float)

    # Ensure row-standardized matrices after subsetting
    WP = row_standardize(WP)
    WS = row_standardize(WS)
    WQ = row_standardize(WQ)

    return df_model, common_ids, WP, WS, WQ


# -----------------------------
# 2. SEM functions
# -----------------------------

def fit_ols(y, X):
    """
    OLS baseline.
    """
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X.dot(beta)

    sse = float(e.T.dot(e))
    n, k = X.shape
    df = n - k

    sigma2 = sse / df
    logL = -(n / 2) * np.log(2 * np.pi * sigma2) - sse / (2 * sigma2)

    AIC = -2 * logL + 2 * k
    AICc = (
        AIC + (2 * k * (k + 1)) / (n - k - 1)
        if (n - k - 1) > 0
        else np.nan
    )

    return {
        "model": "OLS",
        "beta": beta,
        "logL": logL,
        "AICc": AICc,
        "resid": e,
    }


def build_A_and_logdet(lam, W_list, combo="add"):
    """
    Build SEM A matrix and log determinant.
    """
    n = W_list[0].shape[0]
    I = np.eye(n)

    if combo == "add":
        A = I.copy()
        for i, Wk in enumerate(W_list):
            A -= lam[i] * Wk

        sign, logdetA = slogdet(A)

        if sign <= 0:
            return None, None

        return A, logdetA

    raise ValueError("This FAST Table S2 script only supports combo='add'.")


def neglog_sem(params, y, X, W_list, combo="add"):
    """
    SEM negative log-likelihood.
    """
    n, k = X.shape
    m = len(W_list)

    lam = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(lam, W_list, combo=combo)

    if A is None:
        return 1e12

    e = y - X.dot(beta)
    v = A.dot(e)

    sse = float(v.T.dot(v))
    df = n - (k + m)

    if df <= 0:
        return 1e12

    sigma2 = sse / df

    if sigma2 <= 0:
        return 1e12

    logL = logdetA - (n / 2) * np.log(2 * np.pi * sigma2) - sse / (2 * sigma2)

    return -logL


def fit_sem_multi_quick(y, X, W_list, combo="add"):
    """
    Fast SEM fitting used for perturbation experiments.
    """
    n, k = X.shape
    m = len(W_list)

    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)

    x0 = np.r_[np.zeros(m), beta0]
    bounds = [(-0.99, 0.99)] * m + [(-np.inf, np.inf)] * k

    res = minimize(
        neglog_sem,
        x0,
        args=(y, X, W_list, combo),
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": 100,
            "ftol": 1e-4,
            "gtol": 1e-4,
        },
    )

    params = res.x
    lam = params[:m]
    beta = params[m:]

    logL = -res.fun
    p = m + k

    AIC = -2 * logL + 2 * p
    AICc = (
        AIC + (2 * p * (p + 1)) / (n - p - 1)
        if (n - p - 1) > 0
        else np.nan
    )

    A, _ = build_A_and_logdet(lam, W_list, combo=combo)

    if A is None:
        resid = np.full(n, np.nan)
    else:
        resid = A.dot(y - X.dot(beta))

    return {
        "lam": lam,
        "beta": beta,
        "logL": logL,
        "AICc": AICc,
        "resid": resid,
        "success": res.success,
    }


def moran_I_from_array(resid, W_array, ids):
    """
    Compute Moran's I from residuals and a dense W matrix.
    """
    w_obj = util.full2W(W_array, ids=ids)
    mi = Moran(resid, w_obj, permutations=0)

    return mi.I, mi.p_norm


def stage1_selection_fast(y, X, ids, WQ_array, W_dict):
    """
    FAST Stage-1 selection.

    Candidate models:
      - SEM_add_1_WQ
      - SEM_add_2_WP_WQ
      - SEM_add_2_WS_WQ
    """
    rows = []

    ols = fit_ols(y, X)
    I0, p0 = moran_I_from_array(ols["resid"], WQ_array, ids)

    rows.append(
        [
            "OLS",
            "-",
            "OLS",
            ols["logL"],
            ols["AICc"],
            I0,
            p0,
        ]
    )

    tasks = [
        ("SEM_add_1_WQ", [W_dict["WQ"]], "add"),
        ("SEM_add_2_WP_WQ", [W_dict["WP"], W_dict["WQ"]], "add"),
        ("SEM_add_2_WS_WQ", [W_dict["WS"], W_dict["WQ"]], "add"),
    ]

    for model_name, Wset, combo in tasks:
        res = fit_sem_multi_quick(y, X, Wset, combo=combo)
        I, pI = moran_I_from_array(res["resid"], WQ_array, ids)

        wmix = ";".join(model_name.split("_")[3:])

        rows.append(
            [
                model_name,
                wmix,
                combo,
                res["logL"],
                res["AICc"],
                I,
                pI,
            ]
        )

    df_stage1 = pd.DataFrame(
        rows,
        columns=[
            "Model",
            "W_mix",
            "Combo",
            "logL",
            "AICc",
            "MoranI",
            "p_norm",
        ],
    )

    sem_only = df_stage1[df_stage1["Model"] != "OLS"].copy()
    best_model = sem_only.nsmallest(1, "AICc")["Model"].iloc[0]

    return best_model, df_stage1


# -----------------------------
# 3. FAST perturbation functions
# -----------------------------

def perturb_W_matrix(W_base, drop_frac=0.15, noise_sd=0.10, rng=None):
    """
    Perturb a row-standardized network matrix.

    Steps:
      1. Randomly drop a fraction of existing undirected edges.
      2. Add multiplicative noise to remaining positive weights.
      3. Row-standardize the resulting matrix.
    """
    if rng is None:
        rng = np.random.default_rng(RANDOM_SEED + 200)

    M = np.array(W_base, dtype=float).copy()

    # Random edge dropping, using the upper triangle
    iu, ju = np.where(np.triu(M > 0, k=1))
    n_edges = len(iu)

    if n_edges > 0 and drop_frac > 0:
        n_drop = int(np.floor(drop_frac * n_edges))

        if n_drop > 0:
            selected = rng.choice(n_edges, size=n_drop, replace=False)
            di = iu[selected]
            dj = ju[selected]

            M[di, dj] = 0.0
            M[dj, di] = 0.0

    # Multiplicative noise on remaining positive weights
    nz_i, nz_j = np.where(M > 0)

    if len(nz_i) > 0 and noise_sd > 0:
        eps = rng.normal(loc=0.0, scale=noise_sd, size=len(nz_i))
        factor = 1.0 + eps
        factor = np.clip(factor, 0.05, 3.0)

        M[nz_i, nz_j] *= factor

    M = row_standardize(M)

    return M


def channels_from_model_name(model_name):
    """
    Parse channels from model name.
    """
    return model_name.split("_")[3:]


def run_fast_perturbation_for_one_outcome(
    pathogen,
    file_tag,
    y_col,
    gdf_raw,
):
    """
    Run FAST perturbation for one outcome and save intermediate outputs.
    """
    print("\n" + "=" * 72)
    print(f"[FAST Table S2] Start outcome: {pathogen} ({y_col})")
    print("=" * 72)

    # 1. Prepare outcome-specific data
    df_geo = read_and_prepare_outcome_data(gdf_raw, y_col)

    # 2. Build three matrices
    WP_df = build_wp_matrix(df_geo, PCI_PATH)
    WS_df = build_ws_matrix(df_geo, SCI_PATH)
    WQ_df = build_wq_matrix(df_geo)

    # 3. Align data and matrices
    df_model, ids, WP, WS, WQ = align_data_and_matrices(
        df_geo=df_geo,
        WP_df=WP_df,
        WS_df=WS_df,
        WQ_df=WQ_df,
    )

    # 4. Build y and X
    df_model = df_model.dropna(subset=[y_col] + X_COLS).copy()

    # Keep the same order as ids after dropna
    remaining_ids = list(df_model["FIPS"])
    id_pos = {fid: i for i, fid in enumerate(ids)}
    keep_idx = np.array([id_pos[fid] for fid in remaining_ids], dtype=int)

    WP = WP[np.ix_(keep_idx, keep_idx)]
    WS = WS[np.ix_(keep_idx, keep_idx)]
    WQ = WQ[np.ix_(keep_idx, keep_idx)]

    WP = row_standardize(WP)
    WS = row_standardize(WS)
    WQ = row_standardize(WQ)

    ids = remaining_ids

    y = df_model[y_col].to_numpy(dtype=float)
    X_raw = df_model[X_COLS].to_numpy(dtype=float)
    X = np.hstack([np.ones((len(df_model), 1)), X_raw])

    print(f"[INFO] {pathogen}: n counties = {len(df_model)}")

    rng_local = np.random.default_rng(RANDOM_SEED + 300)

    pert_records = []
    all_stage1_records = []

    for b in range(B_PERT):
        if (b % 5) == 0:
            print(f"[FAST perturbation] {pathogen}: iter {b}/{B_PERT}")

        WP_p = perturb_W_matrix(
            WP,
            drop_frac=DROP_FRAC,
            noise_sd=NOISE_SD,
            rng=rng_local,
        )

        WS_p = perturb_W_matrix(
            WS,
            drop_frac=DROP_FRAC,
            noise_sd=NOISE_SD,
            rng=rng_local,
        )

        W_dict_p = {
            "WP": WP_p,
            "WS": WS_p,
            "WQ": WQ,
        }

        best_model_b, stage1_b = stage1_selection_fast(
            y=y,
            X=X,
            ids=ids,
            WQ_array=WQ,
            W_dict=W_dict_p,
        )

        channels_b = channels_from_model_name(best_model_b)

        pert_records.append(
            {
                "Pathogen": pathogen,
                "iter": b,
                "best_model": best_model_b,
                "channels": ",".join(channels_b),
            }
        )

        stage1_b = stage1_b.copy()
        stage1_b.insert(0, "Pathogen", pathogen)
        stage1_b.insert(1, "iter", b)
        all_stage1_records.append(stage1_b)

    pert_df = pd.DataFrame(pert_records)
    stage1_all_df = pd.concat(all_stage1_records, ignore_index=True)

    # Model frequency
    model_counts = (
        pert_df["best_model"]
        .value_counts()
        .sort_values(ascending=False)
    )

    model_freq_df = pd.DataFrame(
        {
            "Pathogen": pathogen,
            "Model": model_counts.index,
            "Count": model_counts.values,
            "Freq": model_counts.values / B_PERT,
        }
    )

    # Channel inclusion rate
    for ch in ["WP", "WS", "WQ"]:
        pert_df[ch] = pert_df["channels"].str.contains(ch, regex=False)

    ch_freq = (
        pert_df[["WP", "WS", "WQ"]]
        .mean()
        .rename("Inclusion_rate")
        .reset_index()
        .rename(columns={"index": "Channel"})
    )

    ch_freq.insert(0, "Pathogen", pathogen)

    # Save intermediate outputs
    best_out = os.path.join(
        OUTDIR,
        f"APPENDIX_fast_network_perturb_best_models_US_{file_tag}.csv",
    )

    ch_out = os.path.join(
        OUTDIR,
        f"APPENDIX_fast_network_perturb_channel_freq_US_{file_tag}.csv",
    )

    iter_out = os.path.join(
        OUTDIR,
        f"APPENDIX_fast_network_perturb_iterations_US_{file_tag}.csv",
    )

    stage1_out = os.path.join(
        OUTDIR,
        f"APPENDIX_fast_network_perturb_stage1_all_US_{file_tag}.csv",
    )

    model_freq_df.to_csv(best_out, index=False, encoding="utf-8-sig")
    ch_freq.to_csv(ch_out, index=False, encoding="utf-8-sig")
    pert_df.to_csv(iter_out, index=False, encoding="utf-8-sig")
    stage1_all_df.to_csv(stage1_out, index=False, encoding="utf-8-sig")

    print(f"[INFO] Saved best-model frequencies: {best_out}")
    print(f"[INFO] Saved channel inclusion rates: {ch_out}")
    print(f"[INFO] Saved perturbation iterations: {iter_out}")
    print(f"[INFO] Saved all Stage-1 perturbation fits: {stage1_out}")

    return model_freq_df, ch_freq, pert_df


# -----------------------------
# 4. Table S2 formatting
# -----------------------------

def model_name_to_structure(model_name):
    """
    Convert internal model name to manuscript structure label.
    """
    mapping = {
        "SEM_add_1_WQ": "W_Q",
        "SEM_add_2_WP_WQ": "W_P+W_Q",
        "SEM_add_2_WS_WQ": "W_S+W_Q",
    }

    return mapping.get(model_name, model_name)


def summarize_best_structures(model_freq_df):
    """
    Create manuscript-style best-structure summary.
    """
    parts = []

    for _, row in model_freq_df.iterrows():
        structure = model_name_to_structure(row["Model"])
        count = int(row["Count"])
        parts.append(f"{structure} ({count}/{B_PERT})")

    return "; ".join(parts)


def get_channel_rate(ch_freq_df, channel):
    """
    Extract channel inclusion rate.
    """
    tmp = ch_freq_df[ch_freq_df["Channel"] == channel]

    if len(tmp) == 0:
        return 0.0

    return float(tmp["Inclusion_rate"].iloc[0])


def generate_table_s2(model_freq_all, ch_freq_all):
    """
    Generate manuscript-ready Table S2.
    """
    rows = []

    for spec in OUTCOME_SPECS:
        pathogen = spec["pathogen"]

        mf = model_freq_all[
            model_freq_all["Pathogen"] == pathogen
        ].copy()

        cf = ch_freq_all[
            ch_freq_all["Pathogen"] == pathogen
        ].copy()

        mf = mf.sort_values(["Count", "Model"], ascending=[False, True])

        rows.append(
            {
                "Pathogen": pathogen,
                "Best structure(s) selected under perturbations": summarize_best_structures(mf),
                "W_P inclusion rate": round(get_channel_rate(cf, "WP"), 2),
                "W_S inclusion rate": round(get_channel_rate(cf, "WS"), 2),
                "W_Q inclusion rate": round(get_channel_rate(cf, "WQ"), 2),
            }
        )

    table_s2 = pd.DataFrame(rows)

    return table_s2


# -----------------------------
# 5. Run all outcomes and save Table S2
# -----------------------------

if not os.path.exists(SHP_PATH):
    raise FileNotFoundError(f"Input shapefile not found:\n{SHP_PATH}")

gdf_raw = gpd.read_file(SHP_PATH)

if "FIPS" not in gdf_raw.columns:
    raise KeyError("Shapefile needs a 'FIPS' field.")

gdf_raw = gdf_raw.copy()
gdf_raw["FIPS"] = clean_fips_series(gdf_raw["FIPS"])
gdf_raw = gdf_raw.drop_duplicates(subset=["FIPS"]).copy()

# Repair truncated 2018 field names if present
for short_name, full_name in RENAME_MAP.items():
    if short_name in gdf_raw.columns and full_name not in gdf_raw.columns:
        gdf_raw = gdf_raw.rename(columns={short_name: full_name})

all_model_freq = []
all_ch_freq = []
all_pert = []

for spec in OUTCOME_SPECS:
    model_freq_df, ch_freq_df, pert_df = run_fast_perturbation_for_one_outcome(
        pathogen=spec["pathogen"],
        file_tag=spec["file_tag"],
        y_col=spec["y_col"],
        gdf_raw=gdf_raw,
    )

    all_model_freq.append(model_freq_df)
    all_ch_freq.append(ch_freq_df)
    all_pert.append(pert_df)

model_freq_all = pd.concat(all_model_freq, ignore_index=True)
ch_freq_all = pd.concat(all_ch_freq, ignore_index=True)
pert_all = pd.concat(all_pert, ignore_index=True)

# Save combined diagnostic outputs
model_freq_all_out = os.path.join(
    OUTDIR,
    "APPENDIX_fast_network_perturb_best_models_US_all_pathogens.csv",
)

ch_freq_all_out = os.path.join(
    OUTDIR,
    "APPENDIX_fast_network_perturb_channel_freq_US_all_pathogens.csv",
)

pert_all_out = os.path.join(
    OUTDIR,
    "APPENDIX_fast_network_perturb_iterations_US_all_pathogens.csv",
)

model_freq_all.to_csv(
    model_freq_all_out,
    index=False,
    encoding="utf-8-sig",
)

ch_freq_all.to_csv(
    ch_freq_all_out,
    index=False,
    encoding="utf-8-sig",
)

pert_all.to_csv(
    pert_all_out,
    index=False,
    encoding="utf-8-sig",
)

# Generate Table S2
table_s2 = generate_table_s2(
    model_freq_all=model_freq_all,
    ch_freq_all=ch_freq_all,
)

table_s2_out = os.path.join(
    OUTDIR,
    "TABLE_S2_fast_perturbation_channel_preference.csv",
)

table_s2.to_csv(
    table_s2_out,
    index=False,
    encoding="utf-8-sig",
)

print("\n[INFO] Saved combined best-model frequencies to:")
print("       ", model_freq_all_out)

print("[INFO] Saved combined channel inclusion rates to:")
print("       ", ch_freq_all_out)

print("[INFO] Saved combined perturbation iterations to:")
print("       ", pert_all_out)

print("[INFO] Saved Table S2 to:")
print("       ", table_s2_out)

print("\n[RESULT] Table S2. FAST perturbations")
print(table_s2)

##Table S3

In [ ]:
# ============================================================
# Table S3. Temporal robustness: 2018 vs 2019
#
# Goal:
#   Re-estimate OLS and pathogen-specific hero SEMs in 2018 and
#   2019 using the same temporal robustness sample.
#
# Hero structures fixed from the 2019 main workflow:
#   - HIV:        W_P + W_Q
#   - Chlamydia:  W_P only
#   - Gonorrhoea: W_S only
#
# Outputs:
#   outputs/TEMPORAL_US_HIV_2018_vs_2019_model_perf.csv
#   outputs/TEMPORAL_US_HIV_2018_vs_2019_hero_lambda.csv
#   outputs/TEMPORAL_US_Chlamydia_2018_vs_2019_model_perf.csv
#   outputs/TEMPORAL_US_Chlamydia_2018_vs_2019_hero_lambda.csv
#   outputs/TEMPORAL_US_Gonorrhea_2018_vs_2019_model_perf.csv
#   outputs/TEMPORAL_US_Gonorrhea_2018_vs_2019_hero_lambda.csv
#   outputs/TABLE_S3_temporal_robustness.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

from numpy.linalg import slogdet
from scipy.optimize import minimize
from esda.moran import Moran
from libpysal.weights import util, Queen


# -----------------------------
# 0. Global config
# -----------------------------

ROOT = os.getcwd()
OUTDIR = os.path.join(ROOT, "outputs")
os.makedirs(OUTDIR, exist_ok=True)

print("[INFO] Working directory:", ROOT)
print("[INFO] Output directory:", OUTDIR)

np.seterr(all="ignore")

SHP_PATH = os.path.join(ROOT, "US_HIV_Merged_total_final.shp")
PCI_PATH = os.path.join(ROOT, "US_County_PCI_2019.csv")
SCI_PATH = os.path.join(ROOT, "processed_sci_summary.csv")

X_COLS = [
    "Population",
    "UrbanRural",
    "Female",
    "Old",
    "Black",
    "Noinsuranc",
    "Poverty",
    "crime16to1",
    "Dissimilar",
]

RENAME_MAP = {
    "RateChlamy": "RateChlamydia2018",
    "RateGonorr": "RateGonorrhea2018",
    "RateHIV201": "RateHIV2018",
}

OUTCOME_SPECS = [
    {
        "pathogen": "HIV",
        "file_tag": "HIV",
        "y2019": "PercenHIVp",
        "y2018": "RateHIV2018",
        "hero_model": "SEM_add_2_WP_WQ",
        "hero_channels": ["WP", "WQ"],
        "structure_label": "W_P+W_Q",
    },
    {
        "pathogen": "Chlamydia",
        "file_tag": "Chlamydia",
        "y2019": "ChlamydiaR",
        "y2018": "RateChlamydia2018",
        "hero_model": "SEM_add_1_WP",
        "hero_channels": ["WP"],
        "structure_label": "W_P only",
    },
    {
        "pathogen": "Gonorrhoea",
        "file_tag": "Gonorrhea",
        "y2019": "GonorrheaR",
        "y2018": "RateGonorrhea2018",
        "hero_model": "SEM_add_1_WS",
        "hero_channels": ["WS"],
        "structure_label": "W_S only",
    },
]


# -----------------------------
# 1. Helper functions
# -----------------------------

def clean_fips_series(s):
    """
    Clean county FIPS codes as strings and remove leading zeros.
    """
    return s.astype(str).str.strip().str.lstrip("0")


def fips_to_state(fips):
    """
    Extract two-digit state FIPS code.
    """
    return str(fips).zfill(5)[:2]


def check_required_columns(df, required_cols, source_name):
    """
    Check whether all required columns exist.
    """
    missing = [c for c in required_cols if c not in df.columns]

    if len(missing) > 0:
        raise KeyError(
            f"The following required columns are missing from {source_name}:\n"
            + "\n".join(missing)
        )


def row_standardize(mat):
    """
    Row-standardize a dense matrix.
    """
    mat = np.asarray(mat, dtype=float)

    row_sum = mat.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0

    out = mat / row_sum
    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

    return out


def make_square_adjacency(df, row_col, col_col, value_col, valid_ids):
    """
    Build a square adjacency matrix from an edge table.
    """
    check_required_columns(
        df,
        [row_col, col_col, value_col],
        f"{value_col} edge table",
    )

    df = df.copy()
    df[row_col] = clean_fips_series(df[row_col])
    df[col_col] = clean_fips_series(df[col_col])

    valid_ids = sorted(set(valid_ids))

    df = df[
        df[row_col].isin(valid_ids)
        & df[col_col].isin(valid_ids)
    ].copy()

    adj = df.pivot_table(
        index=row_col,
        columns=col_col,
        values=value_col,
        fill_value=0,
    )

    adj = adj.reindex(
        index=valid_ids,
        columns=valid_ids,
        fill_value=0,
    )

    np.fill_diagonal(adj.values, 0.0)

    return adj


def build_wp_matrix(df_geo, pci_path):
    """
    Build row-standardized PCI-based mobility matrix W_P.
    """
    if not os.path.exists(pci_path):
        raise FileNotFoundError(f"PCI file not found:\n{pci_path}")

    pci = pd.read_csv(pci_path)
    valid_ids = set(df_geo["FIPS"])

    adj = make_square_adjacency(
        df=pci,
        row_col="place_i",
        col_col="place_j",
        value_col="pci",
        valid_ids=valid_ids,
    )

    A = adj.values.astype(float)

    # Symmetrize before row-standardization, following the main workflow
    A = A + A.T - np.diag(np.diag(A))
    np.fill_diagonal(A, 0.0)

    WP = row_standardize(A)

    return pd.DataFrame(WP, index=adj.index, columns=adj.columns)


def build_ws_matrix(df_geo, sci_path):
    """
    Build row-standardized SCI-based social-tie matrix W_S.
    """
    if not os.path.exists(sci_path):
        raise FileNotFoundError(f"SCI file not found:\n{sci_path}")

    sci = pd.read_csv(sci_path)
    valid_ids = set(df_geo["FIPS"])

    adj = make_square_adjacency(
        df=sci,
        row_col="user_loc",
        col_col="tfr_loc",
        value_col="tscaled_sci",
        valid_ids=valid_ids,
    )

    B = adj.values.astype(float)

    # Symmetrize before row-standardization, following the main workflow
    B = B + B.T - np.diag(np.diag(B))
    np.fill_diagonal(B, 0.0)

    WS = row_standardize(B)

    return pd.DataFrame(WS, index=adj.index, columns=adj.columns)


def build_wq_matrix(df_geo):
    """
    Build row-standardized Queen contiguity matrix W_Q.
    """
    if df_geo.crs is None:
        raise ValueError("Input GeoDataFrame has no CRS.")

    g3857 = df_geo.to_crs(epsg=3857)

    wq = Queen.from_dataframe(
        g3857,
        ids=list(g3857["FIPS"]),
    )

    wq.transform = "R"
    WQ, ids = wq.full()

    return pd.DataFrame(WQ, index=ids, columns=ids)


def structure_label_to_channels(hero_channels, W_dict):
    """
    Convert hero channel names to a list of W matrices.
    """
    return [W_dict[ch] for ch in hero_channels]


# -----------------------------
# 2. SEM functions
# -----------------------------

def fit_ols(y, X):
    """
    Fit OLS and return AICc and residuals.
    """
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X.dot(beta)

    sse = float(resid.T.dot(resid))
    n, k = X.shape
    df = n - k

    sigma2 = sse / df

    logL = (
        -(n / 2) * np.log(2 * np.pi * sigma2)
        - sse / (2 * sigma2)
    )

    AIC = -2 * logL + 2 * k

    AICc = (
        AIC + (2 * k * (k + 1)) / (n - k - 1)
        if (n - k - 1) > 0
        else np.nan
    )

    return {
        "model": "OLS",
        "beta": beta,
        "logL": logL,
        "AICc": AICc,
        "resid": resid,
    }


def build_A_and_logdet(lam, W_list):
    """
    Build A = I - sum(lambda_k W_k) and its log determinant.
    """
    n = W_list[0].shape[0]
    A = np.eye(n)

    for i, Wk in enumerate(W_list):
        A -= lam[i] * Wk

    sign, logdetA = slogdet(A)

    if sign <= 0:
        return None, None

    return A, logdetA


def neglog_sem(params, y, X, W_list):
    """
    Negative log-likelihood for additive SEM.
    """
    n, k = X.shape
    m = len(W_list)

    lam = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(lam, W_list)

    if A is None:
        return 1e12

    e = y - X.dot(beta)
    v = A.dot(e)

    sse = float(v.T.dot(v))

    df = n - (k + m)
    if df <= 0:
        return 1e12

    sigma2 = sse / df
    if sigma2 <= 0:
        return 1e12

    logL = (
        logdetA
        - (n / 2) * np.log(2 * np.pi * sigma2)
        - sse / (2 * sigma2)
    )

    return -logL


def fit_sem_multi(y, X, W_list):
    """
    Fit additive SEM with one or more spatial error channels.
    """
    n, k = X.shape
    m = len(W_list)

    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)

    x0 = np.r_[np.zeros(m), beta0]
    bounds = [(-0.99, 0.99)] * m + [(-np.inf, np.inf)] * k

    res = minimize(
        neglog_sem,
        x0,
        args=(y, X, W_list),
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": 200,
            "ftol": 1e-4,
            "gtol": 1e-4,
        },
    )

    params = res.x
    lam = params[:m]
    beta = params[m:]

    logL = -res.fun
    p = m + k

    AIC = -2 * logL + 2 * p

    AICc = (
        AIC + (2 * p * (p + 1)) / (n - p - 1)
        if (n - p - 1) > 0
        else np.nan
    )

    A, _ = build_A_and_logdet(lam, W_list)

    if A is None:
        resid = np.full(n, np.nan)
    else:
        resid = A.dot(y - X.dot(beta))

    return {
        "model": f"SEM_add_{m}W",
        "lam": lam,
        "beta": beta,
        "logL": logL,
        "AICc": AICc,
        "resid": resid,
        "success": res.success,
    }


def moran_I_from_array(resid, W_array, ids):
    """
    Compute Moran's I on residuals using W_Q.
    """
    w_obj = util.full2W(W_array, ids=ids)
    mi = Moran(resid, w_obj, permutations=0)

    return mi.I, mi.p_norm


# -----------------------------
# 3. Read data and build shared matrices
# -----------------------------

if not os.path.exists(SHP_PATH):
    raise FileNotFoundError(f"Input shapefile not found:\n{SHP_PATH}")

gdf = gpd.read_file(SHP_PATH)

if "FIPS" not in gdf.columns:
    raise KeyError("Shapefile needs a 'FIPS' field.")

gdf = gdf.copy()
gdf["FIPS"] = clean_fips_series(gdf["FIPS"])
gdf = gdf.drop_duplicates(subset=["FIPS"]).copy()

for short_name, full_name in RENAME_MAP.items():
    if short_name in gdf.columns and full_name not in gdf.columns:
        gdf = gdf.rename(columns={short_name: full_name})

required_outcome_cols = []
for spec in OUTCOME_SPECS:
    required_outcome_cols.extend([spec["y2018"], spec["y2019"]])

required_cols = [
    "FIPS",
    *sorted(set(required_outcome_cols)),
    *X_COLS,
    "geometry",
]

check_required_columns(gdf, required_cols, "US shapefile")

df_geo_all = gdf[required_cols].copy()
df_geo_all["STATE"] = df_geo_all["FIPS"].apply(fips_to_state)

df_geo_all = gpd.GeoDataFrame(
    df_geo_all,
    geometry="geometry",
    crs=gdf.crs,
)

# Build a common temporal robustness sample across all three pathogens
complete_cols = sorted(set(required_outcome_cols)) + X_COLS
df_geo_all = df_geo_all.dropna(subset=complete_cols).copy()

print("[INFO] Counties with complete 2018/2019 outcomes and covariates:")
print("       ", len(df_geo_all))

# Build matrices on the complete sample
WP_df = build_wp_matrix(df_geo_all, PCI_PATH)
WS_df = build_ws_matrix(df_geo_all, SCI_PATH)
WQ_df = build_wq_matrix(df_geo_all)

common_ids = sorted(
    set(df_geo_all["FIPS"])
    & set(WP_df.index)
    & set(WS_df.index)
    & set(WQ_df.index)
)

print("[INFO] Common counties after matrix intersection:")
print("       ", len(common_ids))

df_model_base = (
    df_geo_all
    .set_index("FIPS")
    .loc[common_ids]
    .reset_index()
    .copy()
)

WP = WP_df.loc[common_ids, common_ids].values.astype(float)
WS = WS_df.loc[common_ids, common_ids].values.astype(float)
WQ = WQ_df.loc[common_ids, common_ids].values.astype(float)

WP = row_standardize(WP)
WS = row_standardize(WS)
WQ = row_standardize(WQ)

IDS = common_ids

W_DICT = {
    "WP": WP,
    "WS": WS,
    "WQ": WQ,
}


# -----------------------------
# 4. Run temporal robustness models
# -----------------------------

all_perf_rows = []
all_lambda_rows = []

for spec in OUTCOME_SPECS:
    pathogen = spec["pathogen"]
    file_tag = spec["file_tag"]
    y2018_col = spec["y2018"]
    y2019_col = spec["y2019"]
    hero_channels = spec["hero_channels"]
    structure_label = spec["structure_label"]

    print("\n" + "=" * 72)
    print(f"[Temporal robustness] {pathogen}")
    print(f"[Temporal robustness] Hero structure = {structure_label}")
    print("=" * 72)

    W_list_hero = structure_label_to_channels(
        hero_channels=hero_channels,
        W_dict=W_DICT,
    )

    perf_rows = []
    lambda_rows = []

    for year, y_col in [(2018, y2018_col), (2019, y2019_col)]:
        df_year = df_model_base.copy()

        y = df_year[y_col].to_numpy(dtype=float)
        X_raw = df_year[X_COLS].to_numpy(dtype=float)
        X = np.hstack([np.ones((len(df_year), 1)), X_raw])

        # OLS
        ols = fit_ols(y, X)
        I_ols, p_ols = moran_I_from_array(
            resid=ols["resid"],
            W_array=WQ,
            ids=IDS,
        )

        perf_rows.append(
            {
                "Pathogen": pathogen,
                "Year": year,
                "Model type": "OLS",
                "Connectivity structure": "–",
                "AICc": ols["AICc"],
                "Moran’s I": I_ols,
                "p_norm": p_ols,
                "lambda_WP": np.nan,
                "lambda_WS": np.nan,
                "lambda_WQ": np.nan,
            }
        )

        # Hero SEM
        hero = fit_sem_multi(y, X, W_list_hero)

        I_hero, p_hero = moran_I_from_array(
            resid=hero["resid"],
            W_array=WQ,
            ids=IDS,
        )

        lambda_dict = {
            "WP": np.nan,
            "WS": np.nan,
            "WQ": np.nan,
        }

        for ch, lam_val in zip(hero_channels, hero["lam"]):
            lambda_dict[ch] = lam_val

            lambda_rows.append(
                {
                    "Pathogen": pathogen,
                    "Outcome_tag": file_tag,
                    "Year": year,
                    "Channel": ch,
                    "lambda": lam_val,
                }
            )

        perf_rows.append(
            {
                "Pathogen": pathogen,
                "Year": year,
                "Model type": "Hero SEM",
                "Connectivity structure": structure_label,
                "AICc": hero["AICc"],
                "Moran’s I": I_hero,
                "p_norm": p_hero,
                "lambda_WP": lambda_dict["WP"],
                "lambda_WS": lambda_dict["WS"],
                "lambda_WQ": lambda_dict["WQ"],
            }
        )

        print(
            f"[INFO] {pathogen} {year}: "
            f"OLS AICc={ols['AICc']:.1f}, I={I_ols:.3f}; "
            f"Hero AICc={hero['AICc']:.1f}, I={I_hero:.3f}"
        )

    perf_df_one = pd.DataFrame(perf_rows)
    lambda_df_one = pd.DataFrame(lambda_rows)

    perf_out = os.path.join(
        OUTDIR,
        f"TEMPORAL_US_{file_tag}_2018_vs_2019_model_perf.csv",
    )

    lambda_out = os.path.join(
        OUTDIR,
        f"TEMPORAL_US_{file_tag}_2018_vs_2019_hero_lambda.csv",
    )

    perf_df_one.to_csv(perf_out, index=False, encoding="utf-8-sig")
    lambda_df_one.to_csv(lambda_out, index=False, encoding="utf-8-sig")

    print("[INFO] Saved temporal performance table:")
    print("       ", perf_out)
    print("[INFO] Saved temporal lambda table:")
    print("       ", lambda_out)

    all_perf_rows.append(perf_df_one)
    all_lambda_rows.append(lambda_df_one)


perf_all = pd.concat(all_perf_rows, ignore_index=True)
lambda_all = pd.concat(all_lambda_rows, ignore_index=True)


# -----------------------------
# 5. Format manuscript Table S3
# -----------------------------

table_s3 = perf_all.copy()

table_s3["AICc"] = table_s3["AICc"].round(1)
table_s3["Moran’s I"] = table_s3["Moran’s I"].round(3)

table_s3["λW_P"] = table_s3["lambda_WP"].round(3)
table_s3["λW_S"] = table_s3["lambda_WS"].round(3)
table_s3["λW_Q"] = table_s3["lambda_WQ"].round(3)

table_s3 = table_s3[
    [
        "Pathogen",
        "Year",
        "Model type",
        "Connectivity structure",
        "AICc",
        "Moran’s I",
        "λW_P",
        "λW_S",
        "λW_Q",
    ]
].copy()

# Replace NaN lambda cells with manuscript dash
for col in ["λW_P", "λW_S", "λW_Q"]:
    table_s3[col] = table_s3[col].apply(
        lambda x: "–" if pd.isna(x) else f"{x:.3f}"
    )

# Keep AICc and Moran's I numeric formatting as strings for manuscript table
table_s3["AICc"] = table_s3["AICc"].apply(lambda x: f"{x:,.1f}")
table_s3["Moran’s I"] = table_s3["Moran’s I"].apply(lambda x: f"{x:.3f}")

pathogen_order = ["HIV", "Chlamydia", "Gonorrhoea"]
model_order = ["OLS", "Hero SEM"]

table_s3["Pathogen"] = pd.Categorical(
    table_s3["Pathogen"],
    categories=pathogen_order,
    ordered=True,
)

table_s3["Model type"] = pd.Categorical(
    table_s3["Model type"],
    categories=model_order,
    ordered=True,
)

table_s3 = (
    table_s3
    .sort_values(["Pathogen", "Year", "Model type"])
    .reset_index(drop=True)
)

table_s3_out = os.path.join(
    OUTDIR,
    "TABLE_S3_temporal_robustness.csv",
)

table_s3.to_csv(
    table_s3_out,
    index=False,
    encoding="utf-8-sig",
)

# Full diagnostic version
perf_all_out = os.path.join(
    OUTDIR,
    "TEMPORAL_US_all_pathogens_2018_vs_2019_model_perf.csv",
)

lambda_all_out = os.path.join(
    OUTDIR,
    "TEMPORAL_US_all_pathogens_2018_vs_2019_hero_lambda.csv",
)

perf_all.to_csv(
    perf_all_out,
    index=False,
    encoding="utf-8-sig",
)

lambda_all.to_csv(
    lambda_all_out,
    index=False,
    encoding="utf-8-sig",
)

print("\n[INFO] Saved combined temporal performance table:")
print("       ", perf_all_out)

print("[INFO] Saved combined temporal lambda table:")
print("       ", lambda_all_out)

print("[INFO] Saved Table S3:")
print("       ", table_s3_out)

print("\n[RESULT] Table S3. Temporal robustness")
print(table_s3)

##Table S4

In [ ]:
# ============================================
# 0. Imports & Global configs
# ============================================
import numpy as np
import pandas as pd
import geopandas as gpd

from numpy.linalg import slogdet
from scipy.optimize import minimize

from esda.moran import Moran
from libpysal.weights import util, W, Queen
from joblib import Parallel, delayed

np.seterr(all='ignore')

RANDOM_SEED = 7
rng_global = np.random.default_rng(RANDOM_SEED)

# Parallel configuration
N_JOBS = -1   # Use all available cores; set to 1 if parallel execution is unstable

# ------------- Outcome configuration: change one line to switch STI outcome ----------------
# Options: 'PercenHIVp', 'GonorrheaR', 'ChlamydiaR'
y_col = 'PercenHIVp'

OUTCOME_TAG = {
    'PercenHIVp': 'HIV',
    'GonorrheaR': 'Gonorrhea',
    'ChlamydiaR': 'Chlamydia'
}
outcome_tag = OUTCOME_TAG.get(y_col, y_col)

# Temporal-robustness field mapping; only outcome_tag is needed here, without running the 2018 models
TEMPORAL_2018_COL = {
    'PercenHIVp': 'RateHIV2018',
    'GonorrheaR': 'RateGonorrhea2018',
    'ChlamydiaR': 'RateChlamydia2018'
}
outcome_2018_col = TEMPORAL_2018_COL.get(y_col, None)

# ============================================
# 1. Read US shapefile & basic preprocessing
# ============================================

shp_path = "US_HIV_Merged_total_final.shp"
gdf = gpd.read_file(shp_path)

if 'FIPS' not in gdf.columns:
    raise KeyError("Shapefile needs a 'FIPS' field.")

# Clean FIPS codes by removing leading zeros and dropping duplicates
gdf['FIPS'] = gdf['FIPS'].astype(str).str.lstrip('0')
gdf = gdf.drop_duplicates(subset=['FIPS'])

# Repair potentially truncated 2018 field names from shapefile export
rename_map = {
    'RateChlamy': 'RateChlamydia2018',
    'RateGonorr': 'RateGonorrhea2018',
    'RateHIV201': 'RateHIV2018'
}
for short, long_name in rename_map.items():
    if (short in gdf.columns) and (long_name not in gdf.columns):
        gdf = gdf.rename(columns={short: long_name})

# Covariates
X_cols = [
    'Population','UrbanRural','Female','Old','Black',
    'Noinsuranc','Poverty','crime16to1','Dissimilar'
]

cols_keep = [
    'FIPS',
    'GonorrheaR','ChlamydiaR','PercenHIVp',
    *X_cols,
    'RateChlamydia2018','RateGonorrhea2018','RateHIV2018',
    'geometry'
]

df_geo = gdf[cols_keep].copy()

# Initial complete-case filtering for the current 2019 outcome and covariates
df_geo = df_geo.dropna(subset=[y_col] + X_cols)

# State code
def fips2state(x):
    return str(x).zfill(5)[:2]

df_geo['STATE'] = df_geo['FIPS'].apply(fips2state)
df_geo = gpd.GeoDataFrame(df_geo, geometry='geometry', crs=gdf.crs)

# ============================================
# 2. Build spatial weights & design matrix (US)
# ============================================

# 2.1 WP：Twitter PCI mobility
pci_path = "US_County_PCI_2019.csv"
pci = pd.read_csv(pci_path)

pci['place_i'] = pci['place_i'].astype(str).str.lstrip('0')
pci['place_j'] = pci['place_j'].astype(str).str.lstrip('0')

pci_f = pci[
    pci['place_i'].isin(df_geo['FIPS']) &
    pci['place_j'].isin(df_geo['FIPS'])
].copy()

A_pci = pci_f.pivot_table(
    index='place_i', columns='place_j',
    values='pci', fill_value=0
)
np.fill_diagonal(A_pci.values, 0.0)
A = A_pci.values.astype(float)
A = A + A.T - np.diag(np.diag(A))  # Symmetrize
row_sum = A.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WP = A / row_sum
wP = util.full2W(WP, ids=list(A_pci.index))

# 2.2 WS：Facebook SCI social ties
sci_path = "processed_sci_summary.csv"
sci = pd.read_csv(sci_path)

sci['user_loc'] = sci['user_loc'].astype(str).str.lstrip('0')
sci['tfr_loc']  = sci['tfr_loc'].astype(str).str.lstrip('0')

sci_f = sci[
    sci['user_loc'].isin(df_geo['FIPS']) &
    sci['tfr_loc'].isin(df_geo['FIPS'])
].copy()

A_sci = sci_f.pivot_table(
    index='user_loc', columns='tfr_loc',
    values='tscaled_sci', fill_value=0
)
np.fill_diagonal(A_sci.values, 0.0)
B = A_sci.values.astype(float)
B = B + B.T - np.diag(np.diag(B))
row_sum = B.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WS = B / row_sum
wS = util.full2W(WS, ids=list(A_sci.index))

# 2.3 WQ: Queen contiguity for the full US sample
g3857 = df_geo.to_crs(epsg=3857)
wQ = Queen.from_dataframe(g3857, ids=list(df_geo['FIPS']))
wQ.transform = 'R'
WQ, _ = wQ.full()

# 2.4 Align ID sets and ordering across WP, WS, WQ, and df_geo

common = (
    set(df_geo['FIPS']) &
    set(wP.id_order) &
    set(wS.id_order) &
    set(wQ.id_order)
)

def subset_W(w: W, keep_ids):
    keep_ids = list(dict.fromkeys([str(x) for x in keep_ids]))
    keep = set(keep_ids)
    new_neighbors, new_weights = {}, {}
    for i in w.id_order:
        if i not in keep:
            continue
        nbs = w.neighbors.get(i, [])
        wts = w.weights.get(i, [])
        kn, kw = [], []
        for nb, wt in zip(nbs, wts):
            if nb in keep:
                kn.append(nb); kw.append(wt)
        new_neighbors[i] = kn
        new_weights[i] = kw
    return W(new_neighbors, new_weights, ids=keep_ids)

df_geo = df_geo[df_geo['FIPS'].isin(common)].copy()
wP = subset_W(wP, common)
wS = subset_W(wS, common)
wQ = subset_W(wQ, common)

df_geo = df_geo.set_index('FIPS').loc[wQ.id_order].reset_index()
IDs   = list(wQ.id_order)
STATE = df_geo['STATE'].values

# Extract aligned dense matrix forms
WP, _ = wP.full()
WS, _ = wS.full()
WQ, _ = wQ.full()

# Aligned projected coordinates (EPSG:3857)
XY_all = np.vstack([
    g3857.set_index('FIPS').loc[IDs].geometry.centroid.x.values,
    g3857.set_index('FIPS').loc[IDs].geometry.centroid.y.values
]).T

# 2.5 Build outcome and covariate arrays for the current 2019 STI outcome
df = df_geo[['FIPS', y_col] + X_cols + ['STATE','geometry']].dropna().copy()

y = df[y_col].values
X_raw = df[X_cols].values
N = X_raw.shape[0]
X = np.hstack([np.ones((N, 1)), X_raw])   # Add intercept
K = X.shape[1]

# ============================================
# 3. SEM core helpers + Stage-1 & Stage-2 + Gate
# ============================================

# 3.1 Core helper functions --------------------------------------------------

def fit_ols(y, X):
    """
    Fit standard OLS as the baseline model.
    Returns beta, residuals, log-likelihood, and AICc.
    """
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X.dot(beta)
    sse = float(e.T.dot(e))
    N_, K_ = X.shape
    df_ = N_ - K_
    sigma2 = sse / df_
    logL = -(N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    AIC  = -2*logL + 2*K_
    AICc = (AIC + (2*K_*(K_+1)) / (N_-K_-1)
            if (N_-K_-1) > 0 else np.nan)
    return {
        'model': 'OLS',
        'beta': beta,
        'logL': logL,
        'AICc': AICc,
        'resid': e
    }


def build_A_and_logdet(lam, W_list, combo='add'):
    """
    Build the SEM A matrix: A = I - sum(lambda_k W_k) for additive SEM,
    or the product form for multiplicative SEM.
    """
    N_ = W_list[0].shape[0]
    I = np.eye(N_)

    if combo == 'add':
        A = I.copy()
        for i, Wk in enumerate(W_list):
            A -= lam[i] * Wk
        sign, logdetA = slogdet(A)
        if sign <= 0:
            return None, None
        return A, logdetA

    elif combo == 'prod':
        A = I.copy()
        logdet_sum = 0.0
        for i, Wk in enumerate(W_list):
            Mi = I - lam[i] * Wk
            sign_i, logdet_i = slogdet(Mi)
            if sign_i <= 0:
                return None, None
            logdet_sum += logdet_i
            A = Mi @ A
        return A, logdet_sum

    else:
        raise ValueError("combo must be 'add' or 'prod'")


def neglog_sem(params, y, X, W_list, combo='add'):
    """
    Negative log-likelihood for SEM optimization.
    params = [lambda_1, ..., lambda_m, beta_0, ..., beta_{K-1}]
    """
    N_, K_ = X.shape
    m = len(W_list)
    lam = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(lam, W_list, combo=combo)
    if A is None:
        return 1e12

    e = y - X.dot(beta)
    v = A.dot(e)
    sse = float(v.T.dot(v))

    df_ = N_ - (K_ + m)
    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_
    if sigma2 <= 0:
        return 1e12

    logL = logdetA - (N_/2)*np.log(2*np.pi*sigma2) - sse/(2*sigma2)
    return -logL


def fit_sem_multi(y, X, W_list, method='L-BFGS-B', combo='add'):
    """
    Fit the full multi-weight SEM used for Stage-1, LOSO, and add-vs-prod sensitivity analyses.
    """
    N_, K_ = X.shape
    m = len(W_list)

    # Initial values: OLS beta and lambda = 0
    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
    x0 = np.r_[np.zeros(m), beta0]
    bounds = [(-0.99, 0.99)]*m + [(-np.inf, np.inf)]*K_

    res = minimize(
        neglog_sem, x0,
        args=(y, X, W_list, combo),
        method=method,
        bounds=bounds,
        options={'maxiter': 200, 'ftol': 1e-4, 'gtol': 1e-4}
    )

    params = res.x
    lam = params[:m]
    beta = params[m:]
    logL = -res.fun
    p = m + K_
    AIC  = -2*logL + 2*p
    AICc = AIC + (2*p*(p+1))/(N_-p-1) if (N_-p-1) > 0 else np.nan

    A, _ = build_A_and_logdet(lam, W_list, combo=combo)
    v = A.dot(y - X.dot(beta))

    # Approximate standard errors from the Hessian inverse; optional diagnostic output
    if hasattr(res, 'hess_inv'):
        try:
            Hin = res.hess_inv.todense()
        except Exception:
            Hin = np.asarray(res.hess_inv)
        se = np.sqrt(np.diag(Hin)) if Hin.shape == (p, p) else np.full(p, np.nan)
    else:
        se = np.full(p, np.nan)

    return {
        'model': f'SEM_{combo}_{len(W_list)}W',
        'lam': lam,
        'beta': beta,
        'logL': logL,
        'AICc': AICc,
        'resid': v,
        'combo': combo,
        'success': res.success,
        'se': se
    }


def moran_I_val_from_array(resid, W_array, ids):
    """
    Compute Moran's I for residuals using a dense W matrix.
    """
    w_obj = util.full2W(W_array, ids=ids)
    mi = Moran(resid, w_obj, permutations=0)
    return mi.I, mi.p_norm


def row_standardize(A):
    """
    Row-standardize a spatial weight matrix.
    """
    R = A.astype(float)
    rs = R.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1.0
    return R / rs


def ensure_connected_within_state(W_sub, XY_sub):
    """
    If the within-state WQ submatrix has isolated units, connect each isolated unit
    once to its nearest neighbor, then row-standardize the matrix.
    """
    R = W_sub.copy()
    deg = R.sum(axis=1)

    if np.any(deg == 0):
        D = np.linalg.norm(
            XY_sub[:, None, :] - XY_sub[None, :, :], axis=2
        )
        np.fill_diagonal(D, np.inf)
        lonely = np.where(deg == 0)[0]
        for i in lonely:
            j = int(D[i].argmin())
            if np.isfinite(D[i, j]):
                R[i, j] = 1.0
                R[j, i] = 1.0
        R = row_standardize(R)

    return R


def submatrix(M, idx):
    """
    Return the submatrix M[idx, idx].
    """
    return M[np.ix_(idx, idx)]

# 3.2 Candidate weight combinations: WP, WS, WQ, plus WQ-anchored pairs --------

W_dict = {'WP': WP, 'WS': WS, 'WQ': WQ}

def make_tasks_add_single_pair(W_dict_local):
    """
    Single-channel models: WP, WS, and WQ.
    Two-channel models: keep only WQ+WP and WQ+WS, excluding WP+WS.
    """
    keys = list(W_dict_local.keys())
    tasks = []
    # Single-channel models
    for k in keys:
        tasks.append((f'SEM_add_1_{k}', [W_dict_local[k]], 'add'))

    # Two-channel models: only WQ+WP and WQ+WS are allowed
    if ('WP' in keys) and ('WQ' in keys):
        tasks.append(('SEM_add_2_WP_WQ',
                      [W_dict_local['WP'], W_dict_local['WQ']],
                      'add'))
    if ('WS' in keys) and ('WQ' in keys):
        tasks.append(('SEM_add_2_WS_WQ',
                      [W_dict_local['WS'], W_dict_local['WQ']],
                      'add'))
    return tasks

TASKS_MAIN_ADD = make_tasks_add_single_pair(W_dict)

# 3.3 Stage-1: US-wide 2019 in-sample ranking -------------------------------

base_rows = []

# OLS baseline; Moran's I is also reported for interpretability
ols = fit_ols(y, X)
I0, p0 = moran_I_val_from_array(ols['resid'], WQ, ids=IDs)
base_rows.append(['OLS', '-', 'OLS',
                  ols['logL'], ols['AICc'], I0, p0])

# All single- and two-channel SEM candidates
for name, Wset, combo in TASKS_MAIN_ADD:
    res = fit_sem_multi(y, X, Wset, combo=combo)
    I, pI = moran_I_val_from_array(res['resid'], WQ, ids=IDs)
    wmix = ';'.join(name.split('_')[3:])
    base_rows.append([name, wmix, combo,
                      res['logL'], res['AICc'], I, pI])

base_df = pd.DataFrame(
    base_rows,
    columns=['Model', 'W_mix', 'Combo', 'logL', 'AICc', 'MoranI', 'p_norm']
)
base_df = base_df.sort_values(['Combo', 'AICc'])

print(f"\n=== Stage-1: In-sample ranking for {outcome_tag} 2019 (US) ===")
print(base_df.head(10))

# 3.4 Stage-2: Leave-one-state-out validation for shortlisted models ----------

stage1_sem = base_df[(base_df['Model'] != 'OLS') &
                     (base_df['Combo'] == 'add')].copy()

single_pool = stage1_sem[stage1_sem['Model'].str.match(r'^SEM_add_1_')]
best_single_name = single_pool.nsmallest(1, 'AICc')['Model'].iloc[0]

TOP_K_PAIR = 5  # There are few pair candidates; 5 is sufficient here
pair_pool = stage1_sem[stage1_sem['Model'].str.match(r'^SEM_add_2_')]
pair_shortlist = pair_pool.nsmallest(TOP_K_PAIR, 'AICc')['Model'].tolist()

stage2_models = [best_single_name] + pair_shortlist

print(f"\n[Stage-1] Shortlist for LOSO (best single + TOP_{TOP_K_PAIR} pairs):")
print(stage2_models)


def _loso_state_worker(st, IDs_all, df_all, y_all, X_all, tasks,
                       WQ_array, XY_all_, K_):
    """
    One leave-one-state-out fold for a given STATE:
    - fit models on the training states;
    - compute logL, AICc, and Moran's I for the held-out state.
    """
    hold_ids = df_all.loc[df_all['STATE'] == st, 'FIPS'].tolist()
    hold_set = set(hold_ids)

    hold_idx = np.array(
        [i for i, idd in enumerate(IDs_all) if idd in hold_set],
        dtype=int
    )
    train_idx = np.array(
        [i for i in range(len(IDs_all)) if i not in hold_set],
        dtype=int
    )

    out_rows = []

    # Skip states with too few counties
    if len(hold_idx) < 5 or len(train_idx) < (K_ + 1 + 2):
        return out_rows

    # OLS baseline; log-likelihood is retained for a sanity check
    ols_tr = fit_ols(y_all[train_idx], X_all[train_idx, :])
    out_rows.append([st, 'OLS', '-', 'OLS',
                     ols_tr['logL'], np.nan, np.nan, np.nan])

    for name, Wset, combo in tasks:
        # Training set
        y_tr = y_all[train_idx]
        X_tr = X_all[train_idx, :]
        W_tr = [submatrix(Wk, train_idx) for Wk in Wset]
        res_tr = fit_sem_multi(y_tr, X_tr, W_tr, combo=combo)

        # holdout
        y_te = y_all[hold_idx]
        X_te = X_all[hold_idx, :]
        W_te = [submatrix(Wk, hold_idx) for Wk in Wset]
        N_te, K_te = X_te.shape
        m = len(W_te)

        A_te, logdetA_te = build_A_and_logdet(res_tr['lam'], W_te, combo=combo)
        if A_te is None:
            out_rows.append([
                st, name, ';'.join(name.split('_')[3:]), combo,
                -np.inf, np.inf, np.nan, np.nan
            ])
            continue

        e_te = y_te - X_te.dot(res_tr['beta'])
        v_te = A_te.dot(e_te)
        sse = float(v_te.T.dot(v_te))

        df_ = N_te - (K_te + m)
        sigma2 = sse/df_ if df_ > 0 else sse / max(N_te, 1)
        logL = (logdetA_te
                - (N_te/2)*np.log(2*np.pi*sigma2)
                - sse/(2*sigma2))
        AIC  = -2*logL + 2*(K_te + m)
        AICc = (AIC + (2*(K_te+m)*((K_te+m)+1)) /
                (N_te-(K_te+m)-1) if (N_te-(K_te+m)-1) > 0 else np.nan)

        # Moran's I: repair connectivity in the held-out state's WQ if needed
        WQ_te = submatrix(WQ_array, hold_idx)
        XY_te = XY_all_[hold_idx, :]
        WQ_eval = ensure_connected_within_state(WQ_te, XY_te)
        mi = Moran(
            v_te,
            util.full2W(WQ_eval, ids=[IDs_all[i] for i in hold_idx]),
            permutations=0
        )

        out_rows.append([
            st, name, ';'.join(name.split('_')[3:]), combo,
            logL, AICc, mi.I, mi.p_norm
        ])

    return out_rows


states = sorted(df['STATE'].unique())
tasks_for_loso = [t for t in TASKS_MAIN_ADD if t[0] in stage2_models]

res_lists = Parallel(
    n_jobs=N_JOBS, backend='loky', verbose=0
)(
    delayed(_loso_state_worker)(
        st, IDs, df, y, X, tasks_for_loso, WQ, XY_all, K
    )
    for st in states
)

cv_loso = pd.DataFrame(
    [row for rows in res_lists for row in rows],
    columns=['STATE','Model','W_mix','Combo',
             'pred_logL','pred_AICc','pred_MoranI','pred_p']
)

print(f"\n=== Stage-2: LOSO (shortlist, US, outcome={outcome_tag}) — preview ===")
print(cv_loso.head(10))

# 3.5 Forward gate：strict / lenient / super_lenient -------------------------

def summarize_models(cv_df, prefix='SEM_add_'):
    sub = cv_df[cv_df['Model'].str.startswith(prefix)].copy()
    g = (sub.groupby('Model', as_index=False)
           .agg(pred_AICc_mean=('pred_AICc','mean'),
                pred_AICc_sd  =('pred_AICc','std'),
                absMI_mean    =('pred_MoranI',
                                 lambda s: s.abs().mean()),
                absMI_sd      =('pred_MoranI',
                                 lambda s: s.abs().std())))
    return g.dropna(subset=['pred_AICc_mean','absMI_mean'])


sum_tbl = summarize_models(cv_loso, prefix='SEM_add_')
sum_tbl['dAICc_vs_best'] = (
    sum_tbl['pred_AICc_mean'] - sum_tbl['pred_AICc_mean'].min()
)

best_by_AICc_row  = sum_tbl.loc[sum_tbl['pred_AICc_mean'].idxmin()]
best_by_AICc_name = best_by_AICc_row['Model']

# Gate strategy
GATE_PROFILE        = 'super_lenient'   # Recommended: super_lenient
PREFER_PAIR_IF_TIED = True
TIE_DELTA_REL       = 0.01             # Treat relative AICc difference <= 1% as a tie

if GATE_PROFILE == 'strict':
    GATE_DAICC_REL_MAX       = -0.01
    GATE_MI_REL_IMPROVE_MIN  = 0.00
    GATE_WINRATE_MIN         = 0.70
elif GATE_PROFILE == 'lenient':
    GATE_DAICC_REL_MAX       = 0.00
    GATE_MI_REL_IMPROVE_MIN  = 0.05
    GATE_WINRATE_MIN         = 0.50
elif GATE_PROFILE == 'super_lenient':
    GATE_DAICC_REL_MAX       = 0.02
    GATE_MI_REL_IMPROVE_MIN  = -0.10
    GATE_WINRATE_MIN         = 0.10
else:
    raise ValueError("Unknown GATE_PROFILE")

best_single_row = sum_tbl[sum_tbl['Model'] == best_single_name].iloc[0]
pair_tbl = sum_tbl[sum_tbl['Model'].str.match(r'^SEM_add_2_')]

if len(pair_tbl) > 0:
    best_pair_row = pair_tbl.nsmallest(1, 'pred_AICc_mean').iloc[0]

    # Delta AICc: absolute and relative differences
    dAICc = (best_pair_row['pred_AICc_mean'] -
             best_single_row['pred_AICc_mean'])
    AICc_single = best_single_row['pred_AICc_mean']
    dAICc_rel = dAICc / max(abs(AICc_single), 1e-9)

    # Relative improvement in Moran's I
    absMI_single = best_single_row['absMI_mean']
    absMI_pair   = best_pair_row['absMI_mean']
    rel_mi_gain  = (
        (absMI_single - absMI_pair) /
        max(absMI_single, 1e-9)
    )

    # State-level win rate: proportion of held-out states where the pair model has lower AICc
    A_ = cv_loso[cv_loso['Model'] == best_pair_row['Model']][
        ['STATE','pred_AICc']
    ].rename(columns={'pred_AICc':'pair'})
    B_ = cv_loso[cv_loso['Model'] == best_single_name][
        ['STATE','pred_AICc']
    ].rename(columns={'pred_AICc':'single'})
    J = A_.merge(B_, on='STATE', how='inner')
    win_rate = float((J['pair'] < J['single']).mean()) if len(J) > 0 else 0.0

    # Gate decision
    pass_gate = (
        (dAICc_rel <= GATE_DAICC_REL_MAX) and
        (rel_mi_gain >= GATE_MI_REL_IMPROVE_MIN) and
        (win_rate >= GATE_WINRATE_MIN)
    )

    if pass_gate:
        final_model_main = best_pair_row['Model']
        gate_note = (
            f"[{GATE_PROFILE}] Accepted 2-channel by gates: "
            f"ΔAICc={dAICc:.2f}, ΔAICc_rel={dAICc_rel:.3f}<= {GATE_DAICC_REL_MAX}, "
            f"rel|I|↓={rel_mi_gain:.3f}>= {GATE_MI_REL_IMPROVE_MIN}, "
            f"win_rate={win_rate:.2f}>= {GATE_WINRATE_MIN}."
        )
    else:
        # If the gate fails but dAICc_rel is very small, tie-preference can be triggered
        if (GATE_PROFILE == 'super_lenient' and
            PREFER_PAIR_IF_TIED and
            (dAICc_rel <= TIE_DELTA_REL) and
            (rel_mi_gain >= GATE_MI_REL_IMPROVE_MIN) and
            (win_rate >= GATE_WINRATE_MIN/2)):
            final_model_main = best_pair_row['Model']
            gate_note = (
                f"[{GATE_PROFILE}] Tie-preference to 2-channel "
                f"(ΔAICc_rel={dAICc_rel:.3f}<= {TIE_DELTA_REL}); "
                f"rel|I|↓={rel_mi_gain:.3f}, win_rate={win_rate:.2f}."
            )
        else:
            final_model_main = best_single_name
            gate_note = (
                f"[{GATE_PROFILE}] Kept best single: pair failed gates "
                f"and tie-preference not triggered. "
                f"ΔAICc={dAICc:.2f}, ΔAICc_rel={dAICc_rel:.3f}, "
                f"rel|I|↓={rel_mi_gain:.3f}, win_rate={win_rate:.2f}."
            )
else:
    final_model_main = best_single_name
    dAICc = np.nan
    dAICc_rel = np.nan
    rel_mi_gain = np.nan
    win_rate = np.nan
    gate_note = f"[{GATE_PROFILE}] No eligible two-channel; kept best single."

print(f"\n[Final model for {outcome_tag} 2019, US] {final_model_main}")
print("[Gate note] ", gate_note)
print("[Best-by-AICc on shortlist] ", best_by_AICc_name)

# ============================================
# 9. Add vs Prod sensitivity (US, current STD outcome)
#    Compare additive A(lambda) vs product A(lambda) using the same W set as the final additive model
# ============================================

RUN_PROD_SENS = True  # Switch add-vs-prod sensitivity analysis on/off

if RUN_PROD_SENS:
    # 9.1 Parse channel names from the model name (WP / WS / WQ ...)
    def parse_W_keys_from_model(model_name, W_dict_local):
        """
        Examples:
          'SEM_add_1_WS'      -> ['WS']
          'SEM_add_2_WS_WQ'   -> ['WS','WQ']
        Keep only keys that exist in W_dict_local.
        """
        parts = model_name.split('_')
        chans = [p for p in parts[3:] if p in W_dict_local]
        if len(chans) == 0:
            raise ValueError(f"Cannot parse channels from model name: {model_name}")
        return chans

    W_keys_final = parse_W_keys_from_model(final_model_main, W_dict)
    Wset_final   = [W_dict[k] for k in W_keys_final]

    # 9.2 Fit additive and product forms using the same W set

    # Additive form: main specification, refit on the full sample for consistent output
    res_add_full = fit_sem_multi(y, X, Wset_final, combo='add')
    I_add_full, p_add_full = moran_I_val_from_array(
        res_add_full['resid'], WQ, ids=IDs
    )

    # Product form: sensitivity analysis
    res_prod_full = fit_sem_multi(y, X, Wset_final, combo='prod')
    I_prod_full, p_prod_full = moran_I_val_from_array(
        res_prod_full['resid'], WQ, ids=IDs
    )

    # 9.3 Assemble and save the sensitivity table
    sens_df = pd.DataFrame([
        {
            'Outcome_tag': outcome_tag,
            'Model_name': final_model_main,
            'Combo': 'add',
            'W_mix': ';'.join(W_keys_final),
            'logL': res_add_full['logL'],
            'AICc': res_add_full['AICc'],
            'MoranI': I_add_full,
            'absMoranI': abs(I_add_full),
            'p_norm': p_add_full
        },
        {
            'Outcome_tag': outcome_tag,
            'Model_name': final_model_main.replace('SEM_add', 'SEM_prod'),
            'Combo': 'prod',
            'W_mix': ';'.join(W_keys_final),
            'logL': res_prod_full['logL'],
            'AICc': res_prod_full['AICc'],
            'MoranI': I_prod_full,
            'absMoranI': abs(I_prod_full),
            'p_norm': p_prod_full
        }
    ])

    print("\n=== Add vs Prod (sensitivity on final W-combo, US) ===")
    print(sens_df)

    suffix_gate = f"_{GATE_PROFILE}" if 'GATE_PROFILE' in globals() else ""
    out_name = f"APPENDIX_add_vs_prod_sensitivity_US_{outcome_tag}{suffix_gate}.csv"
    sens_df.to_csv(out_name, index=False)

    print(f"\n[Add vs Prod sensitivity] Saved: {out_name}")

else:
    print("\n[Add vs Prod sensitivity] RUN_PROD_SENS = False, skipped.")

##Table S5

In [ ]:
# ============================================================
#  SEM (multi-W, US) - Fast Covariate Robustness Check
#
#  Goal:
#    Covariate sensitivity (US-wide), mirroring the Deep South logic:
#      - 9 baseline covariates
#      - 4 "socio-sensitive" vars: Poverty, Noinsuranc, crime16to1, Dissimilar
#      - 11 covariate profiles:
#          * full (all 9)
#          * drop each of the 4 sensitive vars (x4)
#          * drop pairs of the 4 sensitive vars (C(4,2) = 6)
#
#  Data & W-matrices:
#    - US_HIV_Merged_total_final.shp (same as main US analysis)
#    - US_County_PCI_2019.csv      -> WP (Twitter PCI)
#    - processed_sci_summary.csv   -> WS (Facebook SCI)
#    - Queen contiguity            -> WQ (adjacency)
#
#  Channel search space (consistent with US main SEM analysis):
#    - Singles: WP, WS, WQ
#    - Pairs:   WP+WQ, WS+WQ    (NO WP+WS)
#
#  Output (CSV):
#    - COV_US_in_sample_add_<cov_profile>_<GATE_PROFILE>.csv
#    - COV_US_final_decision_allProfiles_<outcome>_<GATE_PROFILE>.csv
#    - COV_US_model_frequency_<outcome>_<GATE_PROFILE>.csv
#    - COV_US_Wmix_frequency_<outcome>_<GATE_PROFILE>.csv
#    - COV_US_lambda_table_<outcome>_<GATE_PROFILE>.csv
#
#  Note:
#    - No LOSO / cross-validation here (too expensive for 11 profiles).
#    - Only full-sample fits per profile for speed.
# ============================================================

import numpy as np
import pandas as pd
import geopandas as gpd
from itertools import combinations
from numpy.linalg import slogdet
from scipy.optimize import minimize
from esda.moran import Moran
from libpysal.weights import util, W, Queen

np.seterr(all="ignore")
RANDOM_SEED = 7
rng = np.random.default_rng(RANDOM_SEED)

# ----------------------------
# Mode / gate configs
# ----------------------------
RUN_PROD = False  # keep for compatibility; not really used here

# Gate profile for pair vs single (in-sample only)
GATE_PROFILE = "lenient"          # options: "strict" or "lenient"
PREFER_PAIR_IF_TIED = True
TIE_DELTA = 2.0                   # AICc "tie" width (pair - single <= TIE_DELTA)

if GATE_PROFILE == "strict":
    # Pair must clearly outperform single (in-sample)
    GATE_DAICC_THRESH = -4.0      # ΔAICc = pair - single (negative = better)
    GATE_MI_REL_IMPROVE_MIN = 0.00  # |I| at least not worse
else:
    # Lenient: allow small AICc disadvantage if Moran's I improves
    GATE_DAICC_THRESH = +1.0      # allow pair up to +1 AICc worse
    GATE_MI_REL_IMPROVE_MIN = 0.02  # ≥ 2% reduction in |I|


# ============================================================
# A. Read US shapefile & basic preprocessing
# ============================================================

shp_path = "US_HIV_Merged_total_final.shp"
gdf = gpd.read_file(shp_path)

if "FIPS" not in gdf.columns:
    raise KeyError("Shapefile needs a 'FIPS' field named 'FIPS'.")

# Clean FIPS: strip leading zeros, drop duplicates
gdf["FIPS"] = gdf["FIPS"].astype(str).str.lstrip("0")
gdf = gdf.drop_duplicates(subset=["FIPS"])

# Fix possibly truncated 2018 variable names (for consistency; not directly used here)
rename_map = {
    "RateChlamy": "RateChlamydia2018",
    "RateGonorr": "RateGonorrhea2018",
    "RateHIV201": "RateHIV2018"
}
for short, long_name in rename_map.items():
    if (short in gdf.columns) and (long_name not in gdf.columns):
        gdf = gdf.rename(columns={short: long_name})

# Baseline covariates (same 9 as in main US analysis)
X_cols = [
    "Population", "UrbanRural", "Female", "Old", "Black",
    "Noinsuranc", "Poverty", "crime16to1", "Dissimilar"
]

cols_keep = [
    "FIPS",
    "GonorrheaR", "ChlamydiaR", "PercenHIVp",
    *X_cols,
    "RateChlamydia2018", "RateGonorrhea2018", "RateHIV2018",
    "geometry"
]

df_geo = gdf[cols_keep].copy()
df_geo = gpd.GeoDataFrame(df_geo, geometry="geometry", crs=gdf.crs)

# STATE code from FIPS
def fips2state(x):
    return str(x).zfill(5)[:2]

df_geo["STATE"] = df_geo["FIPS"].apply(fips2state)


# ============================================================
# B. Build spatial weights (WP / WS / WQ) — US-wide
# ============================================================

def subset_W(w: W, keep_ids):
    """
    Subset a libpysal W to a given id set, preserving the given ID order.
    """
    keep_ids = list(dict.fromkeys([str(x) for x in keep_ids]))
    keep = set(keep_ids)
    new_neighbors, new_weights = {}, {}
    for i in w.id_order:
        if i not in keep:
            continue
        nbs = w.neighbors.get(i, [])
        wts = w.weights.get(i, [])
        kn, kw = [], []
        for nb, wt in zip(nbs, wts):
            if nb in keep:
                kn.append(nb)
                kw.append(wt)
        new_neighbors[i] = kn
        new_weights[i] = kw
    return W(new_neighbors, new_weights, ids=keep_ids)


# 1) WP: Twitter PCI mobility
pci = pd.read_csv("US_County_PCI_2019.csv")
pci["place_i"] = pci["place_i"].astype(str).str.lstrip("0")
pci["place_j"] = pci["place_j"].astype(str).str.lstrip("0")

pci_f = pci[
    pci["place_i"].isin(df_geo["FIPS"])
    & pci["place_j"].isin(df_geo["FIPS"])
].copy()

A_pci = pci_f.pivot_table(
    index="place_i", columns="place_j",
    values="pci", fill_value=0
)
np.fill_diagonal(A_pci.values, 0.0)
A = A_pci.values.astype(float)
A = A + A.T - np.diag(np.diag(A))  # symmetrize
row_sum = A.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WP_full = A / row_sum
wP = util.full2W(WP_full, ids=list(A_pci.index))

# 2) WS: Facebook SCI ties
sci = pd.read_csv("processed_sci_summary.csv")
sci["user_loc"] = sci["user_loc"].astype(str).str.lstrip("0")
sci["tfr_loc"]  = sci["tfr_loc"].astype(str).str.lstrip("0")

sci_f = sci[
    sci["user_loc"].isin(df_geo["FIPS"])
    & sci["tfr_loc"].isin(df_geo["FIPS"])
].copy()

A_sci = sci_f.pivot_table(
    index="user_loc", columns="tfr_loc",
    values="tscaled_sci", fill_value=0
)
np.fill_diagonal(A_sci.values, 0.0)
B = A_sci.values.astype(float)
B = B + B.T - np.diag(np.diag(B))
row_sum = B.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
WS_full = B / row_sum
wS = util.full2W(WS_full, ids=list(A_sci.index))

# 3) WQ: queen contiguity (row-standardized)
g3857 = df_geo.to_crs(epsg=3857)
wQ = Queen.from_dataframe(g3857, ids=list(df_geo["FIPS"]))
wQ.transform = "R"  # row-standardize

# Harmonize ID set & ordering across df_geo, WP, WS, WQ
common = (
    set(df_geo["FIPS"])
    & set(wP.id_order)
    & set(wS.id_order)
    & set(wQ.id_order)
)

df_geo = df_geo[df_geo["FIPS"].isin(common)].copy()
wP = subset_W(wP, common)
wS = subset_W(wS, common)
wQ = subset_W(wQ, common)

df_geo = df_geo.set_index("FIPS").loc[wQ.id_order].reset_index()
IDs = list(wQ.id_order)
STATE = df_geo["STATE"].values

WP, _ = wP.full()
WS, _ = wS.full()
WQ, _ = wQ.full()


# ============================================================
# C. Outcome & covariate profiles (US-wide)
# ============================================================

# Outcome: HIV prevalence (%)
y_col = "PercenHIVp"   # can be switched to 'GonorrheaR' / 'ChlamydiaR' if needed
outcome_tag = {
    "PercenHIVp": "HIV",
    "GonorrheaR": "Gonorrhea",
    "ChlamydiaR": "Chlamydia"
}.get(y_col, y_col)

# Full 9 covariates (same as X_cols above)
X_cols_full = X_cols.copy()

# Socioeconomic vars that reviewers are most concerned about
SOCIO_SENSITIVE = ["Poverty", "Noinsuranc", "crime16to1", "Dissimilar"]


def build_profile(base_vars, drop_vars):
    drop_set = set(drop_vars)
    return [v for v in base_vars if v not in drop_set]


# Generate 11 covariate profiles: full + drop 1 + drop 2 (Deep South logic)
COV_PROFILES = {}

# 1) full model (9 covariates)
COV_PROFILES["full"] = X_cols_full

# 2) drop single sensitive variable
for v in SOCIO_SENSITIVE:
    name = f"drop_{v}"
    COV_PROFILES[name] = build_profile(X_cols_full, [v])

# 3) drop pairs of sensitive variables
for v1, v2 in combinations(SOCIO_SENSITIVE, 2):
    name = f"drop_{v1}_{v2}"
    COV_PROFILES[name] = build_profile(X_cols_full, [v1, v2])


# ============================================================
# D. OLS / SEM / Moran helpers
# ============================================================

def fit_ols(y, X):
    """
    Standard OLS fit.
    Returns beta, residuals, log-likelihood, and AICc.
    """
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X.dot(beta)
    sse = float(e.T.dot(e))
    N, K = X.shape
    df_ = N - K
    sigma2 = sse / df_
    logL = (
        -(N / 2) * np.log(2 * np.pi * sigma2)
        - sse / (2 * sigma2)
    )
    AIC = -2 * logL + 2 * K
    AICc = (
        AIC + (2 * K * (K + 1)) / (N - K - 1)
        if (N - K - 1) > 0 else np.nan
    )
    return {
        "model": "OLS",
        "beta": beta,
        "logL": logL,
        "AICc": AICc,
        "resid": e
    }


def build_A_and_logdet(lam, W_list, combo="add"):
    """
    Build A and log|A| for SEM:
      y = Xβ + u, (I - Σ λ_k W_k) u = ε.
    """
    N = W_list[0].shape[0]
    I = np.eye(N)

    if combo == "add":
        A = I.copy()
        for i, Wk in enumerate(W_list):
            A -= lam[i] * Wk
        sign, logdetA = slogdet(A)
        if sign <= 0:
            return None, None
        return A, logdetA

    elif combo == "prod":
        # Not used here, kept for compatibility
        A = I.copy()
        logdet_sum = 0.0
        for i, Wk in enumerate(W_list):
            Mi = I - lam[i] * Wk
            sign_i, logdet_i = slogdet(Mi)
            if sign_i <= 0:
                return None, None
            logdet_sum += logdet_i
            A = Mi @ A
        return A, logdet_sum

    else:
        raise ValueError("combo must be 'add' or 'prod'")


def neglog_sem(params, y, X, W_list, combo="add"):
    """
    Negative log-likelihood of SEM for optimization.
    params = [λ_1,...,λ_m, β_0,...,β_{K-1}]
    """
    N, K = X.shape
    m = len(W_list)
    lam = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(lam, W_list, combo=combo)
    if A is None:
        return 1e12

    e = y - X.dot(beta)
    v = A.dot(e)
    sse = float(v.T.dot(v))

    df_ = N - (K + m)
    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_
    if sigma2 <= 0:
        return 1e12

    logL = (
        logdetA
        - (N / 2) * np.log(2 * np.pi * sigma2)
        - sse / (2 * sigma2)
    )
    return -logL


def fit_sem(y, X, W_list, method="L-BFGS-B", combo="add"):
    """
    Single-start ML SEM fit (sufficient for covariate robustness).
    """
    N, K = X.shape
    m = len(W_list)

    # Starting point: OLS beta, λ = 0
    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
    x0 = np.r_[np.zeros(m), beta0]

    bounds = [(-0.99, 0.99)] * m + [(-np.inf, np.inf)] * K

    res = minimize(
        neglog_sem, x0,
        args=(y, X, W_list, combo),
        method=method,
        bounds=bounds,
        options={"maxiter": 400, "ftol": 1e-5, "gtol": 1e-5},
    )

    params = res.x
    lam = params[:m]
    beta = params[m:]
    logL = -res.fun
    p = m + K

    AIC = -2 * logL + 2 * p
    AICc = (
        AIC + (2 * p * (p + 1)) / (N - p - 1)
        if (N - p - 1) > 0 else np.nan
    )

    A, _ = build_A_and_logdet(lam, W_list, combo=combo)
    v = A.dot(y - X.dot(beta))

    return {
        "model": f"SEM_{combo}_{m}W",
        "lam": lam,
        "beta": beta,
        "logL": logL,
        "AICc": AICc,
        "resid": v,
        "combo": combo,
        "success": res.success,
    }


def moran_I_val_from_array(resid, W_array, ids):
    """
    Compute Moran's I using a numpy W-array and ID order.
    """
    w_obj = util.full2W(W_array, ids=ids)
    mi = Moran(resid, w_obj, permutations=0)
    return mi.I, mi.p_norm


# ============================================================
# E. Candidate SEM tasks (single + pair, add) — US channel set
# ============================================================

CHANNEL_KEYS = ["WP", "WS", "WQ"]


def make_tasks_add_single_pair_US(W_dict_local):
    """
    Candidate SEM models (consistent with main US analysis):
      - Singles: WP, WS, WQ
      - Pairs:   WP+WQ, WS+WQ
      (No WP+WS pair)
    """
    tasks = []
    # singles
    for k in ["WP", "WS", "WQ"]:
        if k in W_dict_local:
            tasks.append((f"SEM_add_1_{k}", [W_dict_local[k]], "add"))

    # pairs: WP+WQ / WS+WQ if both exist
    if "WP" in W_dict_local and "WQ" in W_dict_local:
        tasks.append(("SEM_add_2_WP_WQ",
                      [W_dict_local["WP"], W_dict_local["WQ"]], "add"))
    if "WS" in W_dict_local and "WQ" in W_dict_local:
        tasks.append(("SEM_add_2_WS_WQ",
                      [W_dict_local["WS"], W_dict_local["WQ"]], "add"))

    return tasks


def extract_W_mix_from_model(name: str) -> str:
    """
    Extract W-mix part from a model name, e.g.
      'SEM_add_2_WP_WQ' -> 'WP_WQ'
    """
    parts = name.split("_")
    if len(parts) >= 4:
        return "_".join(parts[3:])
    return ""


# ============================================================
# F. Main loop over covariate profiles (US-wide)
# ============================================================

all_decisions = []
lambda_rows = []

for cov_name, X_cols_profile in COV_PROFILES.items():
    print("\n" + "=" * 80)
    print(f"Running US covariate profile: {cov_name}  "
          f"(#{len(X_cols_profile)} covariates)")
    print("X_cols =", X_cols_profile)
    print("=" * 80)

    # 1) Subset data for this profile: require outcome + covariates not NA
    df_sub = df_geo[["FIPS", y_col] + X_cols_profile + ["STATE", "geometry"]].dropna().copy()
    fips_sub = df_sub["FIPS"].tolist()
    keep_set = set(fips_sub)

    # indices in original ID order
    idx_sub = [i for i, idd in enumerate(IDs) if idd in keep_set]
    if len(idx_sub) == 0:
        print(f"[WARN] cov_profile={cov_name} -> empty sample; skip.")
        continue

    IDs_sub = [IDs[i] for i in idx_sub]

    def submatrix(M, idx):
        return M[np.ix_(idx, idx)]

    WP_sub = submatrix(WP, idx_sub)
    WS_sub = submatrix(WS, idx_sub)
    WQ_sub = submatrix(WQ, idx_sub)

    # Reorder df_sub to match IDs_sub
    df_sub = df_sub.set_index("FIPS").loc[IDs_sub].reset_index()

    y = df_sub[y_col].values
    X_raw = df_sub[X_cols_profile].values
    N = X_raw.shape[0]
    X = np.hstack([np.ones((N, 1)), X_raw])
    K = X.shape[1]

    W_dict_local = {
        "WP": WP_sub,
        "WS": WS_sub,
        "WQ": WQ_sub,
    }
    TASKS_MAIN_ADD = make_tasks_add_single_pair_US(W_dict_local)

    # ----------------------------
    # Stage-1: in-sample OLS + SEM
    # ----------------------------
    base_rows = []

    # OLS baseline
    ols = fit_ols(y, X)
    I0, p0 = moran_I_val_from_array(ols["resid"], WQ_sub, ids=IDs_sub)
    base_rows.append(
        ["OLS", "-", "OLS", ols["logL"], ols["AICc"], I0, p0]
    )

    # SEM: all single + pair tasks
    for name, Wset, combo in TASKS_MAIN_ADD:
        res = fit_sem(y, X, Wset, combo=combo)
        I, pI = moran_I_val_from_array(res["resid"], WQ_sub, ids=IDs_sub)
        wmix = ";".join(name.split("_")[3:])
        base_rows.append(
            [name, wmix, combo, res["logL"], res["AICc"], I, pI]
        )

    base_df = pd.DataFrame(
        base_rows,
        columns=[
            "Model", "W_mix", "Combo",
            "logL", "AICc",
            "MoranI", "p_norm"
        ],
    )
    base_df = base_df.sort_values(["Combo", "AICc"])

    print(f"\n=== In-sample ranking (AICc) [US, cov={cov_name}, outcome={outcome_tag}] ===")
    print(base_df.head(10))

    # Save per-profile in-sample table
    base_path = f"COV_US_in_sample_add_{cov_name}_{outcome_tag}_{GATE_PROFILE}.csv"
    base_df.to_csv(base_path, index=False)
    print(f"Saved in-sample table for cov_profile={cov_name} -> {base_path}")

    # ----------------------------
    # Simple gate: best single vs best pair (in-sample)
    # ----------------------------
    stage1_sem = base_df[(base_df["Model"] != "OLS")].copy()
    single_pool = stage1_sem[stage1_sem["Model"].str.match(r"^SEM_add_1_")]
    pair_pool   = stage1_sem[stage1_sem["Model"].str.match(r"^SEM_add_2_")]

    if len(single_pool) == 0:
        print(f"[WARN] No single-channel SEM for cov_profile={cov_name}.")
        continue

    best_single_row = single_pool.nsmallest(1, "AICc").iloc[0]
    best_single_name = best_single_row["Model"]

    if len(pair_pool) > 0:
        best_pair_row = pair_pool.nsmallest(1, "AICc").iloc[0]
        best_pair_name = best_pair_row["Model"]

        dAICc = best_pair_row["AICc"] - best_single_row["AICc"]  # pair - single

        absMI_single = abs(best_single_row["MoranI"])
        absMI_pair   = abs(best_pair_row["MoranI"])
        rel_mi_gain  = (
            (absMI_single - absMI_pair) / max(absMI_single, 1e-9)
        )

        # In cov-robustness, "win_rate" is degenerate: 1 if pair has smaller AICc, else 0.
        win_rate = 1.0 if best_pair_row["AICc"] < best_single_row["AICc"] else 0.0

        pass_gate = (
            (dAICc <= GATE_DAICC_THRESH)
            and (rel_mi_gain >= GATE_MI_REL_IMPROVE_MIN)
        )

        if pass_gate:
            final_model = best_pair_name
            gate_note = (
                f"[{GATE_PROFILE}] Accepted 2-channel by in-sample gate "
                f"(cov={cov_name}, outcome={outcome_tag}): "
                f"ΔAICc={dAICc:.2f} <= {GATE_DAICC_THRESH}, "
                f"rel|I|↓={rel_mi_gain:.3f} >= {GATE_MI_REL_IMPROVE_MIN}."
            )
        else:
            if PREFER_PAIR_IF_TIED and (dAICc <= TIE_DELTA):
                final_model = best_pair_name
                gate_note = (
                    f"[{GATE_PROFILE}] Tie-preference to 2-channel "
                    f"(cov={cov_name}, outcome={outcome_tag}; "
                    f"ΔAICc={dAICc:.2f} <= {TIE_DELTA}); "
                    f"rel|I|↓={rel_mi_gain:.3f}."
                )
            else:
                final_model = best_single_name
                gate_note = (
                    f"[{GATE_PROFILE}] Kept best single (cov={cov_name}, outcome={outcome_tag}): "
                    f"pair failed gate / tie condition. "
                    f"ΔAICc={dAICc:.2f}, rel|I|↓={rel_mi_gain:.3f}."
                )
    else:
        final_model = best_single_name
        dAICc = np.nan
        rel_mi_gain = np.nan
        win_rate = np.nan
        gate_note = (
            f"[{GATE_PROFILE}] No 2-channel candidate (cov={cov_name}, outcome={outcome_tag}); "
            f"kept best single."
        )

    print(f"\n[Final (US)] cov_profile={cov_name}, outcome={outcome_tag} -> {final_model}")
    print("[Gate note] ", gate_note)

    # Record decision line for this profile
    decision_df = pd.DataFrame({
        "Outcome_tag": [outcome_tag],
        "Outcome_col": [y_col],
        "Cov_profile": [cov_name],
        "n_covariates": [len(X_cols_profile)],
        "Profile": [GATE_PROFILE],
        "Final_model": [final_model],
        "Final_W_mix": [extract_W_mix_from_model(final_model)],
        "Best_single": [best_single_name],
        "Decision": [gate_note],
        "AICc_final": [
            float(
                base_df.loc[base_df["Model"] == final_model, "AICc"].iloc[0]
            )
        ],
        "MoranI_final": [
            float(
                base_df.loc[base_df["Model"] == final_model, "MoranI"].iloc[0]
            )
        ],
        "AICc_best_single": [float(best_single_row["AICc"])],
        "MoranI_best_single": [float(best_single_row["MoranI"])],
        "dAICc_pair_vs_single": [dAICc],
        "Rel_MI_gain(pair_vs_single)": [rel_mi_gain],
        "Win_rate_pair_vs_single": [win_rate],
        "GATE_DAICC_THRESH": [GATE_DAICC_THRESH],
        "GATE_MI_REL_IMPROVE_MIN": [GATE_MI_REL_IMPROVE_MIN],
        "PREFER_PAIR_IF_TIED": [PREFER_PAIR_IF_TIED],
        "TIE_DELTA": [TIE_DELTA],
    })
    all_decisions.append(decision_df)

    # Record lambda for the final model (for robustness narrative)
    if final_model.startswith("SEM_add_1_"):
        _, Wkey = final_model.split("SEM_add_1_")
        Wset_final = [W_dict_local[Wkey]]
    elif final_model.startswith("SEM_add_2_"):
        parts = final_model.split("_")
        Wkey1, Wkey2 = parts[3], parts[4]
        Wset_final = [W_dict_local[Wkey1], W_dict_local[Wkey2]]
    else:
        Wset_final = []

    if len(Wset_final) > 0:
        res_final = fit_sem(y, X, Wset_final, combo="add")
        lam = res_final["lam"]
        lam_row = {
            "Outcome_tag": outcome_tag,
            "Outcome_col": y_col,
            "Cov_profile": cov_name,
            "Final_model": final_model,
            "Final_W_mix": extract_W_mix_from_model(final_model),
        }
        for i, val in enumerate(lam):
            lam_row[f"lambda_{i+1}"] = float(val)
        lambda_rows.append(lam_row)


# ============================================================
# G. Cross-profile summaries (US)
# ============================================================

if all_decisions:
    meta_df = pd.concat(all_decisions, ignore_index=True)
    meta_path = (
        f"COV_US_final_decision_allProfiles_{outcome_tag}_{GATE_PROFILE}.csv"
    )
    meta_df.to_csv(meta_path, index=False)

    # Frequency of each specific final model
    model_freq = (
        meta_df.groupby("Final_model", as_index=False)
        .agg(
            n_profiles=("Cov_profile", "nunique"),
            mean_n_covariates=("n_covariates", "mean"),
            mean_AICc_final=("AICc_final", "mean"),
        )
        .sort_values("n_profiles", ascending=False)
    )
    model_freq_path = (
        f"COV_US_model_frequency_{outcome_tag}_{GATE_PROFILE}.csv"
    )
    model_freq.to_csv(model_freq_path, index=False)

    # Frequency of each W-mix (aggregated across profiles)
    mix_freq = (
        meta_df.groupby("Final_W_mix", as_index=False)
        .agg(n_profiles=("Cov_profile", "nunique"))
        .sort_values("n_profiles", ascending=False)
    )
    mix_freq_path = (
        f"COV_US_Wmix_frequency_{outcome_tag}_{GATE_PROFILE}.csv"
    )
    mix_freq.to_csv(mix_freq_path, index=False)

    # Lambda table (for robustness narrative)
    if lambda_rows:
        lam_df = pd.DataFrame(lambda_rows)
        lam_path = f"COV_US_lambda_table_{outcome_tag}_{GATE_PROFILE}.csv"
        lam_df.to_csv(lam_path, index=False)
    else:
        lam_path = "(none)"

    print("\n[Summary across covariate profiles, US]")
    print(f"  - Decisions per profile: {meta_path}")
    print(f"  - Model frequency:       {model_freq_path}")
    print(f"  - W-mix frequency:       {mix_freq_path}")
    print(f"  - Lambda table:          {lam_path}")
else:
    print("\n[INFO] No valid covariate profiles were estimated (US).")


##Table S6

In [ ]:
# ============================================
# OLS vs hero SEM vs hero-structure SLM (US)
# ============================================

import numpy as np
import pandas as pd
import geopandas as gpd

from numpy.linalg import slogdet
from scipy.optimize import minimize
from esda.moran import Moran
from libpysal.weights import util, W, Queen

np.seterr(all="ignore")


# --------------------------------------------
# 0. Global config
# --------------------------------------------

# Outcome column:
#   HIV:        y_col = "PercenHIVp"
#   Gonorrhea:  y_col = "GonorrheaR"
#   Chlamydia:  y_col = "ChlamydiaR"
y_col = "PercenHIVp"

# Pre-specified hero SEM structure.
# Update this according to the selected 2019 hero model:
#   HIV:        "SEM_add_2_WP_WQ"
#   Gonorrhea:  "SEM_add_1_WS"
#   Chlamydia:  "SEM_add_1_WP"
hero_model = "SEM_add_2_WP_WQ"

OUTCOME_TAG = {
    "PercenHIVp": "HIV",
    "GonorrheaR": "Gonorrhea",
    "ChlamydiaR": "Chlamydia",
}

outcome_tag = OUTCOME_TAG.get(y_col, y_col)

# Input file paths
shp_path = "US_HIV_Merged_total_final.shp"
pci_path = "US_County_PCI_2019.csv"
sci_path = "processed_sci_summary.csv"

RANDOM_SEED = 7

# Covariates used in the main analysis
X_cols = [
    "Population",
    "UrbanRural",
    "Female",
    "Old",
    "Black",
    "Noinsuranc",
    "Poverty",
    "crime16to1",
    "Dissimilar",
]


# --------------------------------------------
# 1. Read shapefile and basic preprocessing
# --------------------------------------------

gdf = gpd.read_file(shp_path)

if "FIPS" not in gdf.columns:
    raise KeyError("The shapefile must contain a 'FIPS' field.")

# Clean FIPS codes and remove duplicated county records
gdf["FIPS"] = gdf["FIPS"].astype(str).str.strip().str.lstrip("0")
gdf = gdf.drop_duplicates(subset=["FIPS"]).copy()

# Check required columns
cols_need = ["FIPS", y_col] + X_cols + ["geometry"]
missing_cols = set(cols_need) - set(gdf.columns)

if missing_cols:
    raise KeyError(f"The shapefile is missing required columns: {missing_cols}")

df_geo = gdf[cols_need].copy()

# Drop counties with missing outcome or covariates
df_geo = df_geo.dropna(subset=[y_col] + X_cols).copy()


def fips2state(x):
    """
    Extract two-digit state FIPS code from county FIPS.
    """
    return str(x).zfill(5)[:2]


df_geo["STATE"] = df_geo["FIPS"].apply(fips2state)

df_geo = gpd.GeoDataFrame(
    df_geo,
    geometry="geometry",
    crs=gdf.crs,
)


# --------------------------------------------
# 2. Build spatial weight matrices: WP / WS / WQ
# --------------------------------------------

# 2.1 WP: Twitter PCI mobility network
pci = pd.read_csv(pci_path)

pci["place_i"] = pci["place_i"].astype(str).str.strip().str.lstrip("0")
pci["place_j"] = pci["place_j"].astype(str).str.strip().str.lstrip("0")

pci_f = pci[
    pci["place_i"].isin(df_geo["FIPS"])
    & pci["place_j"].isin(df_geo["FIPS"])
].copy()

A_pci = pci_f.pivot_table(
    index="place_i",
    columns="place_j",
    values="pci",
    fill_value=0.0,
)

np.fill_diagonal(A_pci.values, 0.0)

A = A_pci.values.astype(float)

# Symmetrize and row-standardize
A = A + A.T - np.diag(np.diag(A))
np.fill_diagonal(A, 0.0)

row_sum = A.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0

WP_full = A / row_sum
wP = util.full2W(WP_full, ids=list(A_pci.index))


# 2.2 WS: Facebook SCI social-tie network
sci = pd.read_csv(sci_path)

sci["user_loc"] = sci["user_loc"].astype(str).str.strip().str.lstrip("0")
sci["tfr_loc"] = sci["tfr_loc"].astype(str).str.strip().str.lstrip("0")

sci_f = sci[
    sci["user_loc"].isin(df_geo["FIPS"])
    & sci["tfr_loc"].isin(df_geo["FIPS"])
].copy()

A_sci = sci_f.pivot_table(
    index="user_loc",
    columns="tfr_loc",
    values="tscaled_sci",
    fill_value=0.0,
)

np.fill_diagonal(A_sci.values, 0.0)

B = A_sci.values.astype(float)

# Symmetrize and row-standardize
B = B + B.T - np.diag(np.diag(B))
np.fill_diagonal(B, 0.0)

row_sum = B.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0

WS_full = B / row_sum
wS = util.full2W(WS_full, ids=list(A_sci.index))


# 2.3 WQ: Queen contiguity network
g3857 = df_geo.to_crs(epsg=3857)

wQ = Queen.from_dataframe(
    g3857,
    ids=list(df_geo["FIPS"]),
)

wQ.transform = "R"
WQ_full, _ = wQ.full()


# --------------------------------------------
# 3. Align IDs and matrix order
# --------------------------------------------

common = (
    set(df_geo["FIPS"])
    & set(wP.id_order)
    & set(wS.id_order)
    & set(wQ.id_order)
)


def subset_W(w: W, keep_ids):
    """
    Subset a libpysal W object to the selected IDs.
    """
    keep_ids = list(dict.fromkeys([str(x) for x in keep_ids]))
    keep = set(keep_ids)

    new_neighbors = {}
    new_weights = {}

    for i in w.id_order:
        if i not in keep:
            continue

        nbs = w.neighbors.get(i, [])
        wts = w.weights.get(i, [])

        kept_neighbors = []
        kept_weights = []

        for nb, wt in zip(nbs, wts):
            if nb in keep:
                kept_neighbors.append(nb)
                kept_weights.append(wt)

        new_neighbors[i] = kept_neighbors
        new_weights[i] = kept_weights

    return W(new_neighbors, new_weights, ids=keep_ids)


# Subset to common IDs
df_geo = df_geo[df_geo["FIPS"].isin(common)].copy()

wP = subset_W(wP, common)
wS = subset_W(wS, common)
wQ = subset_W(wQ, common)

# Use WQ order as the common order
df_geo = df_geo.set_index("FIPS").loc[wQ.id_order].reset_index()

IDs = list(wQ.id_order)

# Convert W objects to dense arrays
WP, _ = wP.full()
WS, _ = wS.full()
WQ_array, _ = wQ.full()


# --------------------------------------------
# 4. Build outcome and design matrix
# --------------------------------------------

y = df_geo[y_col].values

X_raw = df_geo[X_cols].values
N = X_raw.shape[0]

# Add intercept
X = np.hstack([np.ones((N, 1)), X_raw])
K = X.shape[1]


# --------------------------------------------
# 5. Core model functions: OLS / SEM / SLM
# --------------------------------------------

def moran_I_val_from_array(resid, W_array, ids):
    """
    Compute Moran's I for residuals using a dense spatial weight matrix.
    """
    w_obj = util.full2W(W_array, ids=ids)
    mi = Moran(resid, w_obj, permutations=0)

    return mi.I, mi.p_norm


def fit_ols(y, X, W_eval, ids):
    """
    Fit OLS and compute residual Moran's I.
    """
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X.dot(beta)

    sse = float(e.T.dot(e))
    N_, K_ = X.shape
    df_ = N_ - K_

    sigma2 = sse / df_

    logL = (
        -(N_ / 2) * np.log(2 * np.pi * sigma2)
        - sse / (2 * sigma2)
    )

    AIC = -2 * logL + 2 * K_

    AICc = (
        AIC + (2 * K_ * (K_ + 1)) / (N_ - K_ - 1)
        if (N_ - K_ - 1) > 0
        else np.nan
    )

    I, pI = moran_I_val_from_array(e, W_eval, ids)

    return {
        "model_type": "OLS",
        "W_mix": "-",
        "logL": logL,
        "AICc": AICc,
        "MoranI": I,
        "absMoranI": abs(I),
        "p_norm": pI,
    }


def build_A_and_logdet(lam, W_list):
    """
    Build A = I - sum(lambda_k W_k) and return A and log|A|.
    """
    N_ = W_list[0].shape[0]
    A = np.eye(N_)

    for i, Wk in enumerate(W_list):
        A -= lam[i] * Wk

    sign, logdetA = slogdet(A)

    if sign <= 0:
        return None, None

    return A, logdetA


def neglog_sem(params, y, X, W_list):
    """
    Negative log-likelihood for a multi-channel spatial error model.

    Model:
        y = X beta + mu
        mu = (I - sum(lambda_k W_k))^{-1} epsilon

    Therefore:
        epsilon = A (y - X beta)
        A = I - sum(lambda_k W_k)
    """
    N_, K_ = X.shape
    m = len(W_list)

    lam = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(lam, W_list)

    if A is None:
        return 1e12

    e = y - X.dot(beta)
    v = A.dot(e)

    sse = float(v.T.dot(v))

    df_ = N_ - (K_ + m)

    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_

    if sigma2 <= 0:
        return 1e12

    logL = (
        logdetA
        - (N_ / 2) * np.log(2 * np.pi * sigma2)
        - sse / (2 * sigma2)
    )

    return -logL


def fit_sem_multi(y, X, W_list):
    """
    Fit a multi-channel spatial error model.
    """
    N_, K_ = X.shape
    m = len(W_list)

    # Initial values: OLS beta and lambda = 0
    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)

    x0 = np.r_[np.zeros(m), beta0]
    bounds = [(-0.99, 0.99)] * m + [(-np.inf, np.inf)] * K_

    res = minimize(
        neglog_sem,
        x0,
        args=(y, X, W_list),
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": 200,
            "ftol": 1e-4,
            "gtol": 1e-4,
        },
    )

    params = res.x

    lam = params[:m]
    beta = params[m:]

    logL = -res.fun

    p = m + K_

    AIC = -2 * logL + 2 * p

    AICc = (
        AIC + (2 * p * (p + 1)) / (N_ - p - 1)
        if (N_ - p - 1) > 0
        else np.nan
    )

    A, _ = build_A_and_logdet(lam, W_list)

    v = A.dot(y - X.dot(beta))

    return lam, beta, logL, AICc, v


def neglog_slm(params, y, X, W_list):
    """
    Negative log-likelihood for a multi-channel spatial lag model.

    Model:
        y = sum(rho_k W_k y) + X beta + u
        u ~ N(0, sigma^2 I)

    Therefore:
        u = A y - X beta
        A = I - sum(rho_k W_k)
    """
    N_, K_ = X.shape
    m = len(W_list)

    rho = params[:m]
    beta = params[m:]

    A, logdetA = build_A_and_logdet(rho, W_list)

    if A is None:
        return 1e12

    u = A.dot(y) - X.dot(beta)

    sse = float(u.T.dot(u))

    df_ = N_ - (K_ + m)

    if df_ <= 0:
        return 1e12

    sigma2 = sse / df_

    if sigma2 <= 0:
        return 1e12

    logL = (
        logdetA
        - (N_ / 2) * np.log(2 * np.pi * sigma2)
        - sse / (2 * sigma2)
    )

    return -logL


def fit_slm_multi(y, X, W_list):
    """
    Fit a multi-channel spatial lag model using the hero connectivity structure.
    """
    N_, K_ = X.shape
    m = len(W_list)

    beta0, *_ = np.linalg.lstsq(X, y, rcond=None)

    x0 = np.r_[np.zeros(m), beta0]
    bounds = [(-0.99, 0.99)] * m + [(-np.inf, np.inf)] * K_

    res = minimize(
        neglog_slm,
        x0,
        args=(y, X, W_list),
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": 200,
            "ftol": 1e-4,
            "gtol": 1e-4,
        },
    )

    params = res.x

    rho = params[:m]
    beta = params[m:]

    logL = -res.fun

    p = m + K_

    AIC = -2 * logL + 2 * p

    AICc = (
        AIC + (2 * p * (p + 1)) / (N_ - p - 1)
        if (N_ - p - 1) > 0
        else np.nan
    )

    A, _ = build_A_and_logdet(rho, W_list)

    u = A.dot(y) - X.dot(beta)

    return rho, beta, logL, AICc, u


# --------------------------------------------
# 6. Parse hero model name and extract channels
# --------------------------------------------

W_dict = {
    "WP": WP,
    "WS": WS,
    "WQ": WQ_array,
}


def get_W_list_from_model_name(model_name, W_dict_local):
    """
    Parse the hero model name and return the corresponding W matrices.

    Examples:
        "SEM_add_1_WS"    -> ["WS"]
        "SEM_add_2_WP_WQ" -> ["WP", "WQ"]
    """
    parts = model_name.split("_")

    chans = [p for p in parts if p in W_dict_local]

    if len(chans) == 0:
        raise ValueError(f"Cannot parse spatial channels from model name: {model_name}")

    W_list = [W_dict_local[ch] for ch in chans]

    return W_list, chans


W_list_hero, hero_channels = get_W_list_from_model_name(hero_model, W_dict)

W_mix_str = "+".join(hero_channels)

print(f"[Config] Outcome = {outcome_tag}, y_col = {y_col}")
print(f"[Config] Hero SEM structure = {hero_model} (channels: {W_mix_str})")
print(f"[Sample] N = {N} counties")


# --------------------------------------------
# 7. Fit OLS, hero SEM, and hero-structure SLM
# --------------------------------------------

# 7.1 OLS
ols_res = fit_ols(
    y,
    X,
    W_eval=WQ_array,
    ids=IDs,
)

# 7.2 Hero SEM: spatial error model
lam_sem, beta_sem, logL_sem, AICc_sem, resid_sem = fit_sem_multi(
    y,
    X,
    W_list_hero,
)

I_sem, p_sem = moran_I_val_from_array(
    resid_sem,
    WQ_array,
    IDs,
)

sem_res = {
    "model_type": "SEM_error",
    "W_mix": W_mix_str,
    "logL": logL_sem,
    "AICc": AICc_sem,
    "MoranI": I_sem,
    "absMoranI": abs(I_sem),
    "p_norm": p_sem,
}

# 7.3 Hero-structure SLM: spatial lag model
rho_slm, beta_slm, logL_slm, AICc_slm, resid_slm = fit_slm_multi(
    y,
    X,
    W_list_hero,
)

I_slm, p_slm = moran_I_val_from_array(
    resid_slm,
    WQ_array,
    IDs,
)

slm_res = {
    "model_type": "SLM_lag",
    "W_mix": W_mix_str,
    "logL": logL_slm,
    "AICc": AICc_slm,
    "MoranI": I_slm,
    "absMoranI": abs(I_slm),
    "p_norm": p_slm,
}


# --------------------------------------------
# 8. Combine results and save appendix table
# --------------------------------------------

res_table = pd.DataFrame(
    [
        ols_res,
        sem_res,
        slm_res,
    ]
)

# Sort by AICc and then by absolute Moran's I
res_table = (
    res_table
    .sort_values(
        by=["AICc", "absMoranI"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

print(f"\n=== OLS vs hero SEM vs hero-structure SLM (US, {outcome_tag}, 2019) ===")
print(res_table)

out_csv = f"APPENDIX_SEM_vs_SLM_US_{outcome_tag}.csv"

res_table.to_csv(
    out_csv,
    index=False,
    encoding="utf-8-sig",
)

print(f"\n[Saved] {out_csv}")